# Refuerzo y cerebro modular

Votación de decisiones, entornos multicuerpo, biomas neuronales, cerebro modular con expertos y tálamo cortical.

Parte del **laboratorio neuronal**. Cada sección es un experimento independiente: ejecuta solo la que te interese, de arriba hacia abajo (cada una instala lo que necesita).

##neuronas individuales con RL

In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Dataset sintético (regresión no lineal)
torch.manual_seed(0)
np.random.seed(0)
X = torch.linspace(-3, 3, 300).unsqueeze(1) # [300, 1]
y = X.pow(3) - X + 0.5 * torch.randn(X.size()) # [300, 1]

# --- Neurona con Bucle de Retroalimentación Local (RL Interno) ---
class RlNeuron(nn.Module):
    def __init__(self, input_features, output_features):
        super().__init__()
        self.input_features = input_features
        self.output_features = output_features

        # Pesos y sesgo estándar de la neurona
        self.weight = nn.Parameter(torch.randn(input_features, output_features) * 0.1)
        self.bias = nn.Parameter(torch.randn(output_features) * 0.1)

        # Parámetro para el "RL Interno"
        # Este factor de adaptabilidad local se ajustará.
        # Lo inicializamos en 1.0 para que no tenga efecto al inicio.
        self.local_adapt_factor = nn.Parameter(torch.ones(output_features) * 0.1) # Un factor por neurona de salida

        # Variable para almacenar el gradiente recibido en la última corrección.
        # No es un Parameter, ya que se calcula en el forward o en el hook.
        self._last_weight_grad_norm = None

        # Registramos un hook para capturar el gradiente de los pesos después del backward pass
        # Este hook se ejecutará DESPUÉS de que .backward() calcule el gradiente para self.weight
        self.weight.register_hook(self._capture_grad_norm)

    def _capture_grad_norm(self, grad):
        # Esta función se llama cuando se calcula el gradiente para self.weight
        # Usamos .detach() para no interferir con el cálculo de gradientes normal
        self._last_weight_grad_norm = grad.norm().item() # Norma L2 del gradiente del peso

    def forward(self, x):
        # La salida de la neurona se multiplica por el factor de adaptabilidad local
        # Esto es una forma de simular el impacto del RL interno en la salida
        # Podrías hacer que afecte la función de activación, el sesgo, etc.
        linear_output = torch.matmul(x, self.weight) + self.bias

        # Aplicamos el factor de adaptabilidad local *después* de la operación lineal
        # Esto significa que la salida de la neurona está "influenciada" por su estado RL
        # F.relu se aplica para mantener la no linealidad
        output = F.relu(linear_output * self.local_adapt_factor)

        return output

    # Este método se llamaría DESPUÉS de optimizer.step()
    # Aquí es donde el "RL interno" entra en juego y corrige su propio estado
    def apply_local_feedback(self, lr_local_feedback=0.01):
        if self._last_weight_grad_norm is not None:
            # Una lógica simple de RL: si el gradiente fue grande, aumenta el factor de adaptabilidad.
            # Podrías usar funciones de recompensa, umbrales, etc.
            # Aquí, simplemente ajustamos el factor en proporción a la magnitud del gradiente del peso
            # y a una tasa de aprendizaje para el RL.
            # Usamos item() para obtener el valor numérico del tensor escalar.

            # Ajustamos el local_adapt_factor
            # NOTA: Directamente modificar self.local_adapt_factor.data
            # evita que esta operación sea parte del grafo de computación principal,
            # lo cual es deseable para un "meta-ajuste" como este.
            adjustment = self._last_weight_grad_norm * lr_local_feedback

            # Podemos limitar el rango del factor para evitar inestabilidad
            self.local_adapt_factor.data.add_(adjustment).clamp_(min=0.01, max=10.0)

            # Reiniciamos el gradiente capturado para la siguiente iteración
            self._last_weight_grad_norm = None


# --- Red Neuronal usando la Neurona con RL Interno ---
class RlNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.rl_neuron1 = RlNeuron(input_size, hidden_size)
        self.rl_neuron2 = RlNeuron(hidden_size, hidden_size)
        # La capa de salida puede ser una neurona normal o también una RLNeuron
        # Para simplificar, usaremos una lineal estándar al final
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h1 = self.rl_neuron1(x)
        h2 = self.rl_neuron2(h1)
        out = self.output_layer(h2)
        return out

    def apply_all_local_feedback(self, lr_local_feedback=0.01):
        # Itera sobre todas las neuronas RL y aplica su retroalimentación local
        self.rl_neuron1.apply_local_feedback(lr_local_feedback)
        self.rl_neuron2.apply_local_feedback(lr_local_feedback)

# --- Red Neuronal Estándar para Comparación ---
class StandardNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h1 = F.relu(self.fc1(x))
        h2 = F.relu(self.fc2(h2)) # ERROR: Debería ser h2 = F.relu(self.fc2(h1))
        out = self.output_layer(h2)
        return out

# --- Parámetros de Entrenamiento ---
LEARNING_RATE_GLOBAL = 0.01
LEARNING_RATE_LOCAL_FEEDBACK = 0.005 # Tasa de aprendizaje para el RL interno
EPOCHS = 1000
LOSS_FN = nn.MSELoss()

INPUT_SIZE = X.shape[1] # 1
HIDDEN_SIZE = 16 # Tamaño de las capas ocultas
OUTPUT_SIZE = y.shape[1] # 1

# --- Inicialización de Modielos ---
print("--- Inicializando y Entrenando Red con Neuronas RL Internas ---")
rl_model = RlNet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)
optimizer_rl = torch.optim.Adam(rl_model.parameters(), lr=LEARNING_RATE_GLOBAL)
historial_loss_rl = []

total_params_rl = sum(p.numel() for p in rl_model.parameters() if p.requires_grad)
print(f"Número de parámetros en Red con Neuronas RL: {total_params_rl}")

for epoch in range(EPOCHS):
    rl_model.train()
    optimizer_rl.zero_grad()

    y_pred = rl_model(X)
    loss = LOSS_FN(y_pred, y)
    historial_loss_rl.append(loss.item())

    loss.backward() # Calcula los gradientes, los hooks se activan aquí

    # Aplicar el bucle de retroalimentación local DESPUÉS de que los gradientes han sido calculados
    # y ANTES de que el optimizador global actualice los pesos.
    # O, más comúnmente, DESPUÉS de optimizer.step() para que el ajuste local sea sobre los pesos ya corregidos.
    # Optaremos por DESPUÉS para que el RL interno reaccione a la corrección FINAL del optimizador.
    optimizer_rl.step()
    rl_model.apply_all_local_feedback(lr_local_feedback=LEARNING_RATE_LOCAL_FEEDBACK)

    if epoch % 1000 == 0:
        print(f"Red RL - Época {epoch} - Pérdida: {loss.item():.6f}")

print(f"Pérdida final (Red RL): {historial_loss_rl[-1]:.6f}\n")


print("--- Inicializando y Entrenando Red Estándar ---")
# Corrección en StandardNet: h2 = F.relu(self.fc2(h1))
class StandardNetCorrected(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h1 = F.relu(self.fc1(x))
        h2 = F.relu(self.fc2(h1)) # Corregido: usa h1 como entrada
        out = self.output_layer(h2)
        return out


standard_model = StandardNetCorrected(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)
optimizer_standard = torch.optim.Adam(standard_model.parameters(), lr=LEARNING_RATE_GLOBAL)
historial_loss_standard = []

total_params_standard = sum(p.numel() for p in standard_model.parameters() if p.requires_grad)
print(f"Número de parámetros en Red Estándar: {total_params_standard}")

for epoch in range(EPOCHS):
    standard_model.train()
    optimizer_standard.zero_grad()

    y_pred = standard_model(X)
    loss = LOSS_FN(y_pred, y)
    historial_loss_standard.append(loss.item())

    loss.backward()
    optimizer_standard.step()

    if epoch % 1000 == 0:
        print(f"Red Estándar - Época {epoch} - Pérdida: {loss.item():.6f}")

print(f"Pérdida final (Red Estándar): {historial_loss_standard[-1]:.6f}\n")


## Visualización de Resultados

rl_model.eval()
standard_model.eval()

with torch.no_grad():
    y_hat_rl = rl_model(X).numpy()
    y_hat_standard = standard_model(X).numpy()

X_np = X.numpy()
y_np = y.numpy()

plt.figure(figsize=(15, 6))

# Gráfico de Predicciones
plt.subplot(1, 2, 1)
plt.title("Comparación de Predicciones (Dataset Regresión)")
plt.plot(X_np, y_np, label="Valores Reales", color='blue', alpha=0.7)
plt.plot(X_np, y_hat_rl, label="Predicción Red RL", color='red', linestyle='--')
plt.plot(X_np, y_hat_standard, label="Predicción Red Estándar", color='green', linestyle=':')
plt.legend()
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)

# Gráfico de Evolución de la Pérdida
plt.subplot(1, 2, 2)
plt.title("Evolución de la Pérdida (MSE)")
plt.plot(historial_loss_rl, label="Pérdida Red RL", color='red')
plt.plot(historial_loss_standard, label="Pérdida Red Estándar", color='green')
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.legend()
plt.grid(True)
plt.yscale('log')
plt.tight_layout()
plt.show()

# Resumen Final de la Pérdida
print(f"--- Resumen de Pérdidas Finales ---")
print(f"Pérdida final (Red con Neuronas RL): {historial_loss_rl[-1]:.6f}")
print(f"Pérdida final (Red Estándar):         {historial_loss_standard[-1]:.6f}")

## Entorno RL para código Python

In [ ]:

!pip install --upgrade datasets fsspec huggingface_hub

from datasets import load_dataset
dataset = load_dataset("xquad", "xquad.es")
print(dataset)

from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, TrainingArguments, Trainer
import torch

# Cargar el dataset XQuAD en español
# Si ya lo tienes cargado como 'dataset', puedes saltarte esta línea
dataset = load_dataset("xquad", "xquad.es")

# Acceder a la parte de validación (es la que tenemos en XQuAD por defecto)
validation_split = dataset['validation']

# Calcular el 0.02% de las filas
# dataset['validation'] tiene 1190 filas
total_rows = len(validation_split)
sample_size = int(total_rows * 0.0002) # 0.02%
if sample_size == 0: # Asegurarse de que al menos haya una fila si el 0.02% es menos de 1
    sample_size = 1
print(f"Número total de filas: {total_rows}")
print(f"Tamaño de la muestra (0.02%): {sample_size}")

# Tomar una muestra aleatoria del dataset
# Usamos 'select' para seleccionar un rango de índices.
# Para una muestra aleatoria, es mejor usar shuffle y luego select.
small_dataset = validation_split.shuffle(seed=42).select(range(sample_size))

print("\nPequeño subconjunto del dataset para entrenamiento:")
print(small_dataset)
print(small_dataset[0]) # Ver la primera entrada de la muestra

# Cargar un tokenizer adecuado para QA multilingüe
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-multilingual-cased")

# Función para preprocesar los datos
def preprocess_function(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=384, # Longitud máxima para las secuencias
        truncation="only_second", # Truncar solo el contexto si es demasiado largo
        return_offsets_mapping=True, # Necesario para mapear caracteres a tokens
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        answer = answers[i]
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])

        sequence_ids = inputs.sequence_ids(i)

        # Encontrar el inicio y fin del contexto
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # Si la respuesta no está dentro del contexto visible
        if offset[context_start][0] > end_char or offset[context_end][1] < start_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # Encontrar las posiciones de inicio y fin de la respuesta en los tokens
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

# Aplicar la función de preprocesamiento al pequeño dataset
tokenized_small_dataset = small_dataset.map(preprocess_function, batched=True, remove_columns=small_dataset.column_names)

# Convertir a formato PyTorch si es necesario (Trainer lo hace automáticamente)
# tokenized_small_dataset.set_format("torch")

# Cargar el modelo pre-entrenado para Question Answering
# Corrected typo in the model name: "multilinguall" -> "multilingual"
model = AutoModelForQuestionAnswering.from_pretrained("distilbert-base-multilingual-cased")

# Configurar los argumentos de entrenamiento
training_args = TrainingArguments(
    output_dir="./qa_model_output", # Directorio donde se guardarán los resultados
    per_device_train_batch_size=8, # Tamaño de batch para entrenamiento
    num_train_epochs=3, # Número de épocas (para una muestra pequeña, 3 es suficiente para demo)
    logging_dir='./logs', # Directorio para los logs
    logging_steps=10, # Frecuencia de los logs
    save_strategy="no", # No guardar el modelo en cada paso (para esta demo)
    report_to="none", # No reportar a ninguna plataforma (para simplicidad)
)

# Inicializar el Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_small_dataset, # Nuestro pequeño dataset
    tokenizer=tokenizer, # Pasamos el tokenizer
)

# Iniciar el entrenamiento
print("\nIniciando el entrenamiento del modelo...")
trainer.train()

print("\nEntrenamiento completado.")

# Cargar un tokenizer adecuado para QA multilingüe
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-multilingual-cased")

# Función para preprocesar los datos
def preprocess_function(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=384, # Longitud máxima para las secuencias
        truncation="only_second", # Truncar solo el contexto si es demasiado largo
        return_offsets_mapping=True, # Necesario para mapear caracteres a tokens
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        answer = answers[i]
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])

        sequence_ids = inputs.sequence_ids(i)

        # Encontrar el inicio y fin del contexto
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # Si la respuesta no está dentro del contexto visible
        if offset[context_start][0] > end_char or offset[context_end][1] < start_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # Encontrar las posiciones de inicio y fin de la respuesta en los tokens
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

# Aplicar la función de preprocesamiento al pequeño dataset
tokenized_small_dataset = small_dataset.map(preprocess_function, batched=True, remove_columns=small_dataset.column_names)

# Convertir a formato PyTorch si es necesario (Trainer lo hace automáticamente)
# tokenized_small_dataset.set_format("torch")

# Definir el contexto y la pregunta
context = "¿Qué es el río Amazonas? El río Amazonas es un largo río que fluye a través de Sudamérica y es el río más grande del mundo por volumen de descarga de agua."
question = "¿Cuál es el río más grande por volumen de descarga de agua?"

# Tokenizar la entrada
# Aquí es crucial que el preprocesamiento sea similar al que se hizo para el entrenamiento
inputs = tokenizer(question, context, return_tensors="pt", max_length=384, truncation="only_second")

# Realizar la inferencia
with torch.no_grad(): # Desactivar el cálculo de gradientes para la inferencia (ahorra memoria y es más rápido)
    outputs = model(**inputs)

# Las salidas del modelo son 'logits' para las posiciones de inicio y fin
start_logits = outputs.start_logits
end_logits = outputs.end_logits

# Encontrar la posición del token con la mayor probabilidad para el inicio y el fin
start_index = torch.argmax(start_logits)
end_index = torch.argmax(end_logits) + 1 # +1 para incluir el token final

# Convertir los IDs de los tokens a una lista
input_ids_list = inputs["input_ids"][0].tolist()

# Decodificar los tokens para obtener la respuesta
# Se asegura que la posición final sea después de la inicial y que ambas estén dentro de los límites
if start_index <= end_index and end_index <= len(input_ids_list):
    answer_tokens = input_ids_list[start_index:end_index]
    answer = tokenizer.decode(answer_tokens, skip_special_tokens=True)
else:
    answer = "No se encontró una respuesta clara en el contexto."

print(f"\nContexto: {context}")
print(f"Pregunta: {question}")
print(f"Respuesta del modelo: {answer}")

from transformers import pipeline

# Crear el pipeline para Question Answering
# Si hubieras guardado tu modelo, usarías model="./qa_model_output"
qa_pipeline = pipeline("question-answering", model="distilbert-base-multilingual-cased", tokenizer="distilbert-base-multilingual-cased")

# Usar el pipeline
result = qa_pipeline(question=question, context=context)
print(f"\nRespuesta del pipeline: {result['answer']}")

In [ ]:
!pip install stable_baselines3

In [ ]:
import random
import gymnasium as gym
from gymnasium import spaces
import numpy as np

# Importa los componentes necesarios de Stable Baselines3
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import VecEnvWrapper

# --- 1. Entorno Base (RawPythonEnv) ---
# Este entorno solo maneja la lógica del estado y las transiciones.
# NO calcula recompensas directamente.
class RawPythonEnv(gym.Env):
    metadata = {'render_modes': ['human'], 'render_fps': 4}

    def __init__(self, target_output="hola", max_code_tokens=15):
        super().__init__()
        self.target_output = target_output
        self.max_code_tokens = max_code_tokens
        self.current_tokens = []
        self.current_output = ""
        self.defined_vars = {}
        self.last_parsed_line = ""
        self.last_line_error = False  # Bandera para indicar si la última línea fue un error

        self.alphabet = 'abcdefghijklmnopqrstuvwxyz'
        self.digits = '0123456789'
        self.operators = ['+', '-', '=', '"', '(', ')', '_'] # Añadido '_' para nombres de variables
        self.keywords = ['print']
        self.action_space_chars = list(self.alphabet) + list(self.digits) + self.operators + self.keywords + ['_EOS_']
        self.action_space = spaces.Discrete(len(self.action_space_chars))

        # Espacio de observación:
        # One-hot de los últimos 5 tokens + presencia de variables + longitud de salida + coincidencia con objetivo
        self.observation_space = spaces.Box(low=0, high=1, shape=(5 * len(self.action_space_chars) + len(self.alphabet) + 2,), dtype=np.float32)

    def _get_obs(self):
        # One-hot encoding de los últimos 5 tokens para dar contexto sintáctico
        obs_tokens = np.zeros(5 * len(self.action_space_chars))
        for i, token in enumerate(self.current_tokens[-5:]):
            if token in self.action_space_chars:
                token_idx = self.action_space_chars.index(token)
                obs_tokens[i * len(self.action_space_chars) + token_idx] = 1

        # Presencia de variables definidas (a-z)
        vars_presence = np.array([1 if var in self.defined_vars else 0 for var in self.alphabet], dtype=np.float32)

        # Longitud de la salida actual y si contiene el objetivo
        output_len = np.array([len(self.current_output) / 100.0], dtype=np.float32)  # Normalizar
        target_match = np.array([1.0 if self.target_output in self.current_output else 0.0], dtype=np.float32)

        obs = np.concatenate([obs_tokens, vars_presence, output_len, target_match])
        return obs

    def _get_info(self):
        # Esta información se pasará al algoritmo de recompensa externo
        return {
            "code_tokens": self.current_tokens,
            "output": self.current_output,
            "defined_vars": self.defined_vars,
            "last_parsed_line": self.last_parsed_line,
            "last_line_error": self.last_line_error,
            "target_output": self.target_output,  # Se pasa el objetivo para el cálculo de recompensa
            "current_token_count": len(self.current_tokens) # Cuántos tokens se han generado en la línea actual
        }

    def _interpret_line(self, line_tokens):
        # Simulación de un intérprete Python básico
        line = "".join(line_tokens)
        temp_output = ""
        error = False

        if line.startswith("print("):
            if line.endswith(")"):
                content = line[6:-1]
                if content.startswith("\"") and content.endswith("\""):
                    try:
                        temp_output = content[1:-1]
                    except Exception:
                        error = True
                elif '+' in content:
                    parts = content.split('+')
                    if len(parts) == 2:
                        var1_name = parts[0].strip()
                        var2_name = parts[1].strip()
                        if var1_name in self.defined_vars and var2_name in self.defined_vars:
                            try:
                                res = self.defined_vars[var1_name] + self.defined_vars[var2_name]
                                temp_output = str(res)
                            except TypeError:
                                error = True
                        else:
                            error = True
                    else:
                        error = True
                elif '-' in content:
                    parts = content.split('-')
                    if len(parts) == 2:
                        var1_name = parts[0].strip()
                        var2_name = parts[1].strip()
                        if var1_name in self.defined_vars and var2_name in self.defined_vars:
                            if isinstance(self.defined_vars[var1_name], (int, float)) and isinstance(self.defined_vars[var2_name], (int, float)):
                                res = self.defined_vars[var1_name] - self.defined_vars[var2_name]
                                temp_output = str(res)
                            else:
                                error = True
                        else:
                            error = True
                    else:
                        error = True
                elif content in self.defined_vars:
                    temp_output = str(self.defined_vars[content])
                else:
                    error = True
            else:
                error = True
        elif '=' in line:  # Asignación de variables
            parts = line.split('=', 1)
            if len(parts) == 2:
                var_name = parts[0].strip()
                value_str = parts[1].strip()
                if var_name in self.alphabet:
                    if value_str.startswith("\"") and value_str.endswith("\""):
                        self.defined_vars[var_name] = value_str[1:-1]
                    elif value_str.isdigit():
                        self.defined_vars[var_name] = int(value_str)
                    elif '+' in value_str:
                        sub_parts = value_str.split('+')
                        if len(sub_parts) == 2:
                            val1_str = sub_parts[0].strip()
                            val2_str = sub_parts[1].strip()
                            val1 = self.defined_vars.get(val1_str) if val1_str in self.defined_vars else (int(val1_str) if val1_str.isdigit() else None)
                            val2 = self.defined_vars.get(val2_str) if val2_str in self.defined_vars else (int(val2_str) if val2_str.isdigit() else None)

                            if val1 is not None and val2 is not None:
                                try: self.defined_vars[var_name] = val1 + val2
                                except TypeError: error = True
                            else: error = True
                        else: error = True
                    elif '-' in value_str:
                        sub_parts = value_str.split('-')
                        if len(sub_parts) == 2:
                            val1_str = sub_parts[0].strip()
                            val2_str = sub_parts[1].strip()
                            val1 = self.defined_vars.get(val1_str) if val1_str in self.defined_vars else (int(val1_str) if val1_str.isdigit() else None)
                            val2 = self.defined_vars.get(val2_str) if val2_str in self.defined_vars else (int(val2_str) if val2_str.isdigit() else None)
                            if val1 is not None and val2 is not None:
                                if isinstance(val1, (int, float)) and isinstance(val2, (int, float)):
                                    self.defined_vars[var_name] = val1 - val2
                                else: error = True
                            else: error = True
                        else: error = True
                    elif value_str in self.defined_vars:
                        self.defined_vars[var_name] = self.defined_vars[value_str]
                    else: error = True
                else: error = True
            else: error = True
        else:
            error = True  # Sintaxis no reconocida

        return temp_output, error

    def step(self, action):
        terminated = False  # Si el episodio termina por éxito o fallo
        truncated = False   # Si el episodio termina por límite de tiempo/pasos

        token_to_add = self.action_space_chars[action]
        self.current_tokens.append(token_to_add)
        self.last_line_error = False  # Resetear la bandera de error para la línea actual

        # Si el agente genera un token de fin de línea, intentar interpretar la línea
        if token_to_add == '_EOS_':
            self.last_parsed_line = "".join(self.current_tokens[:-1])  # La línea sin el _EOS_
            output, error = self._interpret_line(self.current_tokens[:-1])  # Interpretar hasta el _EOS_

            if not error:
                self.current_output += output
            else:
                self.last_line_error = True  # Marcar que hubo un error

            self.current_tokens = []  # Reiniciar tokens para la siguiente línea

        # Condiciones de terminación del episodio (independientes de la recompensa)
        if self.target_output in self.current_output:
            terminated = True # El objetivo se logró
        elif len(self.current_tokens) + len(self.last_parsed_line) >= self.max_code_tokens * 2 and '_EOS_' not in self.current_tokens:
            # Si el agente escribe demasiados tokens en una sola línea sin un _EOS_
            truncated = True
        elif len(self.current_output) > len(self.target_output) * 2 and not (self.target_output in self.current_output):
            # Si la salida acumulada crece demasiado sin alcanzar el objetivo
            truncated = True
        elif len(self.current_tokens) >= self.max_code_tokens * 2 and not terminated:
            # Si el número total de tokens en la línea actual excede el límite
            truncated = True

        obs = self._get_obs()
        info = self._get_info()

        # IMPORTANTE: La recompensa aquí es SIEMPRE 0.0. Será calculada por el algoritmo externo.
        return obs, 0.0, terminated, truncated, info

    def reset(self, seed=None, options=None):
        super().reset(seed=seed, options=options)  # Asegura compatibilidad con Gymnasium
        self.current_tokens = []
        self.current_output = ""
        self.defined_vars = {}
        self.last_parsed_line = ""
        self.last_line_error = False
        obs = self._get_obs()
        info = self._get_info()
        return obs, info

    def render(self, mode='human'):
        if mode == 'human':
            print("\n--- AGENTE CONSTRUYENDO LÍNEA (Entorno Base) ---")
            print("".join(self.current_tokens))
            if self.last_parsed_line:
                print(f"ÚLTIMA LÍNEA INTERPRETADA: {self.last_parsed_line}")
            print(f"SALIDA ACUMULADA: '{self.current_output}'")
            print(f"VARIABLES: {self.defined_vars}")
            print(f"ERROR EN ÚLTIMA LÍNEA: {self.last_line_error}")
            print("---------------------------------")


# --- 2. Algoritmo Externo de Recompensa (calculate_custom_reward) ---
# Esta función es el "otro algoritmo" que decide la recompensa y el castigo.
def calculate_custom_reward(info_from_env: dict, action_taken: int, eos_idx: int, max_code_tokens: int):
    reward = -0.005  # Pequeña penalización por cada token/paso (fomenta la eficiencia)

    # Solo aplicamos recompensas/castigos relacionados con la línea cuando se ha interpretado una línea completa
    # Se detecta que se procesó una línea si current_tokens está vacío y last_parsed_line no lo está
    if not info_from_env["code_tokens"] and info_from_env["last_parsed_line"]:
        if info_from_env["last_line_error"]:
            reward -= 1.0  # **Castigo fuerte** por error de sintaxis en la línea
        else:
            reward += 0.5  # **Recompensa** por una línea sintácticamente válida

            # Recompensas/castigos basados en el objetivo de salida
            if info_from_env["output"] == info_from_env["target_output"]:
                reward += 5.0  # **Gran recompensa** por lograr el objetivo exacto
            elif info_from_env["target_output"] in info_from_env["output"]:
                reward += 2.0  # **Recompensa** por contener el objetivo (progreso hacia la meta)

    # Penalización adicional si el agente excede el número máximo de tokens por línea sin un _EOS_
    if action_taken != eos_idx and info_from_env["current_token_count"] >= max_code_tokens:
        reward -= 0.5 # Pequeña penalización por "divagar" demasiado sin cerrar la línea

    return reward

# --- 3. Wrapper que Inyecta la Recompensa (RewardInjectingWrapper) ---
# Este wrapper se interpone entre el agente y el entorno base para modificar la recompensa.
class RewardInjectingWrapper(gym.Wrapper):
    def __init__(self, env, reward_calculator_func):
        super().__init__(env)
        self.reward_calculator = reward_calculator_func
        # Necesitamos pasar el índice de _EOS_ y max_code_tokens al calculador de recompensa si la usa
        self._eos_idx = self.env.action_space_chars.index('_EOS_')
        self._max_code_tokens = self.env.max_code_tokens

    def step(self, action):
        # Llama al método `step()` del entorno base (RawPythonEnv), que devuelve reward=0.0
        observation, base_reward, terminated, truncated, info = self.env.step(action)

        # Calcula la recompensa real utilizando el algoritmo externo
        # Pasa la información del entorno, la acción tomada, el índice _EOS_ y el max_code_tokens
        custom_reward = self.reward_calculator(info, action, self._eos_idx, self._max_code_tokens)

        # Devuelve la observación, la recompensa CALCULADA por el algoritmo externo, y el resto de la info
        return observation, custom_reward, terminated, truncated, info

    def reset(self, seed=None, options=None):
        return self.env.reset(seed=seed, options=options)


# --- 4. Script Principal de Entrenamiento y Demostración ---
if __name__ == "__main__":
    TARGET_OUTPUT = "hola mundo"
    MAX_CODE_TOKENS_PER_LINE = 20 # Límite de tokens para una sola línea de código

    # Crear el entorno base (que NO calcula recompensa)
    base_env_instance = RawPythonEnv(target_output=TARGET_OUTPUT, max_code_tokens=MAX_CODE_TOKENS_PER_LINE)

    # Definir la función que "make_vec_env" usará para crear instancias de entornos
    # Ahora esta function creará un entorno envuelto en Monitor Y luego en RewardInjectingWrapper
    def make_wrapped_env():
        # Envolver el entorno base con nuestro RewardInjectingWrapper
        wrapped_with_reward = RewardInjectingWrapper(base_env_instance, calculate_custom_reward)
        # Opcional: Envolver con Monitor para ver métricas de entrenamiento en TensorBoard
        # Apply Monitor *before* make_vec_env vectorizes it
        monitored_env = Monitor(wrapped_with_reward)
        return monitored_env # Return the environment wrapped in both

    # Crear un entorno vectorizado (aunque solo sea uno para simplificar)
    # make_vec_env se encargará de llamar a make_wrapped_env n_envs veces
    env = make_vec_env(make_wrapped_env, n_envs=1)

    # El Monitor ya está aplicado dentro de make_wrapped_env,
    # así que no necesitamos aplicarlo de nuevo aquí:
    # env = Monitor(env) # REMOVER esta línea

    # Configuración del Agente PPO (modelo fin a fin)
    # PPO aprende directamente de las observaciones a las acciones, recibiendo recompensas
    # que fueron calculadas por nuestro algoritmo externo.
    model = PPO("MlpPolicy", env, verbose=1, learning_rate=0.0003, n_steps=2048,
                batch_size=64, n_epochs=10, gamma=0.99, gae_lambda=0.95, clip_range=0.2,
                ent_coef=0.01, # Fomenta la exploración
                policy_kwargs=dict(net_arch=dict(pi=[64, 64], vf=[64, 64])), # Redes neuronales más grandes
                tensorboard_log="./ppo_external_reward_tensorboard/")

    print(f"\n--- Entrenando el Agente PPO para generar: '{TARGET_OUTPUT}' ---")
    model.learn(total_timesteps=300000) # Aumenta los pasos para una tarea más compleja
    print("Entrenamiento completado.")

    # --- Demostración del Agente Aprendido ---
    print("\n--- Demostrando el Agente Aprendido (con recompensa externa) ---")

    # Resetear el entorno para la demostración
    # Aquí ya no necesitamos el manejo especial de errores de 'reset()'
    # porque `make_vec_env` y los wrappers lo hacen robusto.
    obs, info = env.reset()

    done = False
    episode_reward = 0
    steps = 0
    print("\n--- AGENTE CONSTRUYENDO Y EJECUTANDO ---")

    # Acceder a la instancia del entorno RawPythonEnv dentro del wrapper
    # Esto es para obtener acceso a atributos como action_space_chars directamente
    # Dado que tenemos Monitor y RewardInjectingWrapper, debemos "desenvolver" dos veces
    # The env object is a DummyVecEnv, which contains the Monitor wrapper,
    # which contains the RewardInjectingWrapper, which contains the RawPythonEnv.
    # So, we need to access env.envs[0].env.env to get the original env instance
    unwrapped_raw_env = env.envs[0].env.env # Assumes n_envs=1


    while not done and steps < unwrapped_raw_env.max_code_tokens * 5: # Límite de pasos para la demo
        action, _states = model.predict(obs, deterministic=True) # Elegir la acción más segura
        # El método step de 'env' (que es el DummyVecEnv con wrappers dentro)
        # devolverá la recompensa calculada por 'calculate_custom_reward'
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated[0] or truncated[0] # terminated and truncated are lists for VecEnvs
        episode_reward += reward[0] # reward is also a list for VecEnvs
        steps += 1

        # Acceder a la información del primer (y único) entorno en la lista de infos
        current_info = info[0]

        # Imprimir el token que el agente añadió
        print(f"Token añadido: '{unwrapped_raw_env.action_space_chars[action.item()]}'") # .item() para desempacar array de 1 elemento

        # Si la línea ha sido parseada (cuando el agente añade '_EOS_')
        # Check if current_info["code_tokens"] is empty and last_parsed_line exists
        if not current_info["code_tokens"] and current_info.get("last_parsed_line"):
            print(f"  LÍNEA INTERPRETADA: '{current_info['last_parsed_line']}'")
            print(f"  SALIDA ACUMULADA: '{current_info['output']}'")
            print(f"  VARIABLES: {current_info['defined_vars']}")
            # The reward printed here is the cumulative reward within the Monitor wrapper
            # For the step-by-step reward, you would access info[0]['episode']['r'] or similar if Monitor records step rewards
            # For simplicity here, we'll print the cumulative reward or the step reward from the step() output
            print(f"  RECOMPENSA DE ESTE PASO: {reward[0]:.4f}") # Print the step reward from the step output
            print("---------------------------------")
            # Resetear la línea interpretada en la info para no repetirla en la impresión
            # Modifying the info dictionary like this won't affect the actual info passed to the next step by the VecEnv/Monitor
            # This is purely for the print statement's logic.
            current_info["last_parsed_line"] = ""

    print(f"\n--- DEMO FINALIZADA ---")
    # Access the final output from the info dictionary after the episode ends
    # This might be slightly different depending on how Monitor aggregates info
    # A safer way is to track the output during the loop or access it directly from the unwrapped env if needed
    final_output_info = info[0] if done else env.envs[0].env.get_info()[0] # Try to get info after episode or last one
    print(f"SALIDA FINAL: '{final_output_info.get('output', 'N/A')}'")
    print(f"RECOMPENSA TOTAL DEL EPISODIO: {episode_reward:.2f}") # episode_reward tracked during the loop

    env.close()

## Red de votación de decisiones (RL)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class EmbeddingNetwork(nn.Module):
    def __init__(self, input_dim, embed_dim):
        super().__init__()
        self.embed = nn.Linear(input_dim, embed_dim)

    def forward(self, x):
        return torch.tanh(self.embed(x))


class DecisionHead(nn.Module):
    def __init__(self, embed_dim, action_dim):
        super().__init__()
        self.fc1 = nn.Linear(embed_dim, 64)
        self.fc2 = nn.Linear(64, action_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        return self.fc2(x)  # Output: logits for each action


class DecisionVotingNet(nn.Module):
    def __init__(self, input_dim, embed_dim, action_dim, num_heads=4):
        super().__init__()
        self.embedding = EmbeddingNetwork(input_dim, embed_dim)
        self.heads = nn.ModuleList([DecisionHead(embed_dim, action_dim) for _ in range(num_heads)])

    def forward(self, x):
        z = self.embedding(x)
        logits_list = [head(z) for head in self.heads]  # Each head proposes an action
        stacked_logits = torch.stack(logits_list, dim=0)  # (num_heads, batch, action_dim)
        mean_logits = torch.mean(stacked_logits, dim=0)   # Aggregate via mean (voting)
        return mean_logits, stacked_logits


# Example usage:
if __name__ == "__main__":
    model = DecisionVotingNet(input_dim=10, embed_dim=32, action_dim=5, num_heads=4)
    dummy_input = torch.randn(3, 10)  # batch of 3 samples
    final_logits, all_logits = model(dummy_input)
    chosen_actions = torch.argmax(final_logits, dim=-1)
    print("Final chosen actions:", chosen_actions)
    print("Logits from each head:", all_logits.shape)  # Should be (4, 3, 5)

In [ ]:

# Correr en Google Colab
import torch
import torch.nn as nn
import torch.nn.functional as F
import math, random

# 1. Dataset de ejemplo (cuentos cortos)
corpus = [
    "Había una vez un pequeño dragón que vivía en la cima de una montaña.",
    "El zorro engañó al cuervo para que cantara y dejara caer el queso.",
    "El lobo sopló y sopló, pero no pudo derribar la casa de ladrillos.",
    "Una tortuga ganó una carrera contra una liebre confiada y rápida."
]

# Tokenización simple
tokens = " ".join(corpus).lower().split()
vocab = sorted(set(tokens))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

# Secuencia como números
data = [word2idx[w] for w in tokens]

# Hiperparámetros
d_model = 32
context_size = 6
vocab_size = len(vocab)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Dataset para entrenamiento (inputs y targets)
def get_batch():
    ix = random.randint(0, len(data) - context_size - 1)
    x = torch.tensor(data[ix:ix+context_size], device=device)
    y = torch.tensor(data[ix+1:ix+context_size+1], device=device)
    return x, y

# 2. Modelo Transformer minimalista
class MiniTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Parameter(torch.randn(context_size, d_model))
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.out = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        tok = self.token_embed(x)  # [B, T, D]
        tok = tok + self.pos_embed  # [B, T, D]

        Q = self.q_proj(tok)
        K = self.k_proj(tok)
        V = self.v_proj(tok)
        att = (Q @ K.transpose(-2, -1)) / math.sqrt(Q.size(-1))

        # Causal mask
        mask = torch.tril(torch.ones(context_size, context_size, device=device))
        att = att.masked_fill(mask == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        attended = att @ V

        x = self.ln1(tok + attended)
        x = self.ln2(x + self.ff(x))
        logits = self.out(x)
        return logits

# 3. Entrenamiento
model = MiniTransformer().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for step in range(1000):
    x, y = get_batch()
    logits = model(x.unsqueeze(0))  # batch dim
    loss = F.cross_entropy(logits.squeeze(0)[:-1], y[:-1])  # Predecir próxima palabra
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"Paso {step}, Pérdida: {loss.item():.4f}")

# 4. Generación de texto
def generar_texto(prompt, max_len=20):
    model.eval()
    x = torch.tensor([word2idx.get(w, 0) for w in prompt.lower().split()], device=device)
    for _ in range(max_len):
        x_cond = x[-context_size:] if len(x) >= context_size else F.pad(x, (context_size-len(x), 0))
        logits = model(x_cond.unsqueeze(0))
        next_token = torch.argmax(logits[0, -1])
        x = torch.cat([x, next_token.unsqueeze(0)])
    return " ".join([idx2word[i.item()] for i in x])

# Ejemplo
print("Generado:", generar_texto("había una vez un"))

In [ ]:
# Google Colab - Transformer con visualización y sampling
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import math, random

# Dataset de cuentos
corpus = [
    "Había una vez un pequeño dragón que vivía en la cima de una montaña.",
    "El zorro engañó al cuervo para que cantara y dejara caer el queso.",
    "El lobo sopló y sopló, pero no pudo derribar la casa de ladrillos.",
    "Una tortuga ganó una carrera contra una liebre confiada y rápida."
]

# Tokenización
tokens = " ".join(corpus).lower().split()
vocab = sorted(set(tokens))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
data = [word2idx[w] for w in tokens]

# Config
d_model = 32
context_size = 6
vocab_size = len(vocab)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Batch simple
def get_batch():
    ix = random.randint(0, len(data) - context_size - 1)
    x = torch.tensor(data[ix:ix+context_size], device=device)
    y = torch.tensor(data[ix+1:ix+context_size+1], device=device)
    return x, y

# Modelo con visualización de atención
class MiniTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Parameter(torch.randn(context_size, d_model))
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.out = nn.Linear(d_model, vocab_size)
        self.attn_weights = None  # Para visualización

    def forward(self, x):
        tok = self.token_embed(x)
        tok = tok + self.pos_embed
        Q = self.q_proj(tok)
        K = self.k_proj(tok)
        V = self.v_proj(tok)

        att = (Q @ K.transpose(-2, -1)) / math.sqrt(Q.size(-1))
        mask = torch.tril(torch.ones(context_size, context_size, device=device))
        att = att.masked_fill(mask == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        self.attn_weights = att.detach().cpu()  # Guardar atención

        attended = att @ V
        x = self.ln1(tok + attended)
        x = self.ln2(x + self.ff(x))
        logits = self.out(x)
        return logits

# Entrenamiento
model = MiniTransformer().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
losses = []

for step in range(1000):
    x, y = get_batch()
    logits = model(x.unsqueeze(0))
    loss = F.cross_entropy(logits.squeeze(0)[:-1], y[:-1])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    if step % 100 == 0:
        print(f"Paso {step}, Pérdida: {loss.item():.4f}")

# Gráfico de pérdida
plt.plot(losses)
plt.title("Pérdida durante el entrenamiento")
plt.xlabel("Iteración")
plt.ylabel("Pérdida")
plt.grid(True)
plt.show()

# Generar texto con sampling
def generar_texto_sampling(prompt, max_len=20, temp=1.0):
    model.eval()
    x = torch.tensor([word2idx.get(w, 0) for w in prompt.lower().split()], device=device)
    for _ in range(max_len):
        x_cond = x[-context_size:] if len(x) >= context_size else F.pad(x, (context_size-len(x), 0))
        logits = model(x_cond.unsqueeze(0))[0, -1] / temp
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        x = torch.cat([x, next_token])
    return " ".join([idx2word[i.item()] for i in x])

# Prueba de generación
prompt = "una tortuga ganó"
generated = generar_texto_sampling(prompt)
print("Texto generado:\n", generated)

# Visualizar atención para prompt dado
def visualizar_atencion(prompt):
    model.eval()
    x = torch.tensor([word2idx.get(w, 0) for w in prompt.lower().split()], device=device)
    x_cond = x[-context_size:] if len(x) >= context_size else F.pad(x, (context_size-len(x), 0))
    _ = model(x_cond.unsqueeze(0))
    attn = model.attn_weights.squeeze(0)  # [T,T]
    plt.imshow(attn, cmap='Blues')
    plt.title(f"Atención para: '{prompt}'")
    plt.xlabel("Atiende a...")
    plt.ylabel("Palabra actual")
    plt.colorbar()
    plt.show()

visualizar_atencion("una tortuga ganó")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class EmbeddingNetwork(nn.Module):
    def __init__(self, input_dim, embed_dim):
        super().__init__()
        self.embed = nn.Linear(input_dim, embed_dim)

    def forward(self, x):
        return torch.tanh(self.embed(x))

class DecisionHead(nn.Module):
    def __init__(self, embed_dim, action_dim):
        super().__init__()
        self.fc1 = nn.Linear(embed_dim, 64)
        self.fc2 = nn.Linear(64, action_dim) # action_dim representa aquí el número de opciones/clases

    def forward(self, x):
        x = F.relu(self.fc1(x))
        return self.fc2(x)

class DecisionVotingNet(nn.Module):
    def __init__(self, input_dim, embed_dim, action_dim, num_heads=4):
        super().__init__()
        self.embedding = EmbeddingNetwork(input_dim, embed_dim)
        self.heads = nn.ModuleList([DecisionHead(embed_dim, action_dim) for _ in range(num_heads)])

    def forward(self, x):
        z = self.embedding(x)
        logits_list = [head(z) for head in self.heads]
        return logits_list

# Definir las posibles acciones/frases (esto es crucial para tu ejemplo)
# Deben corresponder con action_dim
POSSIBLE_PHRASES = [
    "durmiendo",
    "sonando",
    "comiendo",
    "jugando",
    "observando"
]

if __name__ == "__main__":
    action_dim = len(POSSIBLE_PHRASES) # El número de posibles acciones/frases
    model = DecisionVotingNet(input_dim=10, embed_dim=32, action_dim=action_dim, num_heads=4)
    dummy_input = torch.randn(1, 10) # Un solo ejemplo de entrada

    individual_logits = model(dummy_input)

    print("Generación (predicción) de cada cabeza:")
    for i, logits in enumerate(individual_logits):
        probabilities = F.softmax(logits, dim=-1) # Convertir logits a probabilidades

        # Encontrar la acción/frase con la probabilidad más alta
        predicted_index = torch.argmax(probabilities, dim=-1).item()
        predicted_phrase = POSSIBLE_PHRASES[predicted_index]
        confidence = probabilities[0, predicted_index].item() # Probabilidad de la frase elegida

        print(f"Cabeza {i+1}: El perro sigue {predicted_phrase} (Confianza: {confidence:.2f})")

    print(f"\nNúmero de cabezas (salidas): {len(individual_logits)}")

In [ ]:

# Crear archivo .ipynb para ejecución en Google Colab con todo el código incluido

from pathlib import Path

notebook_code = """
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 🧠 Transformer mínimo entrenado con cuentos\n",
    "Este notebook implementa un mini transformer de lenguaje natural en PyTorch entrenado sobre cuentos breves. Incluye:\n",
    "- Visualización de atención\n",
    "- Sampling con temperatura ajustable\n",
    "- Gráfico de pérdida para observar convergencia\n",
    "\n",
    "Puedes modificar los cuentos, probar distintas frases o cambiar la temperatura para ver cómo responde el modelo."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import torch\n",
    "import torch.nn as nn\n",
    "import torch.nn.functional as F\n",
    "import matplotlib.pyplot as plt\n",
    "import math, random\n",
    "\n",
    "# Dataset de cuentos\n",
    "corpus = [\n",
    "    \"Había una vez un pequeño dragón que vivía en la cima de una montaña.\",\n",
    "    \"El zorro engañó al cuervo para que cantara y dejara caer el queso.\",\n",
    "    \"El lobo sopló y sopló, pero no pudo derribar la casa de ladrillos.\",\n",
    "    \"Una tortuga ganó una carrera contra una liebre confiada y rápida.\"\n",
    "]\n",
    "\n",
    "# Tokenización\n",
    "tokens = \" \".join(corpus).lower().split()\n",
    "vocab = sorted(set(tokens))\n",
    "word2idx = {w: i for i, w in enumerate(vocab)}\n",
    "idx2word = {i: w for w, i in word2idx.items()}\n",
    "data = [word2idx[w] for w in tokens]\n",
    "\n",
    "# Config\n",
    "d_model = 32\n",
    "context_size = 6\n",
    "vocab_size = len(vocab)\n",
    "device = 'cuda' if torch.cuda.is_available() else 'cpu'\n",
    "\n",
    "# Batch simple\n",
    "def get_batch():\n",
    "    ix = random.randint(0, len(data) - context_size - 1)\n",
    "    x = torch.tensor(data[ix:ix+context_size], device=device)\n",
    "    y = torch.tensor(data[ix+1:ix+context_size+1], device=device)\n",
    "    return x, y\n",
    "\n",
    "# Modelo con visualización de atención\n",
    "class MiniTransformer(nn.Module):\n",
    "    def __init__(self):\n",
    "        super().__init__()\n",
    "        self.token_embed = nn.Embedding(vocab_size, d_model)\n",
    "        self.pos_embed = nn.Parameter(torch.randn(context_size, d_model))\n",
    "        self.q_proj = nn.Linear(d_model, d_model)\n",
    "        self.k_proj = nn.Linear(d_model, d_model)\n",
    "        self.v_proj = nn.Linear(d_model, d_model)\n",
    "        self.ff = nn.Sequential(\n",
    "            nn.Linear(d_model, d_model),\n",
    "            nn.ReLU(),\n",
    "            nn.Linear(d_model, d_model)\n",
    "        )\n",
    "        self.ln1 = nn.LayerNorm(d_model)\n",
    "        self.ln2 = nn.LayerNorm(d_model)\n",
    "        self.out = nn.Linear(d_model, vocab_size)\n",
    "        self.attn_weights = None  # Para visualización\n",
    "\n",
    "    def forward(self, x):\n",
    "        tok = self.token_embed(x)\n",
    "        tok = tok + self.pos_embed\n",
    "        Q = self.q_proj(tok)\n",
    "        K = self.k_proj(tok)\n",
    "        V = self.v_proj(tok)\n",
    "\n",
    "        att = (Q @ K.transpose(-2, -1)) / math.sqrt(Q.size(-1))\n",
    "        mask = torch.tril(torch.ones(context_size, context_size, device=device))\n",
    "        att = att.masked_fill(mask == 0, float('-inf'))\n",
    "        att = F.softmax(att, dim=-1)\n",
    "        self.attn_weights = att.detach().cpu()\n",
    "\n",
    "        attended = att @ V\n",
    "        x = self.ln1(tok + attended)\n",
    "        x = self.ln2(x + self.ff(x))\n",
    "        logits = self.out(x)\n",
    "        return logits\n",
    "\n",
    "# Entrenamiento\n",
    "model = MiniTransformer().to(device)\n",
    "optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)\n",
    "losses = []\n",
    "\n",
    "for step in range(1000):\n",
    "    x, y = get_batch()\n",
    "    logits = model(x.unsqueeze(0))\n",
    "    loss = F.cross_entropy(logits.squeeze(0)[:-1], y[:-1])\n",
    "    optimizer.zero_grad()\n",
    "    loss.backward()\n",
    "    optimizer.step()\n",
    "    losses.append(loss.item())\n",
    "    if step % 100 == 0:\n",
    "        print(f\"Paso {step}, Pérdida: {loss.item():.4f}\")\n",
    "\n",
    "# Gráfico de pérdida\n",
    "plt.plot(losses)\n",
    "plt.title(\"Pérdida durante el entrenamiento\")\n",
    "plt.xlabel(\"Iteración\")\n",
    "plt.ylabel(\"Pérdida\")\n",
    "plt.grid(True)\n",
    "plt.show()\n",
    "\n",
    "# Generar texto con sampling\n",
    "def generar_texto_sampling(prompt, max_len=20, temp=1.0):\n",
    "    model.eval()\n",
    "    x = torch.tensor([word2idx.get(w, 0) for w in prompt.lower().split()], device=device)\n",
    "    for _ in range(max_len):\n",
    "        x_cond = x[-context_size:] if len(x) >= context_size else F.pad(x, (context_size-len(x), 0))\n",
    "        logits = model(x_cond.unsqueeze(0))[0, -1] / temp\n",
    "        probs = F.softmax(logits, dim=-1)\n",
    "        next_token = torch.multinomial(probs, num_samples=1)\n",
    "        x = torch.cat([x, next_token])\n",
    "    return \" \".join([idx2word[i.item()] for i in x])\n",
    "\n",
    "# Prueba de generación\n",
    "prompt = \"una tortuga ganó\"\n",
    "generated = generar_texto_sampling(prompt, temp=1.0)\n",
    "print(\"Texto generado:\\n\", generated)\n",
    "\n",
    "# Visualizar atención para prompt dado\n",
    "def visualizar_atencion(prompt):\n",
    "    model.eval()\n",
    "    x = torch.tensor([word2idx.get(w, 0) for w in prompt.lower().split()], device=device)\n",
    "    x_cond = x[-context_size:] if len(x) >= context_size else F.pad(x, (context_size-len(x), 0))\n",
    "    _ = model(x_cond.unsqueeze(0))\n",
    "    attn = model.attn_weights.squeeze(0)\n",
    "    plt.imshow(attn, cmap='Blues')\n",
    "    plt.title(f\"Atención para: '{prompt}'\")\n",
    "    plt.xlabel(\"Atiende a...\")\n",
    "    plt.ylabel(\"Palabra actual\")\n",
    "    plt.colorbar()\n",
    "    plt.show()\n",
    "\n",
    "visualizar_atencion(\"una tortuga ganó\")"
   ]Y
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.8"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 2
}
"""

notebook_path = "/mnt/data/Transformer_Cuentos.ipynb"
Path(notebook_path).write_text(notebook_code)
notebook_path

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math, random

# 1. Dataset de ejemplo (cuentos cortos)
corpus = [
    "Había una vez un pequeño dragón que vivía en la cima de una montaña.",
    "El zorro engañó al cuervo para que cantara y dejara caer el queso.",
    "El lobo sopló y sopló, pero no pudo derribar la casa de ladrillos.",
    "Una tortuga ganó una carrera contra una liebre confiada y rápida."
]

# Tokenización simple
tokens = " ".join(corpus).lower().split()
vocab = sorted(set(tokens))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

# Secuencia como números
data = [word2idx[w] for w in tokens]

# Hiperparámetros
d_model = 32
context_size = 5
vocab_size = len(vocab)
num_heads = 16# Nueva: para Multi-Head Attention
num_layers = 12 # Nueva: para bloques del Transformer
batch_size = 8 # Nueva: para el manejo de batches
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Dataset para entrenamiento (inputs y targets)
def get_batch():
    # Nueva: obtiene un batch de secuencias
    batch_x, batch_y = [], []
    for _ in range(batch_size):
        ix = random.randint(0, len(data) - context_size - 1)
        x = torch.tensor(data[ix:ix+context_size], device=device)
        y = torch.tensor(data[ix+1:ix+context_size+1], device=device)
        batch_x.append(x)
        batch_y.append(y)
    return torch.stack(batch_x), torch.stack(batch_y)

# Nueva: Capa de Positional Encoding (2)
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

# Nueva: Multi-Head Attention (3)
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        batch_size, seq_len, d_model = x.size()

        Q = self.q_proj(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.k_proj(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.v_proj(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        # Causal mask
        mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device)).bool()
        scores = scores.masked_fill(mask == 0, float('-inf'))

        attention = F.softmax(scores, dim=-1)
        attended_values = torch.matmul(attention, V)

        attended_values = attended_values.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
        output = self.out_proj(attended_values)
        return output

# Nueva: Bloque del Transformer (4)
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.attn = MultiHeadSelfAttention(d_model, num_heads)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 4), # Generalmente se usa un factor de 4
            nn.ReLU(),
            nn.Linear(d_model * 4, d_model)
        )
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = self.ln1(x + self.attn(x))
        x = self.ln2(x + self.ff(x))
        return x

# 2. Modelo Transformer minimalista (con mejoras 2, 3, 4)
class MiniTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = PositionalEncoding(d_model, context_size) # Usamos la nueva PE
        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads) for _ in range(num_layers)
        ])
        self.out = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        # x tiene ahora la forma [B, T]
        tok = self.token_embed(x)  # [B, T, D]
        x = self.pos_embed(tok)    # [B, T, D]

        for block in self.transformer_blocks:
            x = block(x)

        logits = self.out(x) # [B, T, V]
        return logits

# 3. Entrenamiento
model = MiniTransformer().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

# Nueva: Dataset de validación para evaluación (8)
val_corpus = [
    "un pequeño conejo saltaba feliz por el campo",
    "el sol brillaba y los pájaros cantaban una canción"
]
val_tokens = " ".join(val_corpus).lower().split()
val_data = [word2idx.get(w, 0) for w in val_tokens] # Usar 0 para OOV, o mejor un token UNK

def get_val_batch():
    # Similar a get_batch, pero para el conjunto de validación
    ix = random.randint(0, len(val_data) - context_size - 1)
    x = torch.tensor(val_data[ix:ix+context_size], device=device)
    y = torch.tensor(val_data[ix+1:ix+context_size+1], device=device)
    return x.unsqueeze(0), y.unsqueeze(0) # Para simplificar, un batch de 1 para validación

# Función para calcular perplejidad (8)
def calculate_perplexity(model, data_loader):
    model.eval()
    total_loss = 0
    total_tokens = 0
    with torch.no_grad():
        for _ in range(50): # Calcular perplejidad sobre 50 mini-batches de validación
            x, y = get_val_batch() # Usamos get_val_batch
            logits = model(x)
            # Aplanar para cross_entropy: [B*T, V] y [B*T]
            loss = F.cross_entropy(logits[:, :-1, :].reshape(-1, vocab_size), y[:, :-1].reshape(-1))
            total_loss += loss.item() * (y.size(0) * (y.size(1) - 1)) # Multiplicar por número de elementos predichos
            total_tokens += (y.size(0) * (y.size(1) - 1))
    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)
    model.train()
    return perplexity

for step in range(1000):
    x, y = get_batch() # Nueva: ahora get_batch devuelve un batch
    logits = model(x)  # x ya es [B, T], logits será [B, T, V]

    # Ajustar loss para batch:
    # logits[:, :-1, :] toma todos los elementos del batch, todas las secuencias menos el último token
    # y[:, 1:] toma todos los elementos del batch, todas las secuencias a partir del segundo token
    loss = F.cross_entropy(logits[:, :-1, :].reshape(-1, vocab_size), y[:, 1:].reshape(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"Paso {step}, Pérdida: {loss.item():.4f}")
        # Nueva: Evaluación de perplejidad (8)
        perp = calculate_perplexity(model, None) # data_loader no se usa directamente aquí
        print(f"Perplejidad de validación: {perp:.2f}")

# Nueva: Guardar el modelo (9)
model_path = "mini_transformer.pth"
torch.save(model.state_dict(), model_path)
print(f"Modelo guardado en {model_path}")

# 4. Generación de texto
def generar_texto(prompt, max_len=20, model=None): # Añadir argumento model con valor por defecto None
    # Usar el modelo pasado como argumento, o el modelo global si no se pasa ninguno
    current_model = model if model is not None else globals().get('model')

    if current_model is None:
        raise ValueError("No se proporcionó un modelo ni existe un modelo global.")

    current_model.eval()
    # Nueva: Cargar el modelo si no está en eval
    # Para este ejemplo, lo cargamos justo antes de generar
    # Si quieres cargar un modelo guardado previamente:
    # loaded_model = MiniTransformer().to(device)
    # loaded_model.load_state_dict(torch.load(model_path, weights_only=True))
    # loaded_model.eval()
    # model = loaded_model # Usa el modelo cargado

    x = torch.tensor([word2idx.get(w, 0) for w in prompt.lower().split()], device=device)
    for _ in range(max_len):
        # Aseguramos que la secuencia de entrada sea de tamaño context_size
        x_cond = x[-context_size:] if len(x) >= context_size else F.pad(x, (context_size-len(x), 0), 'constant', 0)

        # Necesitamos un batch dimension para el modelo [1, T]
        logits = current_model(x_cond.unsqueeze(0)) # Usar current_model

        # Tomar la última palabra del output y predecir la siguiente
        next_token_logits = logits[0, -1, :] # [V]
        next_token = torch.argmax(next_token_logits)

        x = torch.cat([x, next_token.unsqueeze(0)])
    return " ".join([idx2word[i.item()] for i in x])

# Ejemplo
print("Generado:", generar_texto("había una vez un"))

# Nueva: Cargar un modelo guardado para demostración (9)
print("\nDemostración de carga de modelo:")
loaded_model = MiniTransformer().to(device)
loaded_model.load_state_dict(torch.load(model_path, weights_only=True))
loaded_model.eval()
# Eliminar la línea problemática
# /content/mini_transformer.pth # <--- Eliminar esta línea

print("Generado (desde modelo cargado):", generar_texto("el lobo sopló y", model=loaded_model)) # Pasamos el modelo cargado

In [ ]:
# Correr en Google Colab
import torch
import torch.nn as nn
import torch.nn.functional as F
import math, random
import os # Para manejar archivos

# 1. Dataset de ejemplo (cuentos cortos)
corpus = [
    "Había una vez un pequeño dragón que vivía en la cima de una montaña.",
    "El zorro engañó al cuervo para que cantara y dejara caer el queso.",
    "El lobo sopló y sopló, pero no pudo derribar la casa de ladrillos.",
    "Una tortuga ganó una carrera contra una liebre confiada y rápida."
]

# Tokenización simple
tokens = " ".join(corpus).lower().split()
vocab = sorted(set(tokens))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

# Secuencia como números
data = [word2idx[w] for w in tokens]

# Hiperparámetros
d_model = 32
context_size = 6
vocab_size = len(vocab)
num_heads = 4 # Nueva: para Multi-Head Attention
num_layers = 2 # Nueva: para bloques del Transformer
batch_size = 8 # Nueva: para el manejo de batches
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Dataset para entrenamiento (inputs y targets)
def get_batch():
    # Nueva: obtiene un batch de secuencias
    batch_x, batch_y = [], []
    for _ in range(batch_size):
        # Asegurarse de que hay suficientes tokens para formar una secuencia de context_size + 1
        if len(data) - context_size - 1 < 0:
            raise ValueError("Corpus demasiado pequeño para el context_size dado.")
        ix = random.randint(0, len(data) - context_size - 1)
        x = torch.tensor(data[ix:ix+context_size], device=device)
        y = torch.tensor(data[ix+1:ix+context_size+1], device=device)
        batch_x.append(x)
        batch_y.append(y)
    return torch.stack(batch_x), torch.stack(batch_y)

# Nueva: Capa de Positional Encoding (mejora 2)
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0) # Añadir dimensión de batch para que sea [1, max_len, d_model]
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x tiene forma [B, T, D]
        # Sumar el positional encoding adecuado a cada secuencia en el batch
        return x + self.pe[:, :x.size(1)]

# Nueva: Multi-Head Attention (mejora 3)
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model debe ser divisible por num_heads"
        self.d_k = d_model // num_heads # Dimensión de cada cabeza
        self.num_heads = num_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        batch_size, seq_len, d_model = x.size()

        # Proyectar y dividir en cabezas
        Q = self.q_proj(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2) # [B, nh, T, dk]
        K = self.k_proj(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2) # [B, nh, T, dk]
        V = self.v_proj(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2) # [B, nh, T, dk]

        # Calcular scores de atención
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k) # [B, nh, T, T]

        # Aplicar máscara causal para que las palabras solo atiendan a las anteriores
        mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device)).bool()
        scores = scores.masked_fill(mask == 0, float('-inf'))

        attention = F.softmax(scores, dim=-1) # [B, nh, T, T]
        attended_values = torch.matmul(attention, V) # [B, nh, T, dk]

        # Concatenar cabezas y proyectar
        attended_values = attended_values.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
        output = self.out_proj(attended_values)
        return output

# Nueva: Bloque del Transformer (mejora 4)
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.attn = MultiHeadSelfAttention(d_model, num_heads)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 4), # Generalmente se usa un factor de 4 para la capa oculta
            nn.ReLU(),
            nn.Linear(d_model * 4, d_model)
        )
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x):
        # Aplicar LayerNorm y residual connection
        x = x + self.attn(self.ln1(x)) # Primero LayerNorm, luego atención, luego residual
        x = x + self.ff(self.ln2(x))   # Primero LayerNorm, luego FF, luego residual
        return x

# 2. Modelo Transformer completo (con mejoras 2, 3, 4)
class MiniTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = PositionalEncoding(d_model, context_size) # Usamos la nueva PE

        # Apilar múltiples bloques del Transformer
        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads) for _ in range(num_layers)
        ])

        self.ln_f = nn.LayerNorm(d_model) # LayerNorm final antes de la capa de salida
        self.out = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        # x tiene ahora la forma [B, T]
        tok = self.token_embed(x)  # [B, T, D]
        x = self.pos_embed(tok)    # [B, T, D]

        for block in self.transformer_blocks:
            x = block(x)

        x = self.ln_f(x) # Aplicar LayerNorm final
        logits = self.out(x) # [B, T, V]
        return logits

# 3. Entrenamiento
model = MiniTransformer().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

# Nueva: Dataset de validación para evaluación (mejora 8)
val_corpus = [
    "un pequeño conejo saltaba feliz por el campo",
    "el sol brillaba y los pájaros cantaban una canción"
]
# Asegurarse de que las palabras del corpus de validación existen en el vocabulario
# Si no, se usaría 0 (el índice de un token desconocido si se hubiera definido)
val_data = [word2idx.get(w, 0) for w in val_corpus if w in word2idx]
# Si el val_data resultante es muy corto, podría causar errores.
# Para este ejemplo, asumimos que habrá suficientes palabras comunes.

def get_val_batch():
    # Similar a get_batch, pero para el conjunto de validación
    # Asegurarse de que hay suficientes tokens para formar una secuencia de context_size + 1
    if len(val_data) - context_size - 1 < 0:
        # Si el corpus de validación es demasiado pequeño, devolver un tensor vacío o manejar el error
        # Para este ejemplo, devolveremos un tensor pequeño para evitar errores si es posible
        if len(val_data) >= context_size + 1:
            ix = 0
        else:
            print("Advertencia: El corpus de validación es demasiado pequeño para formar un batch completo.")
            # Crear un batch mínimo para evitar errores, pero la perplejidad no será significativa
            x = torch.tensor(val_data[:min(len(val_data), context_size)], device=device)
            y = torch.tensor(val_data[1:min(len(val_data), context_size + 1)], device=device)
            return x.unsqueeze(0), y.unsqueeze(0)

    ix = random.randint(0, len(val_data) - context_size - 1)
    x = torch.tensor(val_data[ix:ix+context_size], device=device)
    y = torch.tensor(val_data[ix+1:ix+context_size+1], device=device)
    return x.unsqueeze(0), y.unsqueeze(0) # Un batch de 1 para validación simple

# Función para calcular perplejidad (mejora 8)
def calculate_perplexity(model):
    model.eval() # Poner el modelo en modo evaluación
    total_loss = 0
    total_tokens = 0
    num_val_batches = 50 # Número de mini-batches de validación para calcular la perplejidad

    with torch.no_grad():
        for _ in range(num_val_batches):
            x_val, y_val = get_val_batch()

            # Si el batch de validación es vacío o inválido, saltar
            if x_val.numel() == 0 or y_val.numel() == 0:
                continue

            logits_val = model(x_val)

            # Asegurarse de que las dimensiones coinciden para cross_entropy
            # logits_val[:, :-1, :] -> [B, T-1, V]
            # y_val[:, 1:] -> [B, T-1]
            loss = F.cross_entropy(logits_val[:, :-1, :].reshape(-1, vocab_size), y_val[:, 1:].reshape(-1))

            # Acumular la pérdida multiplicada por el número de tokens predichos en este batch
            num_predicted_tokens = y_val.size(0) * (y_val.size(1) - 1)
            total_loss += loss.item() * num_predicted_tokens
            total_tokens += num_predicted_tokens

    model.train() # Volver el modelo a modo entrenamiento

    if total_tokens == 0:
        return float('inf') # Evitar división por cero si no hay tokens de validación

    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)
    return perplexity

num_steps = 2000 # Aumentar pasos de entrenamiento por ser un modelo más complejo
for step in range(num_steps):
    x, y = get_batch() # get_batch devuelve [B, T]
    logits = model(x)  # logits será [B, T, V]

    # Ajustar cálculo de pérdida para batches:
    # logits[:, :-1, :] toma todos los elementos del batch, todas las secuencias menos el último token
    # y[:, 1:] toma todos los elementos del batch, todas las secuencias a partir del segundo token
    loss = F.cross_entropy(logits[:, :-1, :].reshape(-1, vocab_size), y[:, 1:].reshape(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"Paso {step}, Pérdida de entrenamiento: {loss.item():.4f}")
        # Nueva: Evaluación de perplejidad (mejora 8)
        perp = calculate_perplexity(model)
        print(f"Perplejidad de validación: {perp:.2f}")

# Nueva: Guardar el modelo (mejora 9)
model_path = "mini_transformer.pth"
torch.save(model.state_dict(), model_path)
print(f"Modelo guardado en {model_path}")

# 4. Generación de texto
def generar_texto(prompt, max_len=20, model_to_use=None): # Renombrado para evitar conflicto con el global
    # Usar el modelo pasado como argumento, o el modelo global si no se pasa ninguno
    current_model = model_to_use if model_to_use is not None else globals().get('model')

    if current_model is None:
        raise ValueError("No se proporcionó un modelo ni existe un modelo global.")

    current_model.eval() # Poner el modelo en modo evaluación

    # Convertir el prompt a números
    input_tokens = [word2idx.get(w, 0) for w in prompt.lower().split()]
    x = torch.tensor(input_tokens, device=device).unsqueeze(0) # Añadir dimensión de batch [1, T_prompt]

    for _ in range(max_len):
        # Aseguramos que la secuencia de entrada para el modelo sea de tamaño context_size
        # x es [1, current_seq_len]
        if x.size(1) < context_size:
            padding_needed = context_size - x.size(1)
            # Rellenar con el token 0 (o un token UNK/PAD si existiera)
            x_cond = F.pad(x, (0, padding_needed), 'constant', 0) # Rellenar al final para la entrada
        else:
            x_cond = x[:, -context_size:] # Tomar los últimos 'context_size' tokens

        logits = current_model(x_cond) # logits: [1, T_cond, vocab_size]

        # Tomar los logits del último token predicho en la secuencia
        next_token_logits = logits[0, -1, :] # [vocab_size]
        next_token_probs = F.softmax(next_token_logits, dim=-1) # Probabilidades del último token

        # Muestreo para mayor diversidad (en lugar de argmax determinista)
        next_token = torch.multinomial(next_token_probs, num_samples=1) # [1]

        x = torch.cat([x, next_token.unsqueeze(0)], dim=1) # Concatenar el nuevo token

    return " ".join([idx2word[i.item()] for i in x.squeeze(0)])

# Ejemplo de generación con el modelo entrenado
print("\nGenerado (con el modelo entrenado):", generar_texto("había una vez un"))

# Nueva: Cargar un modelo guardado para demostración (mejora 9)
print("\n--- Demostración de carga de modelo ---")
loaded_model = MiniTransformer().to(device)
loaded_model.load_state_dict(torch.load(model_path, weights_only=True))
loaded_model.eval() # Poner el modelo cargado en modo evaluación

print("Generado (desde modelo cargado):", generar_texto("el lobo sopló y", model_to_use=loaded_model))

In [ ]:


import imageio

In [ ]:
!pip install torch

#Auto-Optimización por Sacarina

1. Estado s
2. Acción a
3. Recompensa r
4. Transición T(s, a)
5. Meta final

6. Criterios de éxito y exploración
5. RL Autoimplosivo ("RL Entropía Negativa")

🧬 Concepto:

El agente comienza con una red neuronal grande. Cada acción le cuesta un trozo de su red. Debe resolver tareas mientras se auto-descompone.

🧪 Mecánica:

Cada decisión requiere apagar nodos o capas enteras.

Recompensa si logra seguir resolviendo tareas con menor estructura.

Penalización si se vuelve inútil demasiado pronto.


🔍 Objetivo:

Estudiar eficiencia mínima, plasticidad y adaptación extrema.

🧩 Único porque:

Es un RL que reduce sus propios grados de libertad como precio por actuar. Al final solo queda un agente degenerado… o optimizado.

## Entorno visual multicuerpo

In [ ]:

!apt-get install -y xvfb x11-utils
!pip install pygame pyvirtualdisplay

from pyvirtualdisplay import Display
display = Display(visible=0, size=(640, 480))
display.start()

In [ ]:
import pygame
import os

class VisualMultiBodyEnv(MultiBodyCubePushEnv):
    def __init__(self, render_interval=24, save_dir="frames"):
        super().__init__()
        self.render_interval = render_interval
        self.save_dir = save_dir
        self.frame_count = 0

        # Inicializar pygame en modo headless
        os.makedirs(self.save_dir, exist_ok=True)
        pygame.init()
        self.screen_size = 480
        self.scale = 100
        self.screen = pygame.Surface((self.screen_size, self.screen_size))

    def draw_circle(self, pos, color, radius=5):
        p = ((pos + 1) * 0.5 * self.screen_size).astype(int)
        pygame.draw.circle(self.screen, color, p, radius)

    def render(self):
        self.screen.fill((255, 255, 255))  # Fondo blanco

        # Dibujar objetos
        self.draw_circle(self.body1_pos, (255, 0, 0), radius=8)     # Rojo
        self.draw_circle(self.body2_pos, (0, 255, 0), radius=8)     # Verde
        self.draw_circle(self.cube_pos, (0, 0, 255), radius=10)     # Azul
        self.draw_circle(self.goal_pos, (255, 165, 0), radius=6)    # Naranja

        # Guardar imagen cada 24 pasos
        if self.frame_count % self.render_interval == 0:
            path = os.path.join(self.save_dir, f"frame_{self.frame_count:04d}.png")
            pygame.image.save(self.screen, path)
            print(f"🖼️ Saved {path}")

        self.frame_count += 1

In [ ]:
env = VisualMultiBodyEnv()
obs, _ = env.reset()

done = False
while not done:
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    env.render()

    if terminated or truncated:
        break

In [ ]:
import imageio.v2 as imageio
import os

images = []
folder = "frames"
for i in sorted(os.listdir(folder)):
    if i.endswith(".png"):
        images.append(imageio.imread(os.path.join(folder, i)))

imageio.mimsave("cube_push.gif", images, fps=10)

In [ ]:
from IPython.display import Image
Image(filename="cube_push.gif")

## Empujar cubos con varios cuerpos (RL)

In [ ]:
# --- 0. Instalación y Configuración Inicial (Necesario en Colab) ---
# Si aún no lo has ejecutado:
!apt-get install -y xvfb x11-utils > /dev/null 2>&1
!pip install pygame pyvirtualdisplay > /dev/null 2>&1
!pip install imageio-ffmpeg > /dev/null 2>&1 # Necesario para guardar GIF/MP4

from pyvirtualdisplay import Display
import os

# Iniciar un display virtual para pygame
# print("Iniciando display virtual...")
# display = Display(visible=0, size=(640, 480))
# display.start()
# print("Display virtual iniciado.")

import pygame
import numpy as np
import gymnasium as gym
from gymnasium import spaces
import matplotlib.pyplot as plt # Mantener para depuración si es necesario
import imageio.v2 as imageio # Importar imageio.v2 explícitamente para evitar advertencias
import time # Para pausas en el renderizado


# --- 1. Tu Entorno MultiBodyCubePushEnv (Copiado de tu código anterior) ---
# Es importante que el entorno base esté correctamente definido
class MultiBodyCubePushEnv(gym.Env):
    """Entorno simple donde una mente controla dos cuerpos que deben empujar un cubo hacia una meta."""
    metadata = {"render_modes": ["human"], "render_fps": 4} # Añadido render_fps para gymnasium

    def __init__(self, render_mode=None): # Añadir render_mode al constructor
        super().__init__()
        self.grid_size = 10
        self.render_mode = render_mode # Almacenar el render_mode

        # Espacio de observación: posiciones de cuerpo1, cuerpo2, cubo y meta (8 valores)
        self.observation_space = spaces.Box(low=0, high=self.grid_size - 1, shape=(8,), dtype=np.int32)

        # Espacio de acción: cada cuerpo puede moverse en 5 direcciones (0=arriba, 1=abajo, 2=izq, 3=der, 4=no mover)
        self.action_space = spaces.MultiDiscrete([5, 5])

        # Posiciones de los elementos (se inicializan en reset)
        self.body1_pos = None
        self.body2_pos = None
        self.cube_pos = None
        self.goal_pos = np.array([self.grid_size - 1, self.grid_size - 1]) # La meta es fija en la esquina

        # Atributos para Pygame render (solo si se usa render_mode='human')
        self.screen = None
        self.clock = None
        self.cell_size = None # Tamaño en píxeles de cada celda

    def _get_obs(self):
        return np.concatenate([
            self.body1_pos, self.body2_pos, self.cube_pos, self.goal_pos
        ]).astype(np.int32)

    def _get_info(self):
        return {"distance_to_goal": np.linalg.norm(self.cube_pos - self.goal_pos)}

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        # Posiciones iniciales aleatorias
        self.body1_pos = self.np_random.integers(0, self.grid_size, size=2)
        self.body2_pos = self.np_random.integers(0, self.grid_size, size=2)
        self.cube_pos = self.np_random.integers(0, self.grid_size, size=2)

        # Asegurarse de que las posiciones iniciales no se superpongan
        # y que el cubo no inicie en la meta
        while np.array_equal(self.body1_pos, self.body2_pos) or \
              np.array_equal(self.body1_pos, self.cube_pos) or \
              np.array_equal(self.body2_pos, self.cube_pos) or \
              np.array_equal(self.cube_pos, self.goal_pos):
            self.body1_pos = self.np_random.integers(0, self.grid_size, size=2)
            self.body2_pos = self.np_random.integers(0, self.grid_size, size=2)
            self.cube_pos = self.np_random.integers(0, self.grid_size, size=2)

        observation = self._get_obs()
        info = self._get_info()

        if self.render_mode == "human":
            self._render_frame()

        return observation, info

    def step(self, action):
        action_body1, action_body2 = action

        def apply_move(pos, a):
            new_pos = pos.copy()
            if a == 0: new_pos[1] -= 1  # Arriba
            elif a == 1: new_pos[1] += 1  # Abajo
            elif a == 2: new_pos[0] -= 1  # Izquierda
            elif a == 3: new_pos[0] += 1  # Derecha
            # Si a == 4, no se mueve

            new_pos = np.clip(new_pos, 0, self.grid_size - 1)
            return new_pos

        # Mover cuerpos a sus nuevas posiciones propuestas
        proposed_body1_pos = apply_move(self.body1_pos, action_body1)
        proposed_body2_pos = apply_move(self.body2_pos, action_body2)

        # Lógica de empuje del cubo
        # El cubo se mueve si:
        # 1. Ambos cuerpos están adyacentes al cubo.
        # 2. Ambos cuerpos intentan empujar en la misma dirección general del cubo.

        can_body1_push = np.array_equal(proposed_body1_pos, self.cube_pos)
        can_body2_push = np.array_equal(proposed_body2_pos, self.cube_pos)

        # Delta de acciones a movimiento (para entender la dirección intentada)
        action_to_delta = {
            0: np.array([0,-1]), # Arriba
            1: np.array([0,1]),  # Abajo
            2: np.array([-1,0]), # Izquierda
            3: np.array([1,0]),  # Derecha
            4: np.array([0,0])   # No mover
        }
        delta_action_b1 = action_to_delta[action_body1]
        delta_action_b2 = action_to_delta[action_body2]

        # Calculamos la posición futura del cubo si se empujara
        new_cube_pos = self.cube_pos.copy()

        # Un empuje solo ocurre si AMBOS cuerpos estan adyacentes al cubo en la dirección de su movimiento intencionado
        # Y AMBOS cuerpos intentan moverse en la dirección que empujaría el cubo

        # Consideramos empuje en X
        if can_body1_push and can_body2_push and \
           delta_action_b1[0] != 0 and delta_action_b1[0] == delta_action_b2[0]: # Si ambos empujan horizontalmente en la misma dirección
            # Y el cubo no está en el límite
            if not (self.cube_pos[0] == 0 and delta_action_b1[0] == -1) and \
               not (self.cube_pos[0] == self.grid_size -1 and delta_action_b1[0] == 1):
                new_cube_pos[0] += delta_action_b1[0] # El cubo se mueve un paso

        # Consideramos empuje en Y
        if can_body1_push and can_body2_push and \
           delta_action_b1[1] != 0 and delta_action_b1[1] == delta_action_b2[1]: # Si ambos empujan verticalmente en la misma dirección
            # Y el cubo no está en el límite
            if not (self.cube_pos[1] == 0 and delta_action_b1[1] == -1) and \
               not (self.cube_pos[1] == self.grid_size -1 and delta_action_b1[1] == 1):
                new_cube_pos[1] += delta_action_b1[1] # El cubo se mueve un paso

        # Si el cubo se movió, y los cuerpos están en la nueva posición del cubo, los cuerpos se mueven con él
        if not np.array_equal(new_cube_pos, self.cube_pos):
            # Aseguramos que los cuerpos se muevan a la posición del cubo si lo empujaron
            # Esto asume que el cuerpo "ocupa" la celda anterior del cubo
            self.body1_pos = self.cube_pos # Asumimos que los cuerpos "siguen" al cubo
            self.body2_pos = self.cube_pos # Esto es una simplificación de la física de empuje

            self.cube_pos = new_cube_pos
        else:
            # Si el cubo no se movió, los cuerpos simplemente se mueven a sus posiciones propuestas
            self.body1_pos = proposed_body1_pos
            self.body2_pos = proposed_body2_pos

        # Clip para asegurar que todo esté dentro de los límites
        self.body1_pos = np.clip(self.body1_pos, 0, self.grid_size - 1)
        self.body2_pos = np.clip(self.body2_pos, 0, self.grid_size - 1)
        self.cube_pos = np.clip(self.cube_pos, 0, self.grid_size - 1)


        reward = 0.0
        terminated = False
        truncated = False # Por ahora no hay truncamiento por límites de tiempo/pasos

        # Recompensa por alcanzar la meta
        if np.array_equal(self.cube_pos, self.goal_pos):
            reward = 10.0 # Recompensa positiva grande
            terminated = True # El episodio termina

        # Recompensa de progreso (negativa si se aleja, positiva si se acerca)
        # Esto ayuda al agente a aprender incluso antes de llegar a la meta
        # (Esto asume que _get_info se llama con la pos_anterior en un state previo, o se guarda)
        old_distance = np.linalg.norm(np.array(self._get_info()["distance_to_goal_before_move"])) # Necesitamos guardar la distancia anterior
        current_distance = np.linalg.norm(self.cube_pos - self.goal_pos)

        # Una recompensa de progreso simple: -1 por paso, +0.1 por cada unidad de distancia que se acerca a la meta
        reward = -0.1 # Pequeña penalización por cada paso (para animar a terminar rápido)
        reward += (old_distance - current_distance) * 0.5 # Más recompensa por acercarse a la meta

        # Si el cubo alcanza la meta, la recompensa es la de meta, no la de progreso
        if terminated:
            reward = 10.0

        observation = self._get_obs()
        info = self._get_info()
        # Actualizamos la info con la distancia actual para el cálculo de progreso en el siguiente paso
        info["distance_to_goal_before_move"] = current_distance


        if self.render_mode == "human":
            self._render_frame()

        return observation, reward, terminated, truncated, info

    def _render_frame(self):
        # Asegurarse de que Pygame esté inicializado y la pantalla creada
        if self.screen is None:
            pygame.init()
            # self.screen_size = self.grid_size * 50 # Cada celda de 50px
            self.cell_size = 50
            self.screen_size_px = self.grid_size * self.cell_size
            self.screen = pygame.display.set_mode((self.screen_size_px, self.screen_size_px))
            pygame.display.set_caption("Multi-Body Cube Push Env")
            self.clock = pygame.time.Clock() # Para controlar los FPS

        # Colores
        WHITE = (255, 255, 255)
        BLACK = (0, 0, 0)
        RED = (255, 0, 0) # Cuerpo 1
        GREEN = (0, 255, 0) # Cuerpo 2
        BLUE = (0, 0, 255) # Cubo
        ORANGE = (255, 165, 0) # Meta

        self.screen.fill(WHITE) # Fondo blanco

        # Dibujar la cuadrícula
        for x in range(self.grid_size):
            for y in range(self.grid_size):
                rect = pygame.Rect(x * self.cell_size, y * self.cell_size, self.cell_size, self.cell_size)
                pygame.draw.rect(self.screen, BLACK, rect, 1) # Borde de celda

        # Función para dibujar objetos en el centro de la celda
        def draw_object(pos_grid, color, shape="circle", radius_ratio=0.8):
            center_x = (pos_grid[0] * self.cell_size) + (self.cell_size / 2)
            center_y = (pos_grid[1] * self.cell_size) + (self.cell_size / 2)
            radius = int(self.cell_size / 2 * radius_ratio)
            if shape == "circle":
                pygame.draw.circle(self.screen, color, (center_x, center_y), radius)
            elif shape == "rect":
                size = int(self.cell_size * radius_ratio)
                rect_pos_x = center_x - size / 2
                rect_pos_y = center_y - size / 2
                pygame.draw.rect(self.screen, color, (rect_pos_x, rect_pos_y, size, size))


        # Dibujar objetos
        draw_object(self.goal_pos, ORANGE, shape="circle", radius_ratio=0.6) # Meta (más pequeña)
        draw_object(self.cube_pos, BLUE, shape="rect", radius_ratio=0.8) # Cubo (rectángulo)
        draw_object(self.body1_pos, RED, shape="circle", radius_ratio=0.8) # Cuerpo 1
        draw_object(self.body2_pos, GREEN, shape="circle", radius_ratio=0.8) # Cuerpo 2

        pygame.event.pump() # Permite a Pygame manejar eventos internos
        pygame.display.flip() # Actualiza la pantalla

        if self.render_mode == "human" and self.clock:
            self.clock.tick(self.metadata["render_fps"]) # Controlar los FPS

    def close(self):
        if self.screen is not None:
            pygame.quit()
            self.screen = None
            self.clock = None

# --- Ahora, integraremos esto en el flujo de entrenamiento de Stable-Baselines3 ---

# No necesitamos la clase VisualMultiBodyEnv aparte si la lógica de render está en el entorno base
# Simplemente instanciamos MultiBodyCubePushEnv con render_mode='human'

# Asegúrate de ejecutar esto primero en Colab para el display virtual
# !apt-get install -y xvfb x11-utils > /dev/null 2>&1
# !pip install pygame pyvirtualdisplay > /dev/null 2>&1
# !pip install imageio[ffmpeg] > /dev/null 2>&1 # Para imageio.mimsave con ffmpeg

# from pyvirtualdisplay import Display
# display = Display(visible=0, size=(640, 480))
# display.start()

# --- Entrenamiento con Stable-Baselines3 (PPO) ---
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import BaseCallback

print("Entorno 'MultiBodyCubePushEnv' con renderizado Pygame.")

# --- Callback para guardar frames para el GIF ---
class PygameRenderCallback(BaseCallback):
    def __init__(self, verbose=0, render_freq=1, save_dir="frames"):
        super(PygameRenderCallback, self).__init__(verbose)
        self.render_freq = render_freq
        self.save_dir = save_dir
        self.frame_count = 0
        os.makedirs(self.save_dir, exist_ok=True)
        self.images = [] # Para almacenar frames para el GIF final

    def _on_training_start(self) -> None:
        if self.model.env.envs[0].render_mode == "human":
            # Inicializar el render Pygame al inicio
            self.model.env.envs[0]._render_frame()

    def _on_step(self) -> bool:
        if self.n_calls % self.render_freq == 0 and self.model.env.envs[0].render_mode == "human":
            self.model.env.envs[0]._render_frame()
            # Capturar la pantalla de Pygame para el GIF
            img_array = pygame.surfarray.array3d(self.model.env.envs[0].screen)
            img_array = np.transpose(img_array, (1, 0, 2)) # Transponer para formato de imagen
            self.images.append(img_array)
            # print(f"🖼️ Captured frame {self.frame_count}") # Descomentar para ver el progreso

        self.frame_count += 1
        return True

    def _on_training_end(self) -> None:
        if self.model.env.envs[0].render_mode == "human":
            self.model.env.envs[0].close()
            # Guardar el GIF al finalizar el entrenamiento
            if self.images:
                gif_path = os.path.join(self.save_dir, "training_animation.gif")
                print(f"\nGenerando GIF de entrenamiento en: {gif_path}")
                imageio.mimsave(gif_path, self.images, fps=self.model.env.envs[0].metadata["render_fps"])
                print("GIF de entrenamiento generado.")


# --- Creación del entorno vectorizado para Stable-Baselines3 ---
env = make_vec_env(MultiBodyCubePushEnv, n_envs=1, env_kwargs={"render_mode": "human"}, seed=0)

# --- Configuración y Entrenamiento del Modelo PPO ---
print("\nIniciando entrenamiento del agente PPO con renderizado Pygame...")
model = PPO("MlpPolicy", env, verbose=1, learning_rate=0.001, n_steps=1024, batch_size=64,
            gamma=0.99, gae_lambda=0.95, clip_range=0.2, ent_coef=0.01,
            tensorboard_log="./discrete_ppo_tensorboard_logs/")

total_timesteps = 100_000
print(f"Entrenando por {total_timesteps} timesteps...")
# Pasamos el callback de PygameRender para capturar frames durante el entrenamiento
model.learn(total_timesteps=total_timesteps, callback=PygameRenderCallback(render_freq=1)) # Capturar cada paso

print("\nEntrenamiento finalizado.")
model.save("discrete_multi_body_cube_push_ppo_pygame")
print("Modelo guardado como 'discrete_multi_body_cube_push_ppo_pygame.zip'")

# --- Prueba del Modelo Entrenado (Demostración y GIF de Demostración) ---
print("\nDemostración del modelo entrenado:")
demo_images = [] # Lista para los frames de la demostración
obs, info = env.reset()
episodes = 3 # Número de episodios para la demostración

for episode in range(episodes):
    done = False
    truncated = False
    episode_reward = 0
    while not done and not truncated:
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        episode_reward += reward
        env.render() # Mostrar en tiempo real durante la demostración

        # Capturar frame para el GIF de demostración
        img_array = pygame.surfarray.array3d(env.envs[0].screen)
        img_array = np.transpose(img_array, (1, 0, 2))
        demo_images.append(img_array)

        if terminated or truncated:
            print(f"Episodio Demostración {episode + 1}: Recompensa = {episode_reward:.2f}, Distancia final a meta: {info[0]['distance_to_goal']:.2f}")
            obs, info = env.reset()
            break # Terminar el bucle interno

env.close() # Cierra el entorno de renderizado al final

# Generar GIF de la demostración
if demo_images:
    demo_gif_path = "cube_push_demo.gif"
    print(f"\nGenerando GIF de demostración en: {demo_gif_path}")
    imageio.mimsave(demo_gif_path, demo_images, fps=env.envs[0].metadata["render_fps"])
    print("GIF de demostración generado.")

    # Mostrar el GIF en Colab
    from IPython.display import Image
    try:
        display_instance = Display(visible=0, size=(640, 480))
        display_instance.start()
        display(Image(filename=demo_gif_path))
        display_instance.stop()
    except Exception as e:
        print(f"No se pudo mostrar el GIF directamente: {e}")
        print(f"Puedes descargar el GIF desde el panel de archivos de Colab: {demo_gif_path}")

In [ ]:
# --- 0. Instalación y Configuración Inicial (Necesario en Colab) ---
# Si aún no lo has ejecutado, asegúrate de hacerlo:
# !apt-get install -y xvfb x11-utils > /dev/null 2>&1
# !pip install pygame pyvirtualdisplay > /dev/null 2>&1
# !pip install imageio[ffmpeg] > /dev/null 2>&1 # Necesario para guardar GIF/MP4

from pyvirtualdisplay import Display
import os

# Iniciar un display virtual para pygame (DESCOMENTAR ESTO PARA EJECUTAR EN COLAB)
# print("Iniciando display virtual...")
# display = Display(visible=0, size=(640, 480))
# display.start()
# print("Display virtual iniciado.")

import pygame
import numpy as np
import gymnasium as gym
from gymnasium import spaces
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import imageio.v2 as imageio
import time

# --- 1. Tu Entorno MultiBodyCubePushEnv (con correcciones y mejoras) ---
class MultiBodyCubePushEnv(gym.Env):
    metadata = {"render_modes": ["human"], "render_fps": 4}

    def __init__(self, render_mode=None):
        super().__init__()
        self.grid_size = 10
        self.render_mode = render_mode

        self.observation_space = spaces.Box(low=0, high=self.grid_size - 1, shape=(8,), dtype=np.int32)
        self.action_space = spaces.MultiDiscrete([5, 5])

        self.body1_pos = None
        self.body2_pos = None
        self.cube_pos = None
        self.goal_pos = np.array([self.grid_size - 1, self.grid_size - 1])

        # Atributos para Pygame render
        self.screen = None
        self.clock = None
        self.cell_size = 50 # Tamaño en píxeles de cada celda

        # Atributo para la recompensa de progreso
        self.last_distance_to_goal = None

    def _get_obs(self):
        return np.concatenate([
            self.body1_pos, self.body2_pos, self.cube_pos, self.goal_pos
        ]).astype(np.int32)

    def _get_info(self):
        return {"distance_to_goal": np.linalg.norm(self.cube_pos - self.goal_pos)}

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self.body1_pos = self.np_random.integers(0, self.grid_size, size=2)
        self.body2_pos = self.np_random.integers(0, self.grid_size, size=2)
        self.cube_pos = self.np_random.integers(0, self.grid_size, size=2)

        while np.array_equal(self.body1_pos, self.body2_pos) or \
              np.array_equal(self.body1_pos, self.cube_pos) or \
              np.array_equal(self.body2_pos, self.cube_pos) or \
              np.array_equal(self.cube_pos, self.goal_pos):
            self.body1_pos = self.np_random.integers(0, self.grid_size, size=2)
            self.body2_pos = self.np_random.integers(0, self.grid_size, size=2)
            self.cube_pos = self.np_random.integers(0, self.grid_size, size=2)

        observation = self._get_obs()
        self.last_distance_to_goal = np.linalg.norm(self.cube_pos - self.goal_pos) # Inicializar para recompensa de progreso
        info = self._get_info()

        if self.render_mode == "human":
            self._render_frame()

        return observation, info

    def step(self, action):
        action_body1, action_body2 = action

        def apply_move(pos, a):
            new_pos = pos.copy()
            if a == 0: new_pos[1] -= 1  # Arriba
            elif a == 1: new_pos[1] += 1  # Abajo
            elif a == 2: new_pos[0] -= 1  # Izquierda
            elif a == 3: new_pos[0] += 1  # Derecha
            # Si a == 4, no se mueve

            new_pos = np.clip(new_pos, 0, self.grid_size - 1)
            return new_pos

        prev_body1_pos = self.body1_pos.copy()
        prev_body2_pos = self.body2_pos.copy()

        # Mover cuerpos a sus nuevas posiciones propuestas
        proposed_body1_pos = apply_move(self.body1_pos, action_body1)
        proposed_body2_pos = apply_move(self.body2_pos, action_body2)

        # Determinar si cada cuerpo quiere moverse a la celda del cubo
        b1_wants_to_move_to_cube = np.array_equal(proposed_body1_pos, self.cube_pos)
        b2_wants_to_move_to_cube = np.array_equal(proposed_body2_pos, self.cube_pos)

        # Lógica de empuje del cubo
        new_cube_pos = self.cube_pos.copy()
        cube_moved = False

        # Delta de acciones a movimiento (para entender la dirección intentada)
        action_to_delta = {
            0: np.array([0,-1]), # Arriba
            1: np.array([0,1]),  # Abajo
            2: np.array([-1,0]), # Izquierda
            3: np.array([1,0]),  # Derecha
            4: np.array([0,0])   # No mover
        }
        delta_action_b1 = action_to_delta[action_body1]
        delta_action_b2 = action_to_delta[action_body2]

        # Condición de empuje: Ambos cuerpos quieren moverse a la posición del cubo Y sus acciones son en la misma dirección
        # (Esto simula que empujan juntos un objeto)
        if b1_wants_to_move_to_cube and b2_wants_to_move_to_cube:
            # Empuje en X
            if delta_action_b1[0] != 0 and np.array_equal(delta_action_b1, delta_action_b2):
                potential_cube_move_x = self.cube_pos[0] + delta_action_b1[0]
                if 0 <= potential_cube_move_x < self.grid_size: # No salirse de límites
                    new_cube_pos[0] = potential_cube_move_x
                    cube_moved = True
            # Empuje en Y
            if delta_action_b1[1] != 0 and np.array_equal(delta_action_b1, delta_action_b2):
                potential_cube_move_y = self.cube_pos[1] + delta_action_b1[1]
                if 0 <= potential_cube_move_y < self.grid_size: # No salirse de límites
                    new_cube_pos[1] = potential_cube_move_y
                    cube_moved = True

        if cube_moved:
            # Si el cubo se movió, los cuerpos que lo empujaron se mueven a la posición que ocupaba el cubo
            self.body1_pos = self.cube_pos # La antigua posición del cubo
            self.body2_pos = self.cube_pos # Asumiendo que ambos empujan y terminan en la misma celda
            self.cube_pos = new_cube_pos
        else:
            # Si el cubo no se movió, los cuerpos se mueven a sus posiciones propuestas,
            # pero no pueden ocupar la celda del cubo si ya está ocupada
            if not b1_wants_to_move_to_cube:
                self.body1_pos = proposed_body1_pos
            else: # Si b1 quería moverse al cubo y no lo movió, se queda donde estaba
                self.body1_pos = prev_body1_pos

            if not b2_wants_to_move_to_cube:
                self.body2_pos = proposed_body2_pos
            else: # Si b2 quería moverse al cubo y no lo movió, se queda donde estaba
                self.body2_pos = prev_body2_pos


        # Clip para asegurar que todo esté dentro de los límites
        self.body1_pos = np.clip(self.body1_pos, 0, self.grid_size - 1)
        self.body2_pos = np.clip(self.body2_pos, 0, self.grid_size - 1)
        self.cube_pos = np.clip(self.cube_pos, 0, self.grid_size - 1)

        reward = 0.0
        terminated = False
        truncated = False

        # Recompensa de progreso
        current_distance = np.linalg.norm(self.cube_pos - self.goal_pos)
        reward = -0.1 # Pequeña penalización por cada paso
        reward += (self.last_distance_to_goal - current_distance) * 0.5 # Recompensa por acercarse a la meta
        self.last_distance_to_goal = current_distance # Actualizar para el siguiente paso

        # Recompensa por alcanzar la meta
        if np.array_equal(self.cube_pos, self.goal_pos):
            reward = 10.0 # Recompensa grande por alcanzar la meta
            terminated = True # El episodio termina

        observation = self._get_obs()
        info = self._get_info()

        if self.render_mode == "human":
            self._render_frame()

        return observation, reward, terminated, truncated, info

    def _render_frame(self):
        # Asegurarse de que Pygame esté inicializado y la pantalla creada
        if self.screen is None:
            pygame.init()
            self.screen_size_px = self.grid_size * self.cell_size
            self.screen = pygame.display.set_mode((self.screen_size_px, self.screen_size_px))
            pygame.display.set_caption("Multi-Body Cube Push Env")
            self.clock = pygame.time.Clock()

        # Colores
        WHITE = (255, 255, 255)
        BLACK = (0, 0, 0)
        RED = (255, 0, 0)     # Cuerpo 1
        GREEN = (0, 255, 0)   # Cuerpo 2
        BLUE = (0, 0, 255)    # Cubo
        ORANGE = (255, 165, 0) # Meta

        self.screen.fill(WHITE) # Fondo blanco

        # Dibujar la cuadrícula
        for x in range(self.grid_size):
            for y in range(self.grid_size):
                rect = pygame.Rect(x * self.cell_size, y * self.cell_size, self.cell_size, self.cell_size)
                pygame.draw.rect(self.screen, BLACK, rect, 1) # Borde de celda

        # Función para dibujar objetos en el centro de la celda
        def draw_object(pos_grid, color, shape="circle", radius_ratio=0.8):
            center_x = (pos_grid[0] * self.cell_size) + (self.cell_size / 2)
            center_y = (pos_grid[1] * self.cell_size) + (self.cell_size / 2)
            radius = int(self.cell_size / 2 * radius_ratio)
            if shape == "circle":
                pygame.draw.circle(self.screen, color, (center_x, center_y), radius)
            elif shape == "rect":
                size = int(self.cell_size * radius_ratio)
                rect_pos_x = center_x - size / 2
                rect_pos_y = center_y - size / 2
                pygame.draw.rect(self.screen, color, (rect_pos_x, rect_pos_y, size, size))

        # Dibujar objetos
        draw_object(self.goal_pos, ORANGE, shape="circle", radius_ratio=0.6) # Meta (más pequeña)
        draw_object(self.cube_pos, BLUE, shape="rect", radius_ratio=0.8) # Cubo (rectángulo)
        draw_object(self.body1_pos, RED, shape="circle", radius_ratio=0.8) # Cuerpo 1
        draw_object(self.body2_pos, GREEN, shape="circle", radius_ratio=0.8) # Cuerpo 2

        pygame.event.pump()
        pygame.display.flip()

        if self.render_mode == "human" and self.clock:
            self.clock.tick(self.metadata["render_fps"])

    def close(self):
        if self.screen is not None:
            pygame.quit()
            self.screen = None
            self.clock = None

# --- Ahora, integraremos esto en el flujo de entrenamiento de Stable-Baselines3 ---

# Asegúrate de ejecutar esto primero en Colab para el display virtual
# !apt-get install -y xvfb x11-utils > /dev/null 2>&1
# !pip install pygame pyvirtualdisplay > /dev/null 2>&1
# !pip install imageio[ffmpeg] > /dev/null 2>&1 # Para imageio.mimsave con ffmpeg

# from pyvirtualdisplay import Display
# display = Display(visible=0, size=(640, 480))
# display.start()

# Importaciones necesarias para Stable-Baselines3
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import BaseCallback

print("Entorno 'MultiBodyCubePushEnv' con renderizado Pygame (corregido).")

# --- Callback para guardar frames para el GIF ---
class PygameRenderCallback(BaseCallback):
    def __init__(self, verbose=0, render_freq=1, save_dir="frames"):
        super(PygameRenderCallback, self).__init__(verbose)
        self.render_freq = render_freq
        self.save_dir = save_dir
        self.frame_count = 0
        os.makedirs(self.save_dir, exist_ok=True)
        self.images = []

    def _on_training_start(self) -> None:
        if self.model.env.envs[0].render_mode == "human":
            # Acceder al entorno base a través del wrapper Monitor (.env)
            self.model.env.envs[0].env._render_frame()

    def _on_step(self) -> bool:
        if self.n_calls % self.render_freq == 0 and self.model.env.envs[0].render_mode == "human":
            # Acceder al entorno base para la pantalla de Pygame
            env_instance = self.model.env.envs[0].env
            env_instance._render_frame() # Asegurarse de que el frame se dibuje

            # Capturar la pantalla de Pygame para el GIF
            img_array = pygame.surfarray.array3d(env_instance.screen)
            img_array = np.transpose(img_array, (1, 0, 2))
            self.images.append(img_array)

        self.frame_count += 1
        return True

    def _on_training_end(self) -> None:
        if self.model.env.envs[0].render_mode == "human":
            self.model.env.envs[0].env.close() # Acceder al entorno base para cerrar
            if self.images:
                gif_path = os.path.join(self.save_dir, "training_animation.gif")
                print(f"\nGenerando GIF de entrenamiento en: {gif_path}")
                imageio.mimsave(gif_path, self.images, fps=self.model.env.envs[0].env.metadata["render_fps"])
                print("GIF de entrenamiento generado.")


# --- Creación del entorno vectorizado para Stable-Baselines3 ---
# make_vec_env envuelve el entorno con Monitor por defecto.
env = make_vec_env(MultiBodyCubePushEnv, n_envs=1, env_kwargs={"render_mode": "human"}, seed=0)

# --- Configuración y Entrenamiento del Modelo PPO ---
print("\nIniciando entrenamiento del agente PPO con renderizado Pygame...")
model = PPO("MlpPolicy", env, verbose=1, learning_rate=0.001, n_steps=1024, batch_size=64,
            gamma=0.99, gae_lambda=0.95, clip_range=0.2, ent_coef=0.01,
            tensorboard_log="./discrete_ppo_tensorboard_logs/")

total_timesteps = 100_000
print(f"Entrenando por {total_timesteps} timesteps...")
# Pasamos el callback de PygameRender para capturar frames durante el entrenamiento
model.learn(total_timesteps=total_timesteps, callback=PygameRenderCallback(render_freq=1))

print("\nEntrenamiento finalizado.")
model.save("discrete_multi_body_cube_push_ppo_pygame")
print("Modelo guardado como 'discrete_multi_body_cube_push_ppo_pygame.zip'")

# --- Prueba del Modelo Entrenado (Demostración y GIF de Demostración) ---
print("\nDemostración del modelo entrenado:")
demo_images = []
obs, info = env.reset() # Resetear el entorno vectorizado

episodes = 3
for episode in range(episodes):
    done = False
    truncated = False
    episode_reward = 0
    while not done and not truncated:
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        episode_reward += reward

        # Para la demostración, directamente llamar al render del entorno base y capturar
        env_instance = env.envs[0].env # Acceder al entorno base para la demostración
        env_instance._render_frame()
        img_array = pygame.surfarray.array3d(env_instance.screen)
        img_array = np.transpose(img_array, (1, 0, 2))
        demo_images.append(img_array)

        if terminated or truncated:
            print(f"Episodio Demostración {episode + 1}: Recompensa = {episode_reward:.2f}, Distancia final a meta: {info[0]['distance_to_goal']:.2f}")
            obs, info = env.reset() # Resetear el entorno vectorizado
            break

env.close() # Cierra el entorno de renderizado al final

# Generar GIF de la demostración
if demo_images:
    demo_gif_path = "cube_push_demo.gif"
    print(f"\nGenerando GIF de demostración en: {demo_gif_path}")
    imageio.mimsave(demo_gif_path, demo_images, fps=env.envs[0].env.metadata["render_fps"])
    print("GIF de demostración generado.")

    # Mostrar el GIF en Colab (descomentar para ejecutar en Colab)
    # from IPython.display import Image
    # try:
    #     display_instance = Display(visible=0, size=(640, 480))
    #     display_instance.start()
    #     display(Image(filename=demo_gif_path))
    #     display_instance.stop()
    # except Exception as e:
    #     print(f"No se pudo mostrar el GIF directamente: {e}")
    #     print(f"Puedes descargar el GIF desde el panel de archivos de Colab: {demo_gif_path}")

In [ ]:
# --- 0. Instalación y Configuración Inicial (Necesario en Colab) ---
# Si aún no lo has ejecutado, asegúrate de hacerlo:
# !apt-get install -y xvfb x11-utils > /dev/null 2>&1
!pip install pygame pyvirtualdisplay > /dev/null 2>&1
# !pip install imageio[ffmpeg] > /dev/null 2>&1 # Necesario para guardar GIF/MP4
!pip install stable-baselines3 torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118  # Instala PyTorch con CUDA y SB3

from pyvirtualdisplay import Display
import os
import torch # Importar torch para verificar la disponibilidad de CUDA

# Iniciar un display virtual para pygame (DESCOMENTAR ESTO PARA EJECUTAR EN COLAB)
# print("Iniciando display virtual...")
# display = Display(visible=0, size=(640, 480))
# display.start()
# print("Display virtual iniciado.")

import pygame
import numpy as np
import gymnasium as gym
from gymnasium import spaces
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import imageio.v2 as imageio
import time

# --- 1. Tu Entorno MultiBodyCubePushEnv (con correcciones y mejoras) ---
class MultiBodyCubePushEnv(gym.Env):
    metadata = {"render_modes": ["human"], "render_fps": 4}

    def __init__(self, render_mode=None):
        super().__init__()
        self.grid_size = 10
        self.render_mode = render_mode

        self.observation_space = spaces.Box(low=0, high=self.grid_size - 1, shape=(8,), dtype=np.int32)
        self.action_space = spaces.MultiDiscrete([5, 5])

        self.body1_pos = None
        self.body2_pos = None
        self.cube_pos = None
        self.goal_pos = np.array([self.grid_size - 1, self.grid_size - 1])

        # Atributos para Pygame render
        self.screen = None
        self.clock = None
        self.cell_size = 50 # Tamaño en píxeles de cada celda

        # Atributo para la recompensa de progreso
        self.last_distance_to_goal = None

    def _get_obs(self):
        return np.concatenate([
            self.body1_pos, self.body2_pos, self.cube_pos, self.goal_pos
        ]).astype(np.int32)

    def _get_info(self):
        return {"distance_to_goal": np.linalg.norm(self.cube_pos - self.goal_pos)}

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self.body1_pos = self.np_random.integers(0, self.grid_size, size=2)
        self.body2_pos = self.np_random.integers(0, self.grid_size, size=2)
        self.cube_pos = self.np_random.integers(0, self.grid_size, size=2)

        while np.array_equal(self.body1_pos, self.body2_pos) or \
              np.array_equal(self.body1_pos, self.cube_pos) or \
              np.array_equal(self.body2_pos, self.cube_pos) or \
              np.array_equal(self.cube_pos, self.goal_pos):
            self.body1_pos = self.np_random.integers(0, self.grid_size, size=2)
            self.body2_pos = self.np_random.integers(0, self.grid_size, size=2)
            self.cube_pos = self.np_random.integers(0, self.grid_size, size=2)

        observation = self._get_obs()
        self.last_distance_to_goal = np.linalg.norm(self.cube_pos - self.goal_pos) # Inicializar para recompensa de progreso
        info = self._get_info()

        if self.render_mode == "human":
            self._render_frame()

        return observation, info

    def step(self, action):
        action_body1, action_body2 = action

        def apply_move(pos, a):
            new_pos = pos.copy()
            if a == 0: new_pos[1] -= 1  # Arriba
            elif a == 1: new_pos[1] += 1  # Abajo
            elif a == 2: new_pos[0] -= 1  # Izquierda
            elif a == 3: new_pos[0] += 1  # Derecha
            # Si a == 4, no se mueve

            new_pos = np.clip(new_pos, 0, self.grid_size - 1)
            return new_pos

        prev_body1_pos = self.body1_pos.copy()
        prev_body2_pos = self.body2_pos.copy()

        # Mover cuerpos a sus nuevas posiciones propuestas
        proposed_body1_pos = apply_move(self.body1_pos, action_body1)
        proposed_body2_pos = apply_move(self.body2_pos, action_body2)

        # Determinar si cada cuerpo quiere moverse a la celda del cubo
        b1_wants_to_move_to_cube = np.array_equal(proposed_body1_pos, self.cube_pos)
        b2_wants_to_move_to_cube = np.array_equal(proposed_body2_pos, self.cube_pos)

        # Lógica de empuje del cubo
        new_cube_pos = self.cube_pos.copy()
        cube_moved = False

        # Delta de acciones a movimiento (para entender la dirección intentada)
        action_to_delta = {
            0: np.array([0,-1]), # Arriba
            1: np.array([0,1]),  # Abajo
            2: np.array([-1,0]), # Izquierda
            3: np.array([1,0]),  # Derecha
            4: np.array([0,0])   # No mover
        }
        delta_action_b1 = action_to_delta[action_body1]
        delta_action_b2 = action_to_delta[action_body2]

        # Condición de empuje: Ambos cuerpos quieren moverse a la posición del cubo Y sus acciones son en la misma dirección
        # (Esto simula que empujan juntos un objeto)
        if b1_wants_to_move_to_cube and b2_wants_to_move_to_cube:
            # Empuje en X
            if delta_action_b1[0] != 0 and np.array_equal(delta_action_b1, delta_action_b2):
                potential_cube_move_x = self.cube_pos[0] + delta_action_b1[0]
                if 0 <= potential_cube_move_x < self.grid_size: # No salirse de límites
                    new_cube_pos[0] = potential_cube_move_x
                    cube_moved = True
            # Empuje en Y
            if delta_action_b1[1] != 0 and np.array_equal(delta_action_b1, delta_action_b2):
                potential_cube_move_y = self.cube_pos[1] + delta_action_b1[1]
                if 0 <= potential_cube_move_y < self.grid_size: # No salirse de límites
                    new_cube_pos[1] = potential_cube_move_y
                    cube_moved = True

        if cube_moved:
            # Si el cubo se movió, los cuerpos que lo empujaron se mueven a la posición que ocupaba el cubo
            self.body1_pos = self.cube_pos # La antigua posición del cubo
            self.body2_pos = self.cube_pos # Asumiendo que ambos empujan y terminan en la misma celda
            self.cube_pos = new_cube_pos
        else:
            # Si el cubo no se movió, los cuerpos se mueven a sus posiciones propuestas,
            # pero no pueden ocupar la celda del cubo si ya está ocupada
            if not b1_wants_to_move_to_cube:
                self.body1_pos = proposed_body1_pos
            else: # Si b1 quería moverse al cubo y no lo movió, se queda donde estaba
                self.body1_pos = prev_body1_pos

            if not b2_wants_to_move_to_cube:
                self.body2_pos = proposed_body2_pos
            else: # Si b2 quería moverse al cubo y no lo movió, se queda donde estaba
                self.body2_pos = prev_body2_pos


        # Clip para asegurar que todo esté dentro de los límites
        self.body1_pos = np.clip(self.body1_pos, 0, self.grid_size - 1)
        self.body2_pos = np.clip(self.body2_pos, 0, self.grid_size - 1)
        self.cube_pos = np.clip(self.cube_pos, 0, self.grid_size - 1)

        reward = 0.0
        terminated = False
        truncated = False

        # Recompensa de progreso
        current_distance = np.linalg.norm(self.cube_pos - self.goal_pos)
        reward = -0.1 # Pequeña penalización por cada paso
        reward += (self.last_distance_to_goal - current_distance) * 0.5 # Recompensa por acercarse a la meta
        self.last_distance_to_goal = current_distance # Actualizar para el siguiente paso

        # Recompensa por alcanzar la meta
        if np.array_equal(self.cube_pos, self.goal_pos):
            reward = 10.0 # Recompensa grande por alcanzar la meta
            terminated = True # El episodio termina

        observation = self._get_obs()
        info = self._get_info()

        if self.render_mode == "human":
            self._render_frame()

        return observation, reward, terminated, truncated, info

    def _render_frame(self):
        # Asegurarse de que Pygame esté inicializado y la pantalla creada
        if self.screen is None:
            pygame.init()
            self.screen_size_px = self.grid_size * self.cell_size
            self.screen = pygame.display.set_mode((self.screen_size_px, self.screen_size_px))
            pygame.display.set_caption("Multi-Body Cube Push Env")
            self.clock = pygame.time.Clock()

        # Colores
        WHITE = (255, 255, 255)
        BLACK = (0, 0, 0)
        RED = (255, 0, 0)     # Cuerpo 1
        GREEN = (0, 255, 0)   # Cuerpo 2
        BLUE = (0, 0, 255)    # Cubo
        ORANGE = (255, 165, 0) # Meta

        self.screen.fill(WHITE) # Fondo blanco

        # Dibujar la cuadrícula
        for x in range(self.grid_size):
            for y in range(self.grid_size):
                rect = pygame.Rect(x * self.cell_size, y * self.cell_size, self.cell_size, self.cell_size)
                pygame.draw.rect(self.screen, BLACK, rect, 1) # Borde de celda

        # Función para dibujar objetos en el centro de la celda
        def draw_object(pos_grid, color, shape="circle", radius_ratio=0.8):
            center_x = (pos_grid[0] * self.cell_size) + (self.cell_size / 2)
            center_y = (pos_grid[1] * self.cell_size) + (self.cell_size / 2)
            radius = int(self.cell_size / 2 * radius_ratio)
            if shape == "circle":
                pygame.draw.circle(self.screen, color, (center_x, center_y), radius)
            elif shape == "rect":
                size = int(self.cell_size * radius_ratio)
                rect_pos_x = center_x - size / 2
                rect_pos_y = center_y - size / 2
                pygame.draw.rect(self.screen, color, (rect_pos_x, rect_pos_y, size, size))

        # Dibujar objetos
        draw_object(self.goal_pos, ORANGE, shape="circle", radius_ratio=0.6) # Meta (más pequeña)
        draw_object(self.cube_pos, BLUE, shape="rect", radius_ratio=0.8) # Cubo (rectángulo)
        draw_object(self.body1_pos, RED, shape="circle", radius_ratio=0.8) # Cuerpo 1
        draw_object(self.body2_pos, GREEN, shape="circle", radius_ratio=0.8) # Cuerpo 2

        pygame.event.pump()
        pygame.display.flip()

        if self.render_mode == "human" and self.clock:
            self.clock.tick(self.metadata["render_fps"])

    def close(self):
        if self.screen is not None:
            pygame.quit()
            self.screen = None
            self.clock = None

# --- Ahora, integraremos esto en el flujo de entrenamiento de Stable-Baselines3 ---

# Importaciones necesarias para Stable-Baselines3
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import BaseCallback

print("Entorno 'MultiBodyCubePushEnv' con renderizado Pygame (corregido).")

# --- Callback para guardar frames para el GIF ---
class PygameRenderCallback(BaseCallback):
    def __init__(self, verbose=0, render_freq=1, save_dir="frames"):
        super(PygameRenderCallback, self).__init__(verbose)
        self.render_freq = render_freq
        self.save_dir = save_dir
        self.frame_count = 0
        os.makedirs(self.save_dir, exist_ok=True)
        self.images = []

    def _on_training_start(self) -> None:
        # Asegurarse de que el entorno exista y tenga render_mode "human"
        if hasattr(self.model.env, 'envs') and self.model.env.envs[0].render_mode == "human":
            # Acceder al entorno base a través del wrapper Monitor (.env)
            self.model.env.envs[0].env._render_frame()

    def _on_step(self) -> bool:
        # Verificar si estamos en el entorno base (no vectorizado si n_envs=1)
        # y si el render_mode es "human" y si es el momento de renderizar
        if self.n_calls % self.render_freq == 0 and \
           hasattr(self.model.env, 'envs') and \
           self.model.env.envs[0].render_mode == "human":

            # Acceder al entorno base para la pantalla de Pygame
            env_instance = self.model.env.envs[0].env
            env_instance._render_frame() # Asegurarse de que el frame se dibuje

            # Capturar la pantalla de Pygame para el GIF
            img_array = pygame.surfarray.array3d(env_instance.screen)
            img_array = np.transpose(img_array, (1, 0, 2)) # Pygame es (W, H, C), necesitamos (H, W, C)
            self.images.append(img_array)

        self.frame_count += 1
        return True

    def _on_training_end(self) -> None:
        if hasattr(self.model.env, 'envs') and self.model.env.envs[0].render_mode == "human":
            self.model.env.envs[0].env.close() # Acceder al entorno base para cerrar
            if self.images:
                gif_path = os.path.join(self.save_dir, "training_animation.gif")
                print(f"\nGenerando GIF de entrenamiento en: {gif_path}")
                # Ajustar fps si es muy rápido
                imageio.mimsave(gif_path, self.images, fps=self.model.env.envs[0].env.metadata["render_fps"])
                print("GIF de entrenamiento generado.")


# --- Definición del dispositivo (GPU o CPU) ---
if torch.cuda.is_available():
    device = "cuda"
    print("GPU (CUDA) disponible. El entrenamiento se ejecutará en la GPU.")
else:
    device = "cpu"
    print("GPU (CUDA) NO disponible. El entrenamiento se ejecutará en la CPU.")


# --- Creación del entorno vectorizado para Stable-Baselines3 ---
# make_vec_env envuelve el entorno con Monitor por defecto.
env = make_vec_env(MultiBodyCubePushEnv, n_envs=1, env_kwargs={"render_mode": "human"}, seed=0)

# --- Configuración y Entrenamiento del Modelo PPO ---
print("\nIniciando entrenamiento del agente PPO con renderizado Pygame...")
model = PPO("MlpPolicy", env, verbose=1, learning_rate=0.001, n_steps=1024, batch_size=64,
            gamma=0.99, gae_lambda=0.95, clip_range=0.2, ent_coef=0.01,
            tensorboard_log="./discrete_ppo_tensorboard_logs/",
            device=device) # <-- ¡Aquí se especifica el dispositivo!

total_timesteps = 100_000
print(f"Entrenando por {total_timesteps} timesteps...")
# Pasamos el callback de PygameRender para capturar frames durante el entrenamiento
model.learn(total_timesteps=total_timesteps, callback=PygameRenderCallback(render_freq=1))

print("\nEntrenamiento finalizado.")
model.save("discrete_multi_body_cube_push_ppo_pygame")
print("Modelo guardado como 'discrete_multi_body_cube_push_ppo_pygame.zip'")

# --- Prueba del Modelo Entrenado (Demostración y GIF de Demostración) ---
print("\nDemostración del modelo entrenado:")
demo_images = []
obs, info = env.reset() # Resetear el entorno vectorizado

episodes = 3
for episode in range(episodes):
    done = False
    truncated = False
    episode_reward = 0
    while not done and not truncated:
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        episode_reward += reward

        # Para la demostración, directamente llamar al render del entorno base y capturar
        env_instance = env.envs[0].env # Acceder al entorno base para la demostración
        env_instance._render_frame()
        img_array = pygame.surfarray.array3d(env_instance.screen)
        img_array = np.transpose(img_array, (1, 0, 2)) # Pygame es (W, H, C), necesitamos (H, W, C)
        demo_images.append(img_array)

        if terminated or truncated:
            print(f"Episodio Demostración {episode + 1}: Recompensa = {episode_reward:.2f}, Distancia final a meta: {info[0]['distance_to_goal']:.2f}")
            obs, info = env.reset() # Resetear el entorno vectorizado
            break

env.close() # Cierra el entorno de renderizado al final

# Generar GIF de la demostración
if demo_images:
    demo_gif_path = "cube_push_demo.gif"
    print(f"\nGenerando GIF de demostración en: {demo_gif_path}")
    imageio.mimsave(demo_gif_path, demo_images, fps=env.envs[0].env.metadata["render_fps"])
    print("GIF de demostración generado.")

    # Mostrar el GIF en Colab (descomentar para ejecutar en Colab)
    # from IPython.display import Image
    # try:
    #     display_instance = Display(visible=0, size=(640, 480))
    #     display_instance.start()
    #     display(Image(filename=demo_gif_path))
    #     display_instance.stop()
    # except Exception as e:
    #     print(f"No se pudo mostrar el GIF directamente: {e}")
    #     print(f"Puedes descargar el GIF desde el panel de archivos de Colab: {demo_gif_path}")

#Ann Biomas

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class NeuralBiomesModel(nn.Module):
    def __init__(self, input_dim, shared_features_dim, memory_hidden_dim, processing_output_dim, output_dim):
        super().__init__()

        # 1. Capa de Entrada Común ("El Mundo")
        self.shared_feature_extractor = nn.Sequential(
            nn.Linear(input_dim, shared_features_dim),
            nn.ReLU(),
            nn.Linear(shared_features_dim, shared_features_dim) # Opcional: más capas
        )

        # 2. Bioma de Almacenamiento (ej. usando LSTM para "memoria" secuencial)
        # Asumimos que procesa secuencias de features compartidos
        # En un modelo simple, puede ser solo una capa lineal para "codificar"
        self.memory_biom = nn.LSTM(shared_features_dim, memory_hidden_dim, batch_first=True)
        # O para un caso más simple, solo una capa que "codifique" para almacenamiento
        # self.memory_encoder = nn.Linear(shared_features_dim, memory_hidden_dim)


        # 3. Bioma de Procesamiento (ej. un MLP para lógica/cálculo)
        # Recibe features del mundo y el estado de la memoria
        self.processing_biom = nn.Sequential(
            nn.Linear(shared_features_dim + memory_hidden_dim, 128), # Combina features y memoria
            nn.ReLU(),
            nn.Linear(128, processing_output_dim),
            nn.ReLU()
        )

        # 4. Capa de Salida Final (combinando resultados de los biomas)
        self.output_layer = nn.Linear(processing_output_dim, output_dim)

    def forward(self, x):
        # Paso 1: Extraer características en el "Mundo"
        shared_features = self.shared_feature_extractor(x)

        # Paso 2: Pasar características al Bioma de Almacenamiento
        # Para LSTM/GRU, necesitamos una secuencia de entrada.
        # Si x es un solo punto de datos, lo tratamos como una secuencia de longitud 1
        shared_features_seq = shared_features.unsqueeze(1) # Añadir dimensión de secuencia
        memory_output, (h_n, c_n) = self.memory_biom(shared_features_seq)
        memory_state = h_n.squeeze(0) # Obtener el estado oculto final del LSTM

        # Paso 3: Pasar características y memoria al Bioma de Procesamiento
        # Concatenar las características compartidas con el estado de la memoria
        combined_features = torch.cat((shared_features, memory_state), dim=1)
        processed_output = self.processing_biom(combined_features)

        # Paso 4: Pasar el resultado procesado a la capa de salida final
        final_output = self.output_layer(processed_output)

        return final_output

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Supongamos que tienes estas listas de pérdidas y precisiones después del entrenamiento
# Estas listas serían el resultado de tu bucle de entrenamiento y evaluación
train_losses = [...] # pérdidas de entrenamiento por época
test_losses = [...]  # pérdidas de prueba por época
train_accuracies = [...] # precisiones de entrenamiento por época
test_accuracies = [...]  # precisiones de prueba por época

epochs = range(1, len(train_losses) + 1)

# Gráfica de Pérdida
plt.figure(figsize=(10, 5))
plt.plot(epochs, train_losses, 'b', label='Pérdida de Entrenamiento')
plt.plot(epochs, test_losses, 'r', label='Pérdida de Prueba')
plt.title('Pérdida de Entrenamiento y Prueba')
plt.xlabel('Épocas')
plt.ylabel('Pérdida')
plt.legend()
plt.grid(True)
plt.show()

# Gráfica de Precisión
plt.figure(figsize=(10, 5))
plt.plot(epochs, train_accuracies, 'b', label='Precisión de Entrenamiento')
plt.plot(epochs, test_accuracies, 'r', label='Precisión de Prueba')
plt.title('Precisión de Entrenamiento y Prueba')
plt.xlabel('Épocas')
plt.ylabel('Precisión')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from sklearn.datasets import make_circles
import matplotlib.pyplot as plt

# Generar 1000 muestras con ruido y un factor de escala para el círculo interno
X, y = make_circles(n_samples=1000, noise=0.05, factor=0.5, random_state=42)

# X son las características (coordenadas x, y)
# y son las etiquetas de clase (0 o 1)

# Visualizar los datos
plt.figure(figsize=(8, 6))
plt.scatter(X[y == 0, 0], X[y == 0, 1], label='Clase 0', alpha=0.7, edgecolors='w')
plt.scatter(X[y == 1, 0], X[y == 1, 1], label='Clase 1', alpha=0.7, edgecolors='w')
plt.title('Datos generados con make_circles')
plt.xlabel('Característica 1')
plt.ylabel('Característica 2')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import numpy as np

# --- 1. Definición del Modelo NeuralBiomesModel (como proporcionado) ---
class NeuralBiomesModel(nn.Module):
    def __init__(self, input_dim, shared_features_dim, memory_hidden_dim, processing_output_dim, output_dim):
        super().__init__()

        # 1. Capa de Entrada Común ("El Mundo")
        self.shared_feature_extractor = nn.Sequential(
            nn.Linear(input_dim, shared_features_dim),
            nn.ReLU(),
            nn.Linear(shared_features_dim, shared_features_dim)
        )

        # 2. Bioma de Almacenamiento (LSTM para "memoria" secuencial)
        self.memory_biom = nn.LSTM(shared_features_dim, memory_hidden_dim, batch_first=True)

        # 3. Bioma de Procesamiento (MLP para lógica/cálculo)
        self.processing_biom = nn.Sequential(
            nn.Linear(shared_features_dim + memory_hidden_dim, 128),
            nn.ReLU(),
            nn.Linear(128, processing_output_dim),
            nn.ReLU()
        )

        # 4. Capa de Salida Final (combinando resultados de los biomas)
        self.output_layer = nn.Linear(processing_output_dim, output_dim)

    def forward(self, x):
        # Paso 1: Extraer características en el "Mundo"
        shared_features = self.shared_feature_extractor(x)

        # Paso 2: Pasar características al Bioma de Almacenamiento
        shared_features_seq = shared_features.unsqueeze(1)
        memory_output, (h_n, c_n) = self.memory_biom(shared_features_seq)
        memory_state = h_n.squeeze(0)

        # Paso 3: Pasar características y memoria al Bioma de Procesamiento
        combined_features = torch.cat((shared_features, memory_state), dim=1)
        processed_output = self.processing_biom(combined_features)

        # Paso 4: Pasar el resultado procesado a la capa de salida final
        final_output = self.output_layer(processed_output)

        return final_output

# --- 2. Generación de Datos con make_circles ---
# Generar 1000 muestras con ruido y un factor de escala para el círculo interno
X, y = make_circles(n_samples=1000, noise=0.05, factor=0.5, random_state=42)

# Convertir a tensores de PyTorch
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(1) # Asegurar dimensión correcta para pérdida

# Dividir los datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_tensor, y_tensor, test_size=0.2, random_state=42)

# --- 3. Inicialización del Modelo, Función de Pérdida y Optimizador ---
input_dim = X.shape[1] # 2 características (x, y)
shared_features_dim = 64
memory_hidden_dim = 32
processing_output_dim = 64
output_dim = 1 # Para clasificación binaria (0 o 1)

model = NeuralBiomesModel(input_dim, shared_features_dim, memory_hidden_dim, processing_output_dim, output_dim)
criterion = nn.BCEWithLogitsLoss() # Para clasificación binaria, maneja sigmoid internamente
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# --- 4. Entrenamiento del Modelo ---
num_epochs = 200
train_losses = []
test_losses = []
train_accuracies = []
test_accuracies = []

for epoch in range(num_epochs):
    model.train() # Poner el modelo en modo entrenamiento
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())

    model.eval() # Poner el modelo en modo evaluación
    with torch.no_grad():
        # Evaluación en entrenamiento
        train_outputs = model(X_train)
        train_preds = (torch.sigmoid(train_outputs) > 0.5).float()
        train_acc = accuracy_score(y_train.numpy(), train_preds.numpy())
        train_accuracies.append(train_acc)

        # Evaluación en prueba
        test_outputs = model(X_test)
        test_loss = criterion(test_outputs, y_test)
        test_losses.append(test_loss.item())
        test_preds = (torch.sigmoid(test_outputs) > 0.5).float()
        test_acc = accuracy_score(y_test.numpy(), test_preds.numpy())
        test_accuracies.append(test_acc)

    if (epoch + 1) % 20 == 0:
        print(f'Época [{epoch+1}/{num_epochs}], Pérdida Entrenamiento: {loss.item():.4f}, Precisión Entrenamiento: {train_acc:.4f}, Pérdida Prueba: {test_loss.item():.4f}, Precisión Prueba: {test_acc:.4f}')

# --- 5. Visualización de Datos make_circles ---
print("\n--- Visualización de Datos make_circles ---")
plt.figure(figsize=(8, 6))
plt.scatter(X[y == 0, 0], X[y == 0, 1], label='Clase 0', alpha=0.7, edgecolors='w')
plt.scatter(X[y == 1, 0], X[y == 1, 1], label='Clase 1', alpha=0.7, edgecolors='w')
plt.title('Datos generados con make_circles')
plt.xlabel('Característica 1')
plt.ylabel('Característica 2')
plt.legend()
plt.grid(True)
plt.show()

# --- 6. Visualización de Gráficas de Entrenamiento y Prueba ---
print("\n--- Gráficas de Entrenamiento y Prueba ---")
epochs_list = range(1, num_epochs + 1)

# Gráfica de Pérdida
plt.figure(figsize=(10, 5))
plt.plot(epochs_list, train_losses, 'b', label='Pérdida de Entrenamiento')
plt.plot(epochs_list, test_losses, 'r', label='Pérdida de Prueba')
plt.title('Pérdida de Entrenamiento y Prueba')
plt.xlabel('Épocas')
plt.ylabel('Pérdida')
plt.legend()
plt.grid(True)
plt.show()

# Gráfica de Precisión
plt.figure(figsize=(10, 5))
plt.plot(epochs_list, train_accuracies, 'b', label='Precisión de Entrenamiento')
plt.plot(epochs_list, test_accuracies, 'r', label='Precisión de Prueba')
plt.title('Precisión de Entrenamiento y Prueba')
plt.xlabel('Épocas')
plt.ylabel('Precisión')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import numpy as np

# --- 1. Definición del Modelo NeuralBiomesModel ---
class NeuralBiomesModel(nn.Module):
    def __init__(self, input_dim, shared_features_dim, memory_hidden_dim, processing_output_dim, output_dim):
        super().__init__()
        self.shared_feature_extractor = nn.Sequential(
            nn.Linear(input_dim, shared_features_dim),
            nn.ReLU(),
            nn.Linear(shared_features_dim, shared_features_dim)
        )
        self.memory_biom = nn.LSTM(shared_features_dim, memory_hidden_dim, batch_first=True)
        self.processing_biom = nn.Sequential(
            nn.Linear(shared_features_dim + memory_hidden_dim, 128),
            nn.ReLU(),
            nn.Linear(128, processing_output_dim),
            nn.ReLU()
        )
        self.output_layer = nn.Linear(processing_output_dim, output_dim)

    def forward(self, x):
        shared_features = self.shared_feature_extractor(x)
        shared_features_seq = shared_features.unsqueeze(1)
        memory_output, (h_n, c_n) = self.memory_biom(shared_features_seq)
        memory_state = h_n.squeeze(0)
        combined_features = torch.cat((shared_features, memory_state), dim=1)
        processed_output = self.processing_biom(combined_features)
        final_output = self.output_layer(processed_output)
        return final_output

# --- 2. Generación de Datos con make_moons ---
X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

X_train, X_test, y_train, y_test = train_test_split(X_tensor, y_tensor, test_size=0.2, random_state=42)

# --- 3. Inicialización del Modelo ---
input_dim = X.shape[1]
shared_features_dim = 64
memory_hidden_dim = 32
processing_output_dim = 64
output_dim = 1

model = NeuralBiomesModel(input_dim, shared_features_dim, memory_hidden_dim, processing_output_dim, output_dim)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# --- 4. Entrenamiento ---
num_epochs = 200
train_losses = []
test_losses = []
train_accuracies = []
test_accuracies = []

for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())

    model.eval()
    with torch.no_grad():
        train_outputs = model(X_train)
        train_preds = (torch.sigmoid(train_outputs) > 0.5).float()
        train_acc = accuracy_score(y_train.numpy(), train_preds.numpy())
        train_accuracies.append(train_acc)

        test_outputs = model(X_test)
        test_loss = criterion(test_outputs, y_test)
        test_losses.append(test_loss.item())
        test_preds = (torch.sigmoid(test_outputs) > 0.5).float()
        test_acc = accuracy_score(y_test.numpy(), test_preds.numpy())
        test_accuracies.append(test_acc)

    if (epoch + 1) % 20 == 0:
        print(f'Época [{epoch+1}/{num_epochs}], Pérdida Entrenamiento: {loss.item():.4f}, Precisión Entrenamiento: {train_acc:.4f}, Pérdida Prueba: {test_loss.item():.4f}, Precisión Prueba: {test_acc:.4f}')

# --- 5. Visualización de Datos make_moons ---
print("\n--- Visualización de Datos make_moons ---")
plt.figure(figsize=(8, 6))
plt.scatter(X[y == 0, 0], X[y == 0, 1], label='Clase 0', alpha=0.7, edgecolors='w')
plt.scatter(X[y == 1, 0], X[y == 1, 1], label='Clase 1', alpha=0.7, edgecolors='w')
plt.title('Datos generados con make_moons')
plt.xlabel('Característica 1')
plt.ylabel('Característica 2')
plt.legend()
plt.grid(True)
plt.show()

# --- 6. Visualización de Gráficas de Entrenamiento y Prueba ---
print("\n--- Gráficas de Entrenamiento y Prueba ---")
epochs_list = range(1, num_epochs + 1)

plt.figure(figsize=(10, 5))
plt.plot(epochs_list, train_losses, 'b', label='Pérdida de Entrenamiento')
plt.plot(epochs_list, test_losses, 'r', label='Pérdida de Prueba')
plt.title('Pérdida de Entrenamiento y Prueba')
plt.xlabel('Épocas')
plt.ylabel('Pérdida')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(epochs_list, train_accuracies, 'b', label='Precisión de Entrenamiento')
plt.plot(epochs_list, test_accuracies, 'r', label='Precisión de Prueba')
plt.title('Precisión de Entrenamiento y Prueba')
plt.xlabel('Épocas')
plt.ylabel('Precisión')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Generador de 2 espirales entrelazadas
def make_spirals(n_points=1000, noise=0.05):
    n = np.sqrt(np.random.rand(n_points)) * 780 * (2 * np.pi) / 360
    d1x = -np.cos(n) * n + np.random.rand(n_points) * noise
    d1y = np.sin(n) * n + np.random.rand(n_points) * noise
    X = np.vstack((np.append(d1x, -d1x), np.append(d1y, -d1y))).T
    y = np.hstack((np.zeros(n_points), np.ones(n_points)))
    return X, y

# Generar los datos
X_spiral, y_spiral = make_spirals(n_points=1000, noise=0.2)

# Visualizar
plt.figure(figsize=(6, 6))
plt.scatter(X_spiral[:, 0], X_spiral[:, 1], c=y_spiral, cmap=plt.cm.Paired, s=10)
plt.title("Dataset de 2 Espirales Enlazadas")
plt.axis("equal")
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# Remove: from sklearn.datasets import make_spiral # This function does not exist in sklearn.datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim

# Generador de 2 espirales entrelazadas (clásico) - Defina la función antes de usarla
def make_two_spirals(n_points, noise=1.0):
    n = np.sqrt(np.random.rand(n_points//2)) * 780 * (2 * np.pi)/360
    d1x = -np.cos(n)*n + np.random.rand(n_points//2) * noise
    d1y = np.sin(n)*n + np.random.rand(n_points//2) * noise
    d2x = np.cos(n)*n + np.random.rand(n_points//2) * noise
    d2y = -np.sin(n)*n + np.random.rand(n_points//2) * noise
    X = np.vstack((np.hstack((d1x,d2x)), np.hstack((d1y,d2y)))).T
    y = np.hstack((np.zeros(n_points//2), np.ones(n_points//2)))
    return X, y


# Preparar datos
X, y = make_two_spirals(1000, noise=0.5) # Usar la función definida localmente
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convertir a tensores
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# Modelo simple
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

# Entrenar modelo
model = SimpleNN()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

epochs = 200
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()
    # Opcional: print de progreso si se desea
    # if (epoch + 1) % 20 == 0:
    #     print(f'Época [{epoch+1}/{epochs}], Pérdida: {loss.item():.4f}')


# Crear malla para visualización
xx, yy = np.meshgrid(np.linspace(X[:,0].min()-1, X[:,0].max()+1, 300),
                     np.linspace(X[:,1].min()-1, X[:,1].max()+1, 300))
grid = np.c_[xx.ravel(), yy.ravel()]
grid_scaled = scaler.transform(grid)
grid_tensor = torch.tensor(grid_scaled, dtype=torch.float32)
with torch.no_grad():
    preds = model(grid_tensor).reshape(xx.shape)

# Graficar
plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, preds, levels=50, cmap="RdBu", alpha=0.6)
plt.scatter(X[:,0], X[:,1], c=y, cmap="bwr", edgecolors='k')
plt.title("Clasificación sobre dos espirales")
plt.xlabel("X1")
plt.ylabel("X2")
plt.tight_layout()
plt.show()

In [ ]:


# Need to import necessary libraries for plotting 3D data
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

# Define the function provided in the prompt if it's not already defined
def make_two_spirals_3D(n_points=1000, noise=0.3):
    n = np.sqrt(np.random.rand(n_points // 2)) * 780 * (2 * np.pi)/360  # ángulos en radianes
    x1 = -np.cos(n) * n + np.random.normal(0, noise, n_points // 2)
    y1 = np.sin(n) * n + np.random.normal(0, noise, n_points // 2)
    z1 = n + np.random.normal(0, noise, n_points // 2)  # añadir dimensión z
    x2 = np.cos(n) * n + np.random.normal(0, noise, n_points // 2)
    y2 = -np.sin(n) * n + np.random.normal(0, noise, n_points // 2)
    z2 = n + np.random.normal(0, noise, n_points // 2)
    X = np.vstack([
        np.stack([x1, y1, z1], axis=1),
        np.stack([x2, y2, z2], axis=1)
    ])
    y = np.hstack([np.zeros(n_points // 2), np.ones(n_points // 2)])
    return X, y

# Generate the 3D spiral data
X_3d, y_3d = make_two_spirals_3D(n_points=2000, noise=0.5) # Increased points for better visualization

# Create a 3D plot
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot the two classes with different colors
ax.scatter(X_3d[y_3d == 0, 0], X_3d[y_3d == 0, 1], X_3d[y_3d == 0, 2], c='blue', marker='o', label='Spiral 0')
ax.scatter(X_3d[y_3d == 1, 0], X_3d[y_3d == 1, 1], X_3d[y_3d == 1, 2], c='red', marker='^', label='Spiral 1')

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('Visualization of Two 3D Spirals')
ax.legend()

# Show the plot
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from mpl_toolkits.mplot3d import Axes3D

# Función para generar espirales 3D
def make_two_spirals_3D(n_points=1000, noise=0.3):
    n = np.sqrt(np.random.rand(n_points // 2)) * 780 * (2 * np.pi)/360
    x1 = -np.cos(n) * n + np.random.normal(0, noise, n_points // 2)
    y1 = np.sin(n) * n + np.random.normal(0, noise, n_points // 2)
    z1 = n + np.random.normal(0, noise, n_points // 2)

    x2 = np.cos(n) * n + np.random.normal(0, noise, n_points // 2)
    y2 = -np.sin(n) * n + np.random.normal(0, noise, n_points // 2)
    z2 = n + np.random.normal(0, noise, n_points // 2)

    X = np.vstack([
        np.stack([x1, y1, z1], axis=1),
        np.stack([x2, y2, z2], axis=1)
    ])
    y = np.hstack([np.zeros(n_points // 2), np.ones(n_points // 2)])
    return X, y

# Preparar datos
X, y = make_two_spirals_3D(2000, noise=0.4)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Tensores
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# Modelo 3D
class Spiral3DClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

model = Spiral3DClassifier()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Entrenamiento
for epoch in range(300):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()
    # print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# Evaluación
model.eval()
with torch.no_grad():
    test_preds = model(X_test_tensor)
    predicted = (test_preds > 0.5).float()
    accuracy = (predicted == y_test_tensor).float().mean().item() * 100
    print(f"Precisión en test: {accuracy:.4f}%")

# Visualización 3D del resultado (solo muestra puntos de testeo con color de predicción)
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Colores reales vs predichos
colors = np.where(predicted.numpy().flatten() == 0, 'blue', 'red')
ax.scatter(X_test[:, 0], X_test[:, 1], X_test[:, 2], c=colors, s=15, alpha=0.6, edgecolors='k')

ax.set_title("Clasificación de Espirales 3D (Predicciones del modelo)")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
plt.tight_layout()
plt.show()

In [ ]:
# Generar una rejilla 3D para la visualización
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
z_min, z_max = X[:, 2].min() - 1, X[:, 2].max() + 1

# Reducir la resolución de la rejilla para un cálculo más rápido
# Considera aumentar el número de puntos si tienes un buen GPU para más detalle
xx, yy, zz = np.meshgrid(
    np.linspace(x_min, x_max, 50), # 50 puntos en cada dimensión
    np.linspace(y_min, y_max, 50),
    np.linspace(z_min, z_max, 50)
)
grid_points = np.c_[xx.ravel(), yy.ravel(), zz.ravel()]

# Escalar los puntos de la rejilla usando el mismo scaler de entrenamiento
grid_points_scaled = scaler.transform(grid_points)
grid_points_tensor = torch.tensor(grid_points_scaled, dtype=torch.float32)

# Obtener predicciones del modelo en la rejilla
model.eval()
with torch.no_grad():
    grid_preds = model(grid_points_tensor)
    grid_predictions = (grid_preds > 0.5).float().numpy().reshape(xx.shape)

# Visualización de la superficie de decisión
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# Visualizar la rejilla 3D con colores de predicción
# Usamos scatter para mostrar la "densidad" de la clasificación
# o puedes usar ax.contourf para superficies si la predicción es más suave
# Aquí, como son 0 o 1, el scatter es más representativo de los vóxeles
ax.scatter(xx[grid_predictions == 0], yy[grid_predictions == 0], zz[grid_predictions == 0],
           color='blue', alpha=0.05, s=1) # S = tamaño del punto
ax.scatter(xx[grid_predictions == 1], yy[grid_predictions == 1], zz[grid_predictions == 1],
           color='red', alpha=0.05, s=1)

# También puedes mostrar algunos puntos de entrenamiento para contexto
ax.scatter(X_train[:, 0], X_train[:, 1], X_train[:, 2], c=y_train, cmap='viridis', s=20, alpha=0.8, edgecolors='k')

ax.set_title("Superficie de Decisión de Espirales 3D del Modelo")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
plt.tight_layout()
plt.show()

In [ ]:
# Función para generar espirales 4D (ejemplo simplificado)
def make_two_spirals_4D(n_points=1000, noise=0.3):
    n = np.sqrt(np.random.rand(n_points // 2)) * 780 * (2 * np.pi) / 360

    # Primera espiral 4D
    x1 = -np.cos(n) * n + np.random.normal(0, noise, n_points // 2)
    y1 = np.sin(n) * n + np.random.normal(0, noise, n_points // 2)
    z1 = n + np.random.normal(0, noise, n_points // 2)
    w1 = np.sin(n * 0.5) + np.random.normal(0, noise, n_points // 2) # Cuarta dimensión

    # Segunda espiral 4D
    x2 = np.cos(n) * n + np.random.normal(0, noise, n_points // 2)
    y2 = -np.sin(n) * n + np.random.normal(0, noise, n_points // 2)
    z2 = n + np.random.normal(0, noise, n_points // 2)
    w2 = -np.sin(n * 0.5) + np.random.normal(0, noise, n_points // 2) # Cuarta dimensión opuesta

    X = np.vstack([
        np.stack([x1, y1, z1, w1], axis=1),
        np.stack([x2, y2, z2, w2], axis=1)
    ])
    y = np.hstack([np.zeros(n_points // 2), np.ones(n_points // 2)])
    return X, y

# Preparar datos 4D
X_4d, y_4d = make_two_spirals_4D(2000, noise=0.4)
X_train_4d, X_test_4d, y_train_4d, y_test_4d = train_test_split(X_4d, y_4d, test_size=0.2, random_state=42)
scaler_4d = StandardScaler()
X_train_scaled_4d = scaler_4d.fit_transform(X_train_4d)
X_test_scaled_4d = scaler_4d.transform(X_test_4d)

# Tensores 4D
X_train_tensor_4d = torch.tensor(X_train_scaled_4d, dtype=torch.float32)
y_train_tensor_4d = torch.tensor(y_train_4d, dtype=torch.float32).unsqueeze(1)
X_test_tensor_4d = torch.tensor(X_test_scaled_4d, dtype=torch.float32)
y_test_tensor_4d = torch.tensor(y_test_4d, dtype=torch.float32).unsqueeze(1)

# Modelo 4D (cambiar la capa de entrada a 4 características)
class Spiral4DClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 64), # Aumentamos la complejidad
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

model_4d = Spiral4DClassifier()
criterion_4d = nn.BCELoss()
optimizer_4d = optim.Adam(model_4d.parameters(), lr=0.01)

# Entrenamiento 4D
print("\nEntrenando modelo 4D...")
for epoch in range(500): # Más épocas para 4D
    model_4d.train()
    optimizer_4d.zero_grad()
    outputs_4d = model_4d(X_train_tensor_4d)
    loss_4d = criterion_4d(outputs_4d, y_train_tensor_4d)
    loss_4d.backward()
    optimizer_4d.step()
    # if (epoch + 1) % 50 == 0:
    #     print(f"Epoch {epoch+1}, Loss: {loss_4d.item():.4f}")

# Evaluación 4D
model_4d.eval()
with torch.no_grad():
    test_preds_4d = model_4d(X_test_tensor_4d)
    predicted_4d = (test_preds_4d > 0.5).float()
    accuracy_4d = (predicted_4d == y_test_tensor_4d).float().mean().item() * 100
    print(f"Precisión en test 4D: {accuracy_4d:.4f}%")

# Visualización 3D (proyectando 4D a 3D o usando un color para la 4ta dimensión)
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Proyectamos los puntos 4D de testeo a 3D (ignorando o usando 'w' como color/tamaño)
# Aquí usamos la cuarta dimensión 'w' para el color, que es una forma de "embeberla" visualmente
scatter = ax.scatter(X_test_4d[:, 0], X_test_4d[:, 1], X_test_4d[:, 2],
                     c=X_test_4d[:, 3], cmap='viridis', s=15, alpha=0.6, edgecolors='k')
fig.colorbar(scatter, ax=ax, label='Cuarta Dimensión (w)')

ax.set_title("Espirales 4D Proyectadas a 3D (Color = 4ta Dimensión)")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from mpl_toolkits.mplot3d import Axes3D # Para la visualización 3D proyectada

# Función para generar espirales 5D entrelazadas con ruido variable
def make_two_intertwined_spirals_5D(n_points=2000, noise_base=0.2, noise_dim_mult=[1.0, 1.0, 1.5, 2.0, 2.5]):
    n_half = n_points // 2
    t = np.sqrt(np.random.rand(n_half)) * 10 * (2 * np.pi) # Aumentamos el rango para más complejidad

    # Espiral 1
    x1 = np.cos(t) * t + np.random.normal(0, noise_base * noise_dim_mult[0], n_half)
    y1 = np.sin(t) * t + np.random.normal(0, noise_base * noise_dim_mult[1], n_half)
    z1 = t + np.random.normal(0, noise_base * noise_dim_mult[2], n_half)
    w1 = np.sin(t * 1.5) * 5 + np.random.normal(0, noise_base * noise_dim_mult[3], n_half) # 4ta dim
    v1 = np.cos(t * 2.0) * 3 + np.random.normal(0, noise_base * noise_dim_mult[4], n_half) # 5ta dim

    # Espiral 2 (entrelazada, con desfase y curvatura diferente)
    # Se entrelaza mediante un desfase de pi y una manipulación de las dimensiones
    x2 = -np.cos(t + np.pi) * t + np.random.normal(0, noise_base * noise_dim_mult[0], n_half)
    y2 = -np.sin(t + np.pi) * t + np.random.normal(0, noise_base * noise_dim_mult[1], n_half)
    z2 = -t + np.random.normal(0, noise_base * noise_dim_mult[2], n_half) # Z invertida para cruce
    w2 = -np.sin(t * 1.5 + np.pi/2) * 5 + np.random.normal(0, noise_base * noise_dim_mult[3], n_half) # 4ta dim entrelazada
    v2 = -np.cos(t * 2.0 + np.pi/2) * 3 + np.random.normal(0, noise_base * noise_dim_mult[4], n_half) # 5ta dim entrelazada

    X = np.vstack([
        np.stack([x1, y1, z1, w1, v1], axis=1),
        np.stack([x2, y2, z2, w2, v2], axis=1)
    ])
    y = np.hstack([np.zeros(n_half), np.ones(n_half)])
    return X, y

# Preparar datos 5D
X_5d, y_5d = make_two_intertwined_spirals_5D(n_points=4000, noise_base=0.3) # Más puntos y ruido base
X_train_5d, X_test_5d, y_train_5d, y_test_5d = train_test_split(X_5d, y_5d, test_size=0.2, random_state=42)
scaler_5d = StandardScaler()
X_train_scaled_5d = scaler_5d.fit_transform(X_train_5d)
X_test_scaled_5d = scaler_5d.transform(X_test_5d)

# Tensores 5D
X_train_tensor_5d = torch.tensor(X_train_scaled_5d, dtype=torch.float32)
y_train_tensor_5d = torch.tensor(y_train_5d, dtype=torch.float32).unsqueeze(1)
X_test_tensor_5d = torch.tensor(X_test_scaled_5d, dtype=torch.float32)
y_test_tensor_5d = torch.tensor(y_test_5d, dtype=torch.float32).unsqueeze(1)

# Modelo 5D (cambiar la capa de entrada a 5 características y aumentar la complejidad)
class Spiral5DClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(5, 128), # Capa de entrada 5, más neuronas
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

model_5d = Spiral5DClassifier()
criterion_5d = nn.BCELoss()
optimizer_5d = optim.Adam(model_5d.parameters(), lr=0.005) # Reducir learning rate para estabilidad

# Entrenamiento 5D
print("\nEntrenando modelo 5D con espirales entrelazadas y ruido variable...")
for epoch in range(1000): # Más épocas y posiblemente más pequeñas para converger
    model_5d.train()
    optimizer_5d.zero_grad()
    outputs_5d = model_5d(X_train_tensor_5d)
    loss_5d = criterion_5d(outputs_5d, y_train_tensor_5d)
    loss_5d.backward()
    optimizer_5d.step()
    # if (epoch + 1) % 100 == 0:
    #     print(f"Epoch {epoch+1}, Loss: {loss_5d.item():.4f}")

# Evaluación 5D
model_5d.eval()
with torch.no_grad():
    test_preds_5d = model_5d(X_test_tensor_5d)
    predicted_5d = (test_preds_5d > 0.5).float()
    accuracy_5d = (predicted_5d == y_test_tensor_5d).float().mean().item() * 100
    print(f"Precisión en test 5D: {accuracy_5d:.4f}%")

# Visualización 3D (proyectando 5D a 3D)
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Proyectamos los puntos 5D de testeo a 3D (por ejemplo, x, y, z)
# Y usamos las otras dos dimensiones (w y v) para el color y el tamaño
# para tratar de visualizar la complejidad adicional.
scatter = ax.scatter(X_test_5d[:, 0], X_test_5d[:, 1], X_test_5d[:, 2],
                     c=X_test_5d[:, 3], # Color basado en la 4ta dimensión
                     s=10 + 20 * (X_test_5d[:, 4] - X_test_5d[:, 4].min()) / (X_test_5d[:, 4].max() - X_test_5d[:, 4].min()), # Tamaño basado en la 5ta dimensión
                     cmap='viridis', alpha=0.6, edgecolors='k')
fig.colorbar(scatter, ax=ax, label='Cuarta Dimensión (w)')

ax.set_title("Espirales 5D Entrelazadas Proyectadas a 3D")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from mpl_toolkits.mplot3d import Axes3D # Para la visualización proyectada
from sklearn.manifold import TSNE # Para visualizar en 2D/3D

# Función para generar datos en 10D con múltiples clases complejas
def make_complex_10D_data(n_points_per_class=1000, n_classes=4, noise_factor=0.8):
    data = []
    labels = []
    total_points = n_points_per_class * n_classes

    # Puntos base para la variación
    base_t = np.linspace(-np.pi, np.pi, n_points_per_class)

    for i in range(n_classes):
        # Cada clase tendrá una estructura ligeramente diferente
        # Usamos funciones trigonométricas y polinómicas para crear no-linealidad
        t = base_t + np.random.rand(n_points_per_class) * 0.5 * (2*np.pi) # Añadir variación a 't'

        # Generar 10 dimensiones para cada clase
        # La complejidad viene de la combinación de senos/cosenos y potencias
        # y la dependencia de 't' y otros factores aleatorios
        d1 = np.sin(t * (i + 1)) * t + np.random.normal(0, noise_factor * 0.5, n_points_per_class)
        d2 = np.cos(t * (i + 1)) * t + np.random.normal(0, noise_factor * 0.5, n_points_per_class)
        d3 = np.sin(t * (i + 0.5)) + np.cos(t * 2) + np.random.normal(0, noise_factor * 0.7, n_points_per_class)
        d4 = np.sin(t * 0.7) * (i + 1) + t**2 * 0.1 + np.random.normal(0, noise_factor * 0.8, n_points_per_class)
        d5 = np.cos(t * 1.2) * (i + 1) - t**2 * 0.1 + np.random.normal(0, noise_factor * 0.9, n_points_per_class)
        d6 = np.sin(t * 0.3) * np.cos(t * 0.8) * (i + 1) + np.random.normal(0, noise_factor * 1.0, n_points_per_class)
        d7 = np.tanh(t * 0.5) * (i + 1) + np.random.normal(0, noise_factor * 1.1, n_points_per_class)
        d8 = np.exp(-t**2 / 10) * np.sin(t * 3) * (i + 1) + np.random.normal(0, noise_factor * 1.2, n_points_per_class)
        d9 = np.log(np.abs(t) + 1) * np.cos(t * 0.5) * (i + 1) + np.random.normal(0, noise_factor * 1.3, n_points_per_class)
        d10 = (t + np.random.rand(n_points_per_class) * 2) * (i % 2 * 2 - 1) + np.random.normal(0, noise_factor * 1.4, n_points_per_class) # Última dim con variación

        class_data = np.vstack([d1, d2, d3, d4, d5, d6, d7, d8, d9, d10]).T
        data.append(class_data)
        labels.append(np.full(n_points_per_class, i)) # Etiquetas de clase

    X = np.vstack(data)
    y = np.hstack(labels)
    return X, y

# Generar datos
n_classes = 4 # Ahora 4 clases
X_10d, y_10d = make_complex_10D_data(n_points_per_class=2000, n_classes=n_classes, noise_factor=0.6) # Más puntos

# Dividir y escalar los datos
X_train_10d, X_test_10d, y_train_10d, y_test_10d = train_test_split(X_10d, y_10d, test_size=0.2, random_state=42)
scaler_10d = StandardScaler()
X_train_scaled_10d = scaler_10d.fit_transform(X_train_10d)
X_test_scaled_10d = scaler_10d.transform(X_test_10d)

# Convertir a tensores de PyTorch
X_train_tensor_10d = torch.tensor(X_train_scaled_10d, dtype=torch.float32)
# Para clasificación multiclase, la etiqueta debe ser LongTensor y no one-hot codificada inicialmente
y_train_tensor_10d = torch.tensor(y_train_10d, dtype=torch.long)
X_test_tensor_10d = torch.tensor(X_test_scaled_10d, dtype=torch.float32)
y_test_tensor_10d = torch.tensor(y_test_10d, dtype=torch.long)

# Modelo 10D (clasificación multiclase)
class Complex10DClassifier(nn.Module):
    def __init__(self, num_features=10, num_classes=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(num_features, 256), # Más neuronas
            nn.ReLU(),
            nn.Dropout(0.3), # Añadimos Dropout para evitar sobreajuste
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes) # Salida con num_classes para clasificación multiclase
        )
    def forward(self, x):
        return self.net(x)

model_10d = Complex10DClassifier(num_features=10, num_classes=n_classes)
# Usamos CrossEntropyLoss para clasificación multiclase
criterion_10d = nn.CrossEntropyLoss()
optimizer_10d = optim.Adam(model_10d.parameters(), lr=0.001, weight_decay=1e-5) # LR más bajo y L2 regularization

# Entrenamiento 10D
print("\nEntrenando modelo 10D con manifolds complejos y ruido heterogéneo...")
num_epochs = 2000 # Más épocas para un problema más difícil
for epoch in range(num_epochs):
    model_10d.train()
    optimizer_10d.zero_grad()
    outputs_10d = model_10d(X_train_tensor_10d)
    loss_10d = criterion_10d(outputs_10d, y_train_tensor_10d)
    loss_10d.backward()
    optimizer_10d.step()
    # if (epoch + 1) % 100 == 0:
    #     print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss_10d.item():.4f}")

# Evaluación 10D
model_10d.eval()
with torch.no_grad():
    test_outputs_10d = model_10d(X_test_tensor_10d)
    # Para CrossEntropyLoss, el output es logits, necesitamos argmax para la clase predicha
    _, predicted_10d = torch.max(test_outputs_10d, 1)
    correct = (predicted_10d == y_test_tensor_10d).float().sum()
    accuracy_10d = (correct / y_test_tensor_10d.size(0)).item() * 100
    print(f"Precisión en test 10D: {accuracy_10d:.4f}%")

# Visualización usando t-SNE para reducir 10D a 2D
# Solo visualizamos un subconjunto de los datos de testeo para que sea manejable
if X_test_scaled_10d.shape[0] > 5000: # Limitamos para t-SNE si hay demasiados puntos
    idx = np.random.choice(X_test_scaled_10d.shape[0], 5000, replace=False)
    X_test_sample = X_test_scaled_10d[idx]
    y_test_sample = y_test_tensor_10d[idx].numpy()
    predicted_sample = predicted_10d[idx].numpy()
else:
    X_test_sample = X_test_scaled_10d
    y_test_sample = y_test_tensor_10d.numpy()
    predicted_sample = predicted_10d.numpy()

print("\nRealizando reducción de dimensionalidad con t-SNE (puede tardar un poco)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000) # Más iteraciones
X_tsne = tsne.fit_transform(X_test_sample)

plt.figure(figsize=(12, 6))

# Subplot 1: Colores según las etiquetas reales
plt.subplot(1, 2, 1)
scatter1 = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_test_sample, cmap='viridis', alpha=0.7, s=10)
plt.title("Proyección t-SNE de Datos Reales (10D a 2D)")
plt.xlabel("Componente t-SNE 1")
plt.ylabel("Componente t-SNE 2")
plt.colorbar(scatter1, ticks=range(n_classes), label='Clase Real')

# Subplot 2: Colores según las predicciones del modelo
plt.subplot(1, 2, 2)
scatter2 = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=predicted_sample, cmap='viridis', alpha=0.7, s=10)
plt.title("Proyección t-SNE de Predicciones del Modelo (10D a 2D)")
plt.xlabel("Componente t-SNE 1")
plt.ylabel("Componente t-SNE 2")
plt.colorbar(scatter2, ticks=range(n_classes), label='Clase Predicha')

plt.tight_layout()
plt.show()

# Opcional: Visualización 3D con t-SNE
# tsne_3d = TSNE(n_components=3, random_state=42, perplexity=30, n_iter=1000)
# X_tsne_3d = tsne_3d.fit_transform(X_test_sample)
# fig_3d = plt.figure(figsize=(10, 8))
# ax_3d = fig_3d.add_subplot(111, projection='3d')
# ax_3d.scatter(X_tsne_3d[:, 0], X_tsne_3d[:, 1], X_tsne_3d[:, 2], c=y_test_sample, cmap='viridis', alpha=0.7, s=10)
# ax_3d.set_title("Proyección t-SNE 3D de Datos Reales")
# plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.manifold import TSNE

# Función para generar datos en 20D con clases caóticas y discontinuas
def make_chaotic_20D_data(n_points=10000, n_classes=4, noise_level=1.0):
    data = np.random.rand(n_points, 20) * 10 - 5 # Puntos aleatorios en un rango de -5 a 5
    labels = np.zeros(n_points, dtype=int)

    for i in range(n_points):
        p = data[i]

        # Criterios de clasificación caóticos y combinatorios
        # Esto NO es una función suave. Son condiciones booleanas arbitrarias.
        if (p[0] > 2 and p[3] < -1 and p[7] > 3) or \
           (p[1] < -3 and p[5] > 4) or \
           (np.abs(p[9]) < 0.5 and p[12] > 2 and p[18] < -2):
            labels[i] = 0
        elif (p[2] > 1 and p[4] < -0.5) or \
             (p[6] < -2 and p[8] > 1.5 and np.sum(p[10:13]) > 5):
            labels[i] = 1
        elif (p[10] > 0 and p[11] < 0 and p[13] > 0 and p[14] < 0) or \
             (np.sin(p[15]) * np.cos(p[16]) > 0.5 and p[17] < 1):
            labels[i] = 2
        else:
            labels[i] = 3 # Clase por defecto, o la cuarta clase

        # Añadir ruido con distribución no gaussiana y asimétrica
        # Ruido exponencial en algunas dimensiones para hacerla más difícil
        data[i, 0] += np.random.exponential(noise_level * 0.1) * np.random.choice([-1, 1])
        data[i, 1] += np.random.normal(0, noise_level * 0.05)
        data[i, 5] += np.random.poisson(noise_level * 0.5) * np.random.choice([-1, 1])
        data[i, 19] += np.random.uniform(-noise_level, noise_level) # Ruido uniforme fuerte

    # Asegurarse de que haya al menos 4 clases (si no, el critero aleatorio podría no generarlas todas)
    unique_labels = np.unique(labels)
    if len(unique_labels) < n_classes:
        print(f"Advertencia: Generadas solo {len(unique_labels)} clases. Aumenta n_points o ajusta la lógica.")
        # Ajustar n_classes si no se generan todas
        n_classes = len(unique_labels)


    return data, labels, n_classes

# Generar datos caóticos
n_classes_chaos = 4 # Intentamos 4 clases, la función podría generar menos si los criterios son restrictivos
X_chaos, y_chaos, n_actual_classes = make_chaotic_20D_data(n_points=20000, n_classes=n_classes_chaos, noise_level=0.8) # Más puntos

# Dividir y escalar los datos
X_train_chaos, X_test_chaos, y_train_chaos, y_test_chaos = train_test_split(X_chaos, y_chaos, test_size=0.2, random_state=42)
scaler_chaos = StandardScaler()
X_train_scaled_chaos = scaler_chaos.fit_transform(X_train_chaos)
X_test_scaled_chaos = scaler_chaos.transform(X_test_chaos)

# Convertir a tensores de PyTorch
X_train_tensor_chaos = torch.tensor(X_train_scaled_chaos, dtype=torch.float32)
y_train_tensor_chaos = torch.tensor(y_train_chaos, dtype=torch.long)
X_test_tensor_chaos = torch.tensor(X_test_scaled_chaos, dtype=torch.float32)
y_test_tensor_chaos = torch.tensor(y_test_chaos, dtype=torch.long)

# Modelo 20D (aún más profundo y ancho)
class Chaotic20DClassifier(nn.Module):
    def __init__(self, num_features=20, num_classes=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(num_features, 512), # Gran primera capa
            nn.LeakyReLU(0.01), # Usar LeakyReLU para evitar neuronas muertas
            nn.Dropout(0.4),
            nn.Linear(512, 512),
            nn.LeakyReLU(0.01),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.01),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.01),
            nn.Linear(128, num_classes) # Salida con num_classes
        )
    def forward(self, x):
        return self.net(x)

model_chaos = Chaotic20DClassifier(num_features=20, num_classes=n_actual_classes)
criterion_chaos = nn.CrossEntropyLoss()
optimizer_chaos = optim.Adam(model_chaos.parameters(), lr=0.0005, weight_decay=1e-5) # LR aún más bajo, fuerte regularización

# Entrenamiento
print(f"\nEntrenando modelo 20D con {n_actual_classes} clases caóticas y ruido extremo...")
num_epochs_chaos = 3000 # Muchas más épocas
for epoch in range(num_epochs_chaos):
    model_chaos.train()
    optimizer_chaos.zero_grad()
    outputs_chaos = model_chaos(X_train_tensor_chaos)
    loss_chaos = criterion_chaos(outputs_chaos, y_train_tensor_chaos)
    loss_chaos.backward()
    optimizer_chaos.step()
    # if (epoch + 1) % 200 == 0:
    #     print(f"Epoch {epoch+1}/{num_epochs_chaos}, Loss: {loss_chaos.item():.4f}")

# Evaluación
model_chaos.eval()
with torch.no_grad():
    test_outputs_chaos = model_chaos(X_test_tensor_chaos)
    _, predicted_chaos = torch.max(test_outputs_chaos, 1)
    correct_chaos = (predicted_chaos == y_test_tensor_chaos).float().sum()
    accuracy_chaos = (correct_chaos / y_test_tensor_chaos.size(0)).item() * 100
    print(f"Precisión en test 20D (Caótico): {accuracy_chaos:.4f}%")

# Visualización usando t-SNE para reducir 20D a 2D (con un subconjunto)
if X_test_scaled_chaos.shape[0] > 5000:
    idx_tsne = np.random.choice(X_test_scaled_chaos.shape[0], 5000, replace=False)
    X_test_sample_chaos = X_test_scaled_chaos[idx_tsne]
    y_test_sample_chaos = y_test_tensor_chaos[idx_tsne].numpy()
    predicted_sample_chaos = predicted_chaos[idx_tsne].numpy()
else:
    X_test_sample_chaos = X_test_scaled_chaos
    y_test_sample_chaos = y_test_tensor_chaos.numpy()
    predicted_sample_chaos = predicted_chaos.numpy()


print("\nRealizando reducción de dimensionalidad con t-SNE para visualizar el caos (puede tardar bastante)...")
tsne_chaos = TSNE(n_components=2, random_state=42, perplexity=50, n_iter=2000, learning_rate='auto', init='random') # Ajustes para datos complejos
X_tsne_chaos = tsne_chaos.fit_transform(X_test_sample_chaos)

plt.figure(figsize=(14, 7))

plt.subplot(1, 2, 1)
scatter1_chaos = plt.scatter(X_tsne_chaos[:, 0], X_tsne_chaos[:, 1], c=y_test_sample_chaos, cmap='viridis', alpha=0.7, s=10)
plt.title("Proyección t-SNE de Datos Reales (20D a 2D) - Caótico")
plt.xlabel("Componente t-SNE 1")
plt.ylabel("Componente t-SNE 2")
plt.colorbar(scatter1_chaos, ticks=range(n_actual_classes), label='Clase Real')

plt.subplot(1, 2, 2)
scatter2_chaos = plt.scatter(X_tsne_chaos[:, 0], X_tsne_chaos[:, 1], c=predicted_sample_chaos, cmap='viridis', alpha=0.7, s=10)
plt.title("Proyección t-SNE de Predicciones del Modelo (20D a 2D) - Caótico")
plt.xlabel("Componente t-SNE 1")
plt.ylabel("Componente t-SNE 2")
plt.colorbar(scatter2_chaos, ticks=range(n_actual_classes), label='Clase Predicha')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.manifold import TSNE

# Detectar si hay una GPU disponible y usarla, de lo contrario, usar la CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

# Función para generar datos en 20D con clases caóticas y discontinuas
def make_chaotic_20D_data(n_points=10000, n_classes=4, noise_level=1.0):
    data = np.random.rand(n_points, 20) * 10 - 5 # Puntos aleatorios en un rango de -5 a 5
    labels = np.zeros(n_points, dtype=int)

    for i in range(n_points):
        p = data[i]

        # Criterios de clasificación caóticos y combinatorios
        # Esto NO es una función suave. Son condiciones booleanas arbitrarias.
        if (p[0] > 2 and p[3] < -1 and p[7] > 3) or \
           (p[1] < -3 and p[5] > 4) or \
           (np.abs(p[9]) < 0.5 and p[12] > 2 and p[18] < -2):
            labels[i] = 0
        elif (p[2] > 1 and p[4] < -0.5) or \
             (p[6] < -2 and p[8] > 1.5 and np.sum(p[10:13]) > 5):
            labels[i] = 1
        elif (p[10] > 0 and p[11] < 0 and p[13] > 0 and p[14] < 0) or \
             (np.sin(p[15]) * np.cos(p[16]) > 0.5 and p[17] < 1):
            labels[i] = 2
        else:
            labels[i] = 3 # Clase por defecto, o la cuarta clase

        # Añadir ruido con distribución no gaussiana y asimétrica
        # Ruido exponencial en algunas dimensiones para hacerla más difícil
        data[i, 0] += np.random.exponential(noise_level * 0.1) * np.random.choice([-1, 1])
        data[i, 1] += np.random.normal(0, noise_level * 0.05)
        data[i, 5] += np.random.poisson(noise_level * 0.5) * np.random.choice([-1, 1])
        data[i, 19] += np.random.uniform(-noise_level, noise_level) # Ruido uniforme fuerte

    # Asegurarse de que haya al menos 4 clases (si no, el critero aleatorio podría no generarlas todas)
    unique_labels = np.unique(labels)
    if len(unique_labels) < n_classes:
        print(f"Advertencia: Generadas solo {len(unique_labels)} clases. Aumenta n_points o ajusta la lógica.")
        # Ajustar n_classes si no se generan todas
        n_classes = len(unique_labels)


    return data, labels, n_classes

# Generar datos caóticos
n_classes_chaos = 4 # Intentamos 4 clases, la función podría generar menos si los criterios son restrictivos
X_chaos, y_chaos, n_actual_classes = make_chaotic_20D_data(n_points=20000, n_classes=n_classes_chaos, noise_level=0.8) # Más puntos

# Dividir y escalar los datos
X_train_chaos, X_test_chaos, y_train_chaos, y_test_chaos = train_test_split(X_chaos, y_chaos, test_size=0.2, random_state=42)
scaler_chaos = StandardScaler()
X_train_scaled_chaos = scaler_chaos.fit_transform(X_train_chaos)
X_test_scaled_chaos = scaler_chaos.transform(X_test_chaos)

# Convertir a tensores de PyTorch y moverlos a la GPU
X_train_tensor_chaos = torch.tensor(X_train_scaled_chaos, dtype=torch.float32).to(device)
y_train_tensor_chaos = torch.tensor(y_train_chaos, dtype=torch.long).to(device)
X_test_tensor_chaos = torch.tensor(X_test_scaled_chaos, dtype=torch.float32).to(device)
y_test_tensor_chaos = torch.tensor(y_test_chaos, dtype=torch.long).to(device)

# Modelo 20D (aún más profundo y ancho)
class Chaotic20DClassifier(nn.Module):
    def __init__(self, num_features=20, num_classes=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(num_features, 512), # Gran primera capa
            nn.LeakyReLU(0.01), # Usar LeakyReLU para evitar neuronas muertas
            nn.Dropout(0.4),
            nn.Linear(512, 512),
            nn.LeakyReLU(0.01),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.01),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.01),
            nn.Linear(128, num_classes) # Salida con num_classes
        )
    def forward(self, x):
        return self.net(x)

model_chaos = Chaotic20DClassifier(num_features=20, num_classes=n_actual_classes).to(device) # Mover el modelo a la GPU
criterion_chaos = nn.CrossEntropyLoss()
optimizer_chaos = optim.Adam(model_chaos.parameters(), lr=0.0005, weight_decay=1e-5) # LR aún más bajo, fuerte regularización

# Entrenamiento
print(f"\nEntrenando modelo 20D con {n_actual_classes} clases caóticas y ruido extremo en la {device}...")
num_epochs_chaos = 3000 # Muchas más épocas
for epoch in range(num_epochs_chaos):
    model_chaos.train()
    optimizer_chaos.zero_grad()
    outputs_chaos = model_chaos(X_train_tensor_chaos)
    loss_chaos = criterion_chaos(outputs_chaos, y_train_tensor_chaos)
    loss_chaos.backward()
    optimizer_chaos.step()
    # if (epoch + 1) % 200 == 0:
    #     print(f"Epoch {epoch+1}/{num_epochs_chaos}, Loss: {loss_chaos.item():.4f}")

# Evaluación
model_chaos.eval()
with torch.no_grad():
    test_outputs_chaos = model_chaos(X_test_tensor_chaos)
    _, predicted_chaos = torch.max(test_outputs_chaos, 1)
    correct_chaos = (predicted_chaos == y_test_tensor_chaos).float().sum()
    accuracy_chaos = (correct_chaos / y_test_tensor_chaos.size(0)).item() * 100
    print(f"Precisión en test 20D (Caótico): {accuracy_chaos:.4f}%")

# Visualización usando t-SNE para reducir 20D a 2D (con un subconjunto)
# Los datos para t-SNE deben estar en la CPU
if X_test_scaled_chaos.shape[0] > 5000:
    idx_tsne = np.random.choice(X_test_scaled_chaos.shape[0], 5000, replace=False)
    X_test_sample_chaos = X_test_scaled_chaos[idx_tsne]
    y_test_sample_chaos = y_test_tensor_chaos[idx_tsne].cpu().numpy() # Mover a CPU
    predicted_sample_chaos = predicted_chaos[idx_tsne].cpu().numpy() # Mover a CPU
else:
    X_test_sample_chaos = X_test_scaled_chaos
    y_test_sample_chaos = y_test_tensor_chaos.cpu().numpy() # Mover a CPU
    predicted_sample_chaos = predicted_chaos.cpu().numpy() # Mover a CPU


print("\nRealizando reducción de dimensionalidad con t-SNE para visualizar el caos (puede tardar bastante)...")
tsne_chaos = TSNE(n_components=2, random_state=42, perplexity=50, n_iter=2000, learning_rate='auto', init='random') # Ajustes para datos complejos
X_tsne_chaos = tsne_chaos.fit_transform(X_test_sample_chaos)

plt.figure(figsize=(14, 7))

plt.subplot(1, 2, 1)
scatter1_chaos = plt.scatter(X_tsne_chaos[:, 0], X_tsne_chaos[:, 1], c=y_test_sample_chaos, cmap='viridis', alpha=0.7, s=10)
plt.title("Proyección t-SNE de Datos Reales (20D a 2D) - Caótico")
plt.xlabel("Componente t-SNE 1")
plt.ylabel("Componente t-SNE 2")
plt.colorbar(scatter1_chaos, ticks=range(n_actual_classes), label='Clase Real')

plt.subplot(1, 2, 2)
scatter2_chaos = plt.scatter(X_tsne_chaos[:, 0], X_tsne_chaos[:, 1], c=predicted_sample_chaos, cmap='viridis', alpha=0.7, s=10)
plt.title("Proyección t-SNE de Predicciones del Modelo (20D a 2D) - Caótico")
plt.xlabel("Componente t-SNE 1")
plt.ylabel("Componente t-SNE 2")
plt.colorbar(scatter2_chaos, ticks=range(n_actual_classes), label='Clase Predicha')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.manifold import TSNE

# Asegúrate de usar CUDA si está disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

# --- Función para generar datos en 20D (mismo que antes) ---
def make_chaotic_20D_data(n_points=10000, n_classes=4, noise_level=1.0):
    data = np.random.rand(n_points, 20) * 10 - 5
    labels = np.zeros(n_points, dtype=int)

    for i in range(n_points):
        p = data[i]

        if (p[0] > 2 and p[3] < -1 and p[7] > 3) or \
           (p[1] < -3 and p[5] > 4) or \
           (np.abs(p[9]) < 0.5 and p[12] > 2 and p[18] < -2):
            labels[i] = 0
        elif (p[2] > 1 and p[4] < -0.5) or \
             (p[6] < -2 and p[8] > 1.5 and np.sum(p[10:13]) > 5):
            labels[i] = 1
        elif (p[10] > 0 and p[11] < 0 and p[13] > 0 and p[14] < 0) or \
             (np.sin(p[15]) * np.cos(p[16]) > 0.5 and p[17] < 1):
            labels[i] = 2
        else:
            labels[i] = 3

        data[i, 0] += np.random.exponential(noise_level * 0.1) * np.random.choice([-1, 1])
        data[i, 1] += np.random.normal(0, noise_level * 0.05)
        data[i, 5] += np.random.poisson(noise_level * 0.5) * np.random.choice([-1, 1])
        data[i, 19] += np.random.uniform(-noise_level, noise_level)

    unique_labels = np.unique(labels)
    if len(unique_labels) < n_classes:
        print(f"Advertencia: Generadas solo {len(unique_labels)} clases. Ajustando n_classes.")
        n_classes = len(unique_labels)

    return data, labels, n_classes

# --- Generar datos ---
n_classes_chaos = 4
X_chaos, y_chaos, n_actual_classes = make_chaotic_20D_data(n_points=20000, n_classes=n_classes_chaos, noise_level=0.8)

# --- Dividir y escalar ---
X_train_chaos, X_test_chaos, y_train_chaos, y_test_chaos = train_test_split(X_chaos, y_chaos, test_size=0.2, random_state=42)
scaler_chaos = StandardScaler()
X_train_scaled_chaos = scaler_chaos.fit_transform(X_train_chaos)
X_test_scaled_chaos = scaler_chaos.transform(X_test_chaos)

# --- Convertir a tensores y mover a CUDA ---
X_train_tensor_chaos = torch.tensor(X_train_scaled_chaos, dtype=torch.float32).to(device)
y_train_tensor_chaos = torch.tensor(y_train_chaos, dtype=torch.long).to(device)
X_test_tensor_chaos = torch.tensor(X_test_scaled_chaos, dtype=torch.float32).to(device)
y_test_tensor_chaos = torch.tensor(y_test_chaos, dtype=torch.long).to(device)

# --- Modelo 20D (mismo que antes) ---
class Chaotic20DClassifier(nn.Module):
    def __init__(self, num_features=20, num_classes=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(num_features, 512),
            nn.LeakyReLU(0.01),
            nn.Dropout(0.4),
            nn.Linear(512, 512),
            nn.LeakyReLU(0.01),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.01),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.01),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        return self.net(x)

model_chaos = Chaotic20DClassifier(num_features=20, num_classes=n_actual_classes).to(device) # Mover modelo a CUDA
criterion_chaos = nn.CrossEntropyLoss()
optimizer_chaos = optim.Adam(model_chaos.parameters(), lr=0.0005, weight_decay=1e-5)

# --- Añadir el Learning Rate Scheduler ---
# ReduceLROnPlateau: Reduce el LR cuando la pérdida de validación deja de mejorar
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_chaos,
    mode='min',        # Monitorear la pérdida de entrenamiento para minimizarla
    factor=0.5,        # Multiplicar LR por 0.5 cuando se activa
    patience=200,      # Esperar 200 épocas sin mejora antes de reducir el LR
    verbose=True,      # Imprimir mensaje cuando el LR se reduce
    min_lr=1e-7        # LR mínimo permitido
)

# --- Entrenamiento (con scheduler) ---
print(f"\nEntrenando modelo 20D con {n_actual_classes} clases caóticas y ruido extremo (con scheduler)...")
num_epochs_chaos = 3000
for epoch in range(num_epochs_chaos):
    model_chaos.train()
    optimizer_chaos.zero_grad()
    outputs_chaos = model_chaos(X_train_tensor_chaos)
    loss_chaos = criterion_chaos(outputs_chaos, y_train_tensor_chaos)
    loss_chaos.backward()
    optimizer_chaos.step()

    # Paso del scheduler después de cada época, basado en la pérdida de entrenamiento
    scheduler.step(loss_chaos)

    # if (epoch + 1) % 200 == 0:
    #     print(f"Epoch {epoch+1}/{num_epochs_chaos}, Loss: {loss_chaos.item():.4f}, Current LR: {optimizer_chaos.param_groups[0]['lr']:.6f}")

# --- Evaluación ---
model_chaos.eval()
with torch.no_grad():
    test_outputs_chaos = model_chaos(X_test_tensor_chaos)
    _, predicted_chaos = torch.max(test_outputs_chaos, 1)
    correct_chaos = (predicted_chaos == y_test_tensor_chaos).float().sum()
    accuracy_chaos = (correct_chaos / y_test_tensor_chaos.size(0)).item() * 100
    print(f"Precisión en test 20D (Caótico, Mejorado): {accuracy_chaos:.4f}%")

# --- Visualización con t-SNE (mismo que antes) ---
# Mover datos a CPU para t-SNE (scikit-learn no usa CUDA)
X_test_scaled_chaos_cpu = X_test_scaled_chaos
y_test_tensor_chaos_cpu = y_test_tensor_chaos.cpu().numpy()
predicted_chaos_cpu = predicted_chaos.cpu().numpy()


if X_test_scaled_chaos_cpu.shape[0] > 5000:
    idx_tsne = np.random.choice(X_test_scaled_chaos_cpu.shape[0], 5000, replace=False)
    X_test_sample_chaos = X_test_scaled_chaos_cpu[idx_tsne]
    y_test_sample_chaos = y_test_tensor_chaos_cpu[idx_tsne]
    predicted_sample_chaos = predicted_chaos_cpu[idx_tsne]
else:
    X_test_sample_chaos = X_test_scaled_chaos_cpu
    y_test_sample_chaos = y_test_tensor_chaos_cpu
    predicted_sample_chaos = predicted_chaos_cpu


print("\nRealizando reducción de dimensionalidad con t-SNE para visualizar el caos (puede tardar bastante)...")
tsne_chaos = TSNE(n_components=2, random_state=42, perplexity=50, n_iter=2000, learning_rate='auto', init='random')
X_tsne_chaos = tsne_chaos.fit_transform(X_test_sample_chaos)

plt.figure(figsize=(14, 7))

plt.subplot(1, 2, 1)
scatter1_chaos = plt.scatter(X_tsne_chaos[:, 0], X_tsne_chaos[:, 1], c=y_test_sample_chaos, cmap='viridis', alpha=0.7, s=10)
plt.title("Proyección t-SNE de Datos Reales (20D a 2D) - Caótico")
plt.xlabel("Componente t-SNE 1")
plt.ylabel("Componente t-SNE 2")
plt.colorbar(scatter1_chaos, ticks=range(n_actual_classes), label='Clase Real')

plt.subplot(1, 2, 2)
scatter2_chaos = plt.scatter(X_tsne_chaos[:, 0], X_tsne_chaos[:, 1], c=predicted_sample_chaos, cmap='viridis', alpha=0.7, s=10)
plt.title("Proyección t-SNE de Predicciones del Modelo (20D a 2D) - Caótico")
plt.xlabel("Componente t-SNE 1")
plt.ylabel("Componente t-SNE 2")
plt.colorbar(scatter2_chaos, ticks=range(n_actual_classes), label='Clase Predicha')

plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score

class ImprovedChaotic20DClassifier(nn.Module):
    def __init__(self, num_features=20, num_classes=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(num_features, 1024),
            nn.BatchNorm1d(1024),
            nn.LeakyReLU(0.01),
            nn.Dropout(0.4),

            nn.Linear(1024, 1024),
            nn.BatchNorm1d(1024),
            nn.LeakyReLU(0.01),
            nn.Dropout(0.4),

            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.01),
            nn.Dropout(0.3),

            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.01),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.01),
            nn.Dropout(0.2),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.01),

            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)

def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    all_preds = []
    all_targets = []

    for x, y in dataloader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        preds = out.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(y.cpu().numpy())

    avg_loss = total_loss / len(dataloader.dataset)
    acc = accuracy_score(all_targets, all_preds)
    return avg_loss, acc

def eval_epoch(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)

            total_loss += loss.item() * x.size(0)
            preds = out.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(y.cpu().numpy())

    avg_loss = total_loss / len(dataloader.dataset)
    acc = accuracy_score(all_targets, all_preds)
    return avg_loss, acc

def train_loop(model, train_loader, val_loader, criterion, optimizer, scheduler, device, epochs=300):
    best_val_loss = float('inf')
    for epoch in range(1, epochs+1):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = eval_epoch(model, val_loader, criterion, device)

        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'best_model.pth')

        if epoch % 10 == 0 or epoch == 1:
            print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
                  f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

if __name__ == "__main__":
    # Configuración básica
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    num_features = 20
    num_classes = 4

    # Dataset dummy (reemplaza con tus datos)
    X_train = torch.randn(1000, num_features)
    y_train = torch.randint(0, num_classes, (1000,))
    X_val = torch.randn(200, num_features)
    y_val = torch.randint(0, num_classes, (200,))

    train_dataset = TensorDataset(X_train, y_train)
    val_dataset = TensorDataset(X_val, y_val)
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=64)

    # Modelo, criterio, optimizador y scheduler
    model = ImprovedChaotic20DClassifier(num_features, num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=100, verbose=True)

    # Entrenamiento
    train_loop(model, train_loader, val_loader, criterion, optimizer, scheduler, device, epochs=300)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.manifold import TSNE

# Configuración del dispositivo (CUDA o CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")



## Generación de Datos Caóticos en 20D

def make_chaotic_20D_data(n_points=10000, n_classes=4, noise_level=1.0):
    data = np.random.rand(n_points, 20) * 10 - 5
    labels = np.zeros(n_points, dtype=int)

    for i in range(n_points):
        p = data[i]
        # Criterios de clasificación caóticos y combinatorios
        if (p[0] > 2 and p[3] < -1 and p[7] > 3) or \
           (p[1] < -3 and p[5] > 4) or \
           (np.abs(p[9]) < 0.5 and p[12] > 2 and p[18] < -2):
            labels[i] = 0
        elif (p[2] > 1 and p[4] < -0.5) or \
             (p[6] < -2 and p[8] > 1.5 and np.sum(p[10:13]) > 5):
            labels[i] = 1
        elif (p[10] > 0 and p[11] < 0 and p[13] > 0 and p[14] < 0) or \
             (np.sin(p[15]) * np.cos(p[16]) > 0.5 and p[17] < 1):
            labels[i] = 2
        else:
            labels[i] = 3

        # Añadir ruido con distribución no gaussiana y asimétrica
        data[i, 0] += np.random.exponential(noise_level * 0.1) * np.random.choice([-1, 1])
        data[i, 1] += np.random.normal(0, noise_level * 0.05)
        data[i, 5] += np.random.poisson(noise_level * 0.5) * np.random.choice([-1, 1])
        data[i, 19] += np.random.uniform(-noise_level, noise_level)

    unique_labels = np.unique(labels)
    if len(unique_labels) < n_classes:
        print(f"Advertencia: Generadas solo {len(unique_labels)} clases. Ajustando n_classes.")
        n_classes = len(unique_labels)

    return data, labels, n_classes

# Generar, dividir, escalar y convertir datos a tensores PyTorch
n_classes_chaos = 4
X_chaos, y_chaos, n_actual_classes = make_chaotic_20D_data(n_points=20000, n_classes=n_classes_chaos, noise_level=0.8)

X_train_chaos, X_test_chaos, y_train_chaos, y_test_chaos = train_test_split(X_chaos, y_chaos, test_size=0.2, random_state=42)
scaler_chaos = StandardScaler()
X_train_scaled_chaos = scaler_chaos.fit_transform(X_train_chaos)
X_test_scaled_chaos = scaler_chaos.transform(X_test_chaos)

X_train_tensor_chaos = torch.tensor(X_train_scaled_chaos, dtype=torch.float32).to(device)
y_train_tensor_chaos = torch.tensor(y_train_chaos, dtype=torch.long).to(device)
X_test_tensor_chaos = torch.tensor(X_test_scaled_chaos, dtype=torch.float32).to(device)
y_test_tensor_chaos = torch.tensor(y_test_chaos, dtype=torch.long).to(device)

class Chaotic20DClassifier(nn.Module):
    def __init__(self, num_features=20, num_classes=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(num_features, 1024),
            nn.BatchNorm1d(1024),
            nn.LeakyReLU(0.01),
            nn.Dropout(0.4),

            nn.Linear(1024, 1024),
            nn.BatchNorm1d(1024),
            nn.LeakyReLU(0.01),
            nn.Dropout(0.4),

            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.01),
            nn.Dropout(0.3),

            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.01),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.01),
            nn.Dropout(0.2),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.01),

            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        return self.net(x)

model_chaos = Chaotic20DClassifier(num_features=20, num_classes=n_actual_classes).to(device)
criterion_chaos = nn.CrossEntropyLoss()
optimizer_chaos = optim.Adam(model_chaos.parameters(), lr=0.0005, weight_decay=1e-5)

# Programador de Tasa de Aprendizaje (ReduceLROnPlateau)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_chaos,
    mode='min',
    factor=0.5,
    patience=200,
    verbose=True,
    min_lr=1e-7
)

print(f"\nEntrenando modelo 20D con {n_actual_classes} clases caóticas y ruido extremo (con scheduler)...")
num_epochs_chaos = 3000
for epoch in range(num_epochs_chaos):
    model_chaos.train()
    optimizer_chaos.zero_grad()
    outputs_chaos = model_chaos(X_train_tensor_chaos)
    loss_chaos = criterion_chaos(outputs_chaos, y_train_tensor_chaos)
    loss_chaos.backward()
    optimizer_chaos.step()

    scheduler.step(loss_chaos)

model_chaos.eval()
with torch.no_grad():
    test_outputs_chaos = model_chaos(X_test_tensor_chaos)
    _, predicted_chaos = torch.max(test_outputs_chaos, 1)
    correct_chaos = (predicted_chaos == y_test_tensor_chaos).float().sum()
    accuracy_chaos = (correct_chaos / y_test_tensor_chaos.size(0)).item() * 100
    print(f"Precisión en test 20D (Caótico, Mejorado): {accuracy_chaos:.4f}%")

X_test_scaled_chaos_cpu = X_test_scaled_chaos
y_test_tensor_chaos_cpu = y_test_tensor_chaos.cpu().numpy()
predicted_chaos_cpu = predicted_chaos.cpu().numpy()

if X_test_scaled_chaos_cpu.shape[0] > 5000:
    idx_tsne = np.random.choice(X_test_scaled_chaos_cpu.shape[0], 5000, replace=False)
    X_test_sample_chaos = X_test_scaled_chaos_cpu[idx_tsne]
    y_test_sample_chaos = y_test_tensor_chaos_cpu[idx_tsne]
    predicted_sample_chaos = predicted_chaos_cpu[idx_tsne]
else:
    X_test_sample_chaos = X_test_scaled_chaos_cpu
    y_test_sample_chaos = y_test_tensor_chaos_cpu
    predicted_sample_chaos = predicted_chaos_cpu

print("\nRealizando reducción de dimensionalidad con t-SNE para visualizar el caos (puede tardar bastante)...")
tsne_chaos = TSNE(n_components=2, random_state=42, perplexity=50, n_iter=2000, learning_rate='auto', init='random')
X_tsne_chaos = tsne_chaos.fit_transform(X_test_sample_chaos)

plt.figure(figsize=(14, 7))

plt.subplot(1, 2, 1)
scatter1_chaos = plt.scatter(X_tsne_chaos[:, 0], X_tsne_chaos[:, 1], c=y_test_sample_chaos, cmap='viridis', alpha=0.7, s=10)
plt.title("Proyección t-SNE de Datos Reales (20D a 2D) - Caótico")
plt.xlabel("Componente t-SNE 1")
plt.ylabel("Componente t-SNE 2")
plt.colorbar(scatter1_chaos, ticks=range(n_actual_classes), label='Clase Real')

plt.subplot(1, 2, 2)
scatter2_chaos = plt.scatter(X_tsne_chaos[:, 0], X_tsne_chaos[:, 1], c=predicted_sample_chaos, cmap='viridis', alpha=0.7, s=10)
plt.title("Proyección t-SNE de Predicciones del Modelo (20D a 2D) - Caótico")
plt.xlabel("Componente t-SNE 1")
plt.ylabel("Componente t-SNE 2")
plt.colorbar(scatter2_chaos, ticks=range(n_actual_classes), label='Clase Predicha')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.manifold import TSNE

# Configuración del dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

# --- Generación de datos 20D caóticos ---
def make_chaotic_20D_data(n_points=10000, n_classes=4, noise_level=1.0):
    data = np.random.rand(n_points, 20) * 10 - 5
    labels = np.zeros(n_points, dtype=int)
    for i in range(n_points):
        p = data[i]
        if (p[0] > 2 and p[3] < -1 and p[7] > 3) or \
           (p[1] < -3 and p[5] > 4) or \
           (np.abs(p[9]) < 0.5 and p[12] > 2 and p[18] < -2): labels[i] = 0
        elif (p[2] > 1 and p[4] < -0.5) or \
             (p[6] < -2 and p[8] > 1.5 and np.sum(p[10:13]) > 5): labels[i] = 1
        elif (p[10] > 0 and p[11] < 0 and p[13] > 0 and p[14] < 0) or \
             (np.sin(p[15]) * np.cos(p[16]) > 0.5 and p[17] < 1): labels[i] = 2
        else: labels[i] = 3
        data[i, 0] += np.random.exponential(noise_level * 0.1) * np.random.choice([-1, 1])
        data[i, 1] += np.random.normal(0, noise_level * 0.05)
        data[i, 5] += np.random.poisson(noise_level * 0.5) * np.random.choice([-1, 1])
        data[i, 19] += np.random.uniform(-noise_level, noise_level)
    n_classes = len(np.unique(labels)) if len(np.unique(labels)) < n_classes else n_classes # Ajusta n_classes si no se generan todas
    return data, labels, n_classes

# Generar datos y prepararlos
n_classes_chaos = 4
X_chaos, y_chaos, n_actual_classes = make_chaotic_20D_data(n_points=20000, n_classes=n_classes_chaos, noise_level=0.8)
X_train, X_test, y_train, y_test = train_test_split(X_chaos, y_chaos, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convertir a tensores y mover a CUDA
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train, dtype=torch.long).to(device)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test, dtype=torch.long).to(device)

# --- Definición del modelo 20D ---
class Chaotic20DClassifier(nn.Module):
    def __init__(self, num_features=20, num_classes=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(num_features, 1024), nn.BatchNorm1d(1024), nn.LeakyReLU(0.01), nn.Dropout(0.4),
            nn.Linear(1024, 1024), nn.BatchNorm1d(1024), nn.LeakyReLU(0.01), nn.Dropout(0.4),
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.LeakyReLU(0.01), nn.Dropout(0.3),
            nn.Linear(512, 512), nn.BatchNorm1d(512), nn.LeakyReLU(0.01), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.LeakyReLU(0.01), nn.Dropout(0.2),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.LeakyReLU(0.01),
            nn.Linear(128, num_classes)
        )
    def forward(self, x): return self.net(x)

# Inicializar modelo, criterio, optimizador y scheduler
model = Chaotic20DClassifier(num_features=20, num_classes=n_actual_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=200, verbose=True, min_lr=1e-7)

# --- Entrenamiento del modelo ---
print(f"\nEntrenando modelo 20D con {n_actual_classes} clases caóticas y ruido extremo (con scheduler)...")
num_epochs = 3000
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    loss = criterion(model(X_train_tensor), y_train_tensor)
    loss.backward()
    optimizer.step()
    scheduler.step(loss)

# --- Evaluación del modelo ---
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    _, predicted = torch.max(test_outputs, 1)
    accuracy = (predicted == y_test_tensor).float().sum() / y_test_tensor.size(0) * 100
    print(f"Precisión en test 20D (Caótico, Mejorado): {accuracy:.4f}%")

# --- Visualización con t-SNE ---
X_test_sample = X_test_scaled[np.random.choice(X_test_scaled.shape[0], 5000, replace=False)] if X_test_scaled.shape[0] > 5000 else X_test_scaled
y_test_sample = y_test_tensor.cpu().numpy()[np.random.choice(y_test_tensor.size(0), 5000, replace=False)] if y_test_tensor.size(0) > 5000 else y_test_tensor.cpu().numpy()
predicted_sample = predicted.cpu().numpy()[np.random.choice(predicted.size(0), 5000, replace=False)] if predicted.size(0) > 5000 else predicted.cpu().numpy()

print("\nRealizando reducción de dimensionalidad con t-SNE para visualizar el caos (puede tardar bastante)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=50, n_iter=2000, learning_rate='auto', init='random')
X_tsne = tsne.fit_transform(X_test_sample)

plt.figure(figsize=(14, 7))
plt.subplot(1, 2, 1)
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_test_sample, cmap='viridis', alpha=0.7, s=10)
plt.title("Proyección t-SNE de Datos Reales (20D a 2D) - Caótico")
plt.xlabel("Componente t-SNE 1"); plt.ylabel("Componente t-SNE 2")
plt.colorbar(ticks=range(n_actual_classes), label='Clase Real')

plt.subplot(1, 2, 2)
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=predicted_sample, cmap='viridis', alpha=0.7, s=10)
plt.title("Proyección t-SNE de Predicciones del Modelo (20D a 2D) - Caótico")
plt.xlabel("Componente t-SNE 1"); plt.ylabel("Componente t-SNE 2")
plt.colorbar(ticks=range(n_actual_classes), label='Clase Predicha')
plt.tight_layout(); plt.show()

##RL cerebro

##🧠 Prototipo de RL + Cerebro Modular con MoE

In [ ]:
# 🧠 Prototipo de RL + Cerebro Modular con MoE

import torch
import torch.nn as nn
import torch.nn.functional as F
import random

# ---------- Configuracion base ----------
EMBED_DIM = 32  # Tamaño de las representaciones semánticas
NUM_EXPERTS = 3  # Número inicial de expertos (puedes agregar más)

# ---------- Encoder Semántico ----------
class SemanticEncoder(nn.Module):
    def __init__(self, input_dim, embed_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),
            nn.Linear(64, embed_dim)
        )
    def forward(self, x):
        return self.net(x)

# ---------- Memoria tipo clave-valor ----------
class EpisodicMemory:
    def __init__(self):
        self.keys = []
        self.values = []

    def add(self, key, value):
        self.keys.append(key.detach())
        self.values.append(value)

    def retrieve(self, query, top_k=1):
        if not self.keys:
            return None
        dists = [torch.norm(query - k) for k in self.keys]
        idxs = sorted(range(len(dists)), key=lambda i: dists[i])[:top_k]
        return [self.values[i] for i in idxs]

# ---------- Router Neuronal (RL simple) ----------
class Router(nn.Module):
    def __init__(self, embed_dim, num_experts):
        super().__init__()
        self.linear = nn.Linear(embed_dim, num_experts)
    def forward(self, embedding):
        logits = self.linear(embedding)
        return F.softmax(logits, dim=-1)

# ---------- Expertos (simples y especializados) ----------
class ExpertSum(nn.Module):
    def forward(self, x):  # x = [a, b, ?]
        return torch.tensor([x[0] + x[1]])

class ExpertRand(nn.Module):
    def forward(self, x):
        return torch.tensor([random.randint(0, 10)])

class ExpertZero(nn.Module):
    def forward(self, x):
        return torch.tensor([0.0])

# ---------- Sistema completo ----------
class CognitiveMoE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = SemanticEncoder(input_dim=3, embed_dim=EMBED_DIM)
        self.router = Router(embed_dim=EMBED_DIM, num_experts=NUM_EXPERTS)
        self.memory = EpisodicMemory()
        self.experts = nn.ModuleList([ExpertSum(), ExpertRand(), ExpertZero()])

    def forward(self, x):
        emb = self.encoder(x.float())
        probs = self.router(emb)
        choice = torch.multinomial(probs, 1).item()
        output = self.experts[choice](x)

        # Guardar experiencia
        self.memory.add(emb, choice)

        return output, choice

# ---------- Ejemplo de entorno simulado ----------
def environment(x):  # Input = [a, b, ?], output = reward si acierta
    target = x[0] + x[1]
    return target

# ---------- Ciclo de entrenamiento simulado ----------
model = CognitiveMoE()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for step in range(500):
    a, b = random.randint(0, 10), random.randint(0, 10)
    x = torch.tensor([a, b, 0])
    y_true = environment(x)

    y_pred, expert_id = model(x)
    reward = -abs(y_pred.item() - y_true)  # reward negativo = error

    # Reinforce loss para entrenar router
    emb = model.encoder(x.float())
    probs = model.router(emb)
    log_prob = torch.log(probs[expert_id] + 1e-8)
    loss = -log_prob * reward

    opt.zero_grad()
    loss.backward()
    opt.step()

    if step % 50 == 0:
        print(f"Paso {step}: a={a}, b={b}, experto={expert_id}, pred={y_pred.item():.2f}, reward={reward:.2f}")

#NeuralBiomesModel

In [ ]:
!pip install torch_geometric

In [ ]:
import torch
import torch.nn as nn

class NeuralBiomesModel(nn.Module):
    def __init__(self, input_dim, shared_features_dim, memory_hidden_dim, processing_output_dim, output_dim):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_dim, shared_features_dim),
            nn.ReLU(),
            nn.Linear(shared_features_dim, shared_features_dim)
        )
        self.memory = nn.LSTM(shared_features_dim, memory_hidden_dim, batch_first=True)
        self.processor = nn.Sequential(
            nn.Linear(shared_features_dim + memory_hidden_dim, 128),
            nn.ReLU(),
            nn.Linear(128, processing_output_dim),
            nn.ReLU()
        )
        self.output = nn.Linear(processing_output_dim, output_dim)

    def forward(self, x):
        x_shared = self.shared(x)
        x_seq = x_shared.unsqueeze(1)
        _, (h_n, _) = self.memory(x_seq)
        h_n = h_n.squeeze(0)
        x_combined = torch.cat((x_shared, h_n), dim=1)
        x_processed = self.processor(x_combined)
        return self.output(x_processed)

##Modelo Multimodal Bioinspirado: Imagen + Texto Sintético

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.distributions import Categorical
from torchvision import datasets, transforms
from tqdm import tqdm

# Fake multimodal embedding
def generate_fake_text_embedding(bs, dim=50):
    return torch.randn(bs, dim, device='cuda')

# Router
class Router(nn.Module):
    def __init__(self, in_dim, n_experts):
        super().__init__()
        self.fc = nn.Linear(in_dim, n_experts)

    def forward(self, x):
        probs = F.softmax(self.fc(x), dim=1)
        dist = Categorical(probs)
        idx = dist.sample()
        return idx, dist.log_prob(idx), probs

# Expertos
class MemoryExpert(nn.Module):
    def __init__(self, in_dim, h):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, h, batch_first=True)
        self.fc = nn.Linear(h, 10)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x.unsqueeze(1))
        return self.fc(h_n.squeeze(0))

class LogicExpert(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 10))

    def forward(self, x): return self.net(x)

class InstinctExpert(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, 32), nn.ReLU(), nn.Linear(32, 10))

    def forward(self, x): return self.net(x)

class ContextExpert(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.q = nn.Linear(in_dim, 64)
        self.k = nn.Linear(in_dim, 64)
        self.v = nn.Linear(in_dim, 64)
        self.fc = nn.Linear(64, 10)

    def forward(self, x):
        q, k, v = self.q(x).unsqueeze(1), self.k(x).unsqueeze(1), self.v(x).unsqueeze(1)
        attn = F.softmax(torch.bmm(q, k.transpose(1, 2)), dim=-1)
        return self.fc(torch.bmm(attn, v).squeeze(1))

# MoE
class CorticalMoE(nn.Module):
    def __init__(self, img_dim, txt_dim):
        super().__init__()
        in_dim = img_dim + txt_dim
        self.router = Router(in_dim, 4)
        self.experts = nn.ModuleList([
            MemoryExpert(in_dim, 128),
            LogicExpert(in_dim),
            InstinctExpert(in_dim),
            ContextExpert(in_dim)
        ])

    def forward(self, img, txt):
        x = torch.cat((img, txt), dim=1)
        idx, logp, probs = self.router(x)
        out = torch.stack([e(x) for e in self.experts])
        sel = out[idx, torch.arange(x.size(0))]
        return sel, logp, idx, probs

# Entrenamiento
def train_loop(model, loader, opt, opt_r, epoch, text_dim=50):
    model.train()
    usage = torch.zeros(4, device='cuda')
    total_loss = 0

    for img, lbl in tqdm(loader, desc=f"Época {epoch}", leave=False):
        img, lbl = img.view(img.size(0), -1).cuda(), lbl.cuda()
        txt = generate_fake_text_embedding(img.size(0), text_dim)

        opt.zero_grad(); opt_r.zero_grad()
        out, logp, idx, _ = model(img, txt)

        ce_loss = F.cross_entropy(out, lbl)
        reward = -ce_loss.detach()
        rl_loss = -logp.mean() * reward
        loss = ce_loss + rl_loss

        loss.backward()
        opt.step(); opt_r.step()

        usage += torch.bincount(idx, minlength=4)
        total_loss += loss.item()

    print(f"Loss: {total_loss/len(loader):.4f} | Uso: {usage.tolist()}")

# Main
def main():
    tfm = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x.view(-1))])
    ds = datasets.MNIST('./data', train=True, download=True, transform=tfm)
    dl = DataLoader(ds, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)

    model = CorticalMoE(28*28, 50).cuda()
    opt_r = torch.optim.Adam(model.router.parameters(), lr=1e-3)
    opt = torch.optim.Adam([p for n, p in model.named_parameters() if 'router' not in n], lr=1e-3)

    for e in range(1, 6):
        train_loop(model, dl, opt, opt_r, e)

if __name__ == "__main__":
    torch.manual_seed(42)
    main()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.distributions import Categorical
from torchvision import datasets, transforms
import numpy as np
import random
import matplotlib.pyplot as plt

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# ----------------------------
# 1. Generador de texto falso (embedding aleatorio por imagen)
# ----------------------------
def generate_fake_text_embedding(batch_size, embed_dim=50):
    return torch.randn(batch_size, embed_dim)

# ----------------------------
# 2. Router Tálamo-Cortical
# ----------------------------
class Router(nn.Module):
    def __init__(self, input_dim, n_experts):
        super().__init__()
        self.fc = nn.Linear(input_dim, n_experts)

    def forward(self, x):
        logits = self.fc(x)
        probs = F.softmax(logits, dim=1)
        dist = Categorical(probs)
        expert_idx = dist.sample()
        log_prob = dist.log_prob(expert_idx)
        return expert_idx, log_prob, probs

# ----------------------------
# 3. Expertos
# ----------------------------
class MemoryExpert(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 10)

    def forward(self, x):
        x = x.unsqueeze(1)
        _, (h_n, _) = self.lstm(x)
        return self.fc(h_n.squeeze(0))

class LogicExpert(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x): return self.net(x)

class InstinctExpert(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32), nn.ReLU(),
            nn.Linear(32, 10)
        )

    def forward(self, x): return self.net(x)

class ContextExpert(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.query = nn.Linear(input_dim, 64)
        self.key = nn.Linear(input_dim, 64)
        self.value = nn.Linear(input_dim, 64)
        self.fc = nn.Linear(64, 10)

    def forward(self, x):
        q, k, v = self.query(x).unsqueeze(1), self.key(x).unsqueeze(1), self.value(x).unsqueeze(1)
        attn_weights = F.softmax(torch.bmm(q, k.transpose(1, 2)), dim=-1)
        context = torch.bmm(attn_weights, v).squeeze(1)
        return self.fc(context)

# ----------------------------
# 4. Mezcla Cortical Multimodal
# ----------------------------
class CorticalMoE(nn.Module):
    def __init__(self, image_dim, text_dim):
        super().__init__()
        self.input_dim = image_dim + text_dim
        self.router = Router(self.input_dim, 4)
        self.experts = nn.ModuleList([
            MemoryExpert(self.input_dim, 128),
            LogicExpert(self.input_dim),
            InstinctExpert(self.input_dim),
            ContextExpert(self.input_dim)
        ])

    def forward(self, image, text_embed):
        x = torch.cat((image, text_embed), dim=1)
        expert_idx, log_prob, probs = self.router(x)
        outputs = torch.stack([expert(x) for expert in self.experts])
        selected_output = outputs[expert_idx, torch.arange(x.size(0))]
        return selected_output, log_prob, expert_idx, probs

# ----------------------------
# 5. Entrenamiento Multimodal
# ----------------------------
def train_loop(model, dataloader, optimizer, router_optimizer, epoch, text_embed_dim=50):
    model.train()
    total_loss = 0
    expert_usage = torch.zeros(4)

    for images, labels in dataloader:
        images = images.view(images.size(0), -1)
        text_embeds = generate_fake_text_embedding(images.size(0), text_embed_dim)

        optimizer.zero_grad()
        router_optimizer.zero_grad()

        output, log_prob, expert_idx, _ = model(images, text_embeds)
        classification_loss = F.cross_entropy(output, labels)
        reward = -classification_loss.detach()
        reinforce_loss = -log_prob.mean() * reward
        loss = classification_loss + reinforce_loss

        loss.backward()
        optimizer.step()
        router_optimizer.step()

        for i in expert_idx:
            expert_usage[i] += 1
        total_loss += loss.item()

    print(f"Época {epoch} | Pérdida: {total_loss/len(dataloader):.4f} | Uso por experto (entrenamiento): {expert_usage.tolist()}")

# ----------------------------
# 6. Evaluación (Test)
# ----------------------------
def test_loop(model, dataloader, text_embed_dim=50):
    model.eval()
    correct = 0
    total = 0
    expert_usage = torch.zeros(4)

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.view(images.size(0), -1)
            text_embeds = generate_fake_text_embedding(images.size(0), text_embed_dim)

            output, _, expert_idx, _ = model(images, text_embeds)
            _, predicted = torch.max(output.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            for i in expert_idx:
                expert_usage[i] += 1

    accuracy = 100 * correct / total
    print(f'Precisión del modelo en el conjunto de prueba: {accuracy:.2f}%')
    print(f"Uso por experto (prueba): {expert_usage.tolist()}")
    return accuracy, expert_usage

# ----------------------------
# 7. Main
# ----------------------------
def main():
    transform = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x.view(-1))])
    train_set = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    test_set = datasets.MNIST(root='./data', train=False, download=True, transform=transform) # Test set
    train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=64, shuffle=False) # Test loader

    image_dim = 28*28
    text_dim = 50

    model = CorticalMoE(image_dim, text_dim)
    router_params = model.router.parameters()
    expert_params = [p for n, p in model.named_parameters() if 'router' not in n]

    optimizer = torch.optim.Adam(expert_params, lr=1e-3)
    router_optimizer = torch.optim.Adam(router_params, lr=1e-3)

    for epoch in range(1, 6):
        train_loop(model, train_loader, optimizer, router_optimizer, epoch, text_embed_dim=text_dim)

    # Evaluate the model after training
    accuracy, expert_usage = test_loop(model, test_loader, text_embed_dim=text_dim)

    # Visualize expert usage
    expert_names = ['Memoria', 'Lógica', 'Instinto', 'Contexto']
    expert_usage_percentage = (expert_usage / expert_usage.sum() * 100).tolist()

    plt.figure(figsize=(8, 6))
    plt.bar(expert_names, expert_usage_percentage, color=['skyblue', 'lightcoral', 'lightgreen', 'gold'])
    plt.xlabel('Expertos')
    plt.ylabel('Porcentaje de Uso')
    plt.title('Uso de Expertos en el Conjunto de Prueba')
    plt.ylim(0, 100)
    plt.grid(axis='y', linestyle='--')
    plt.tight_layout()
    plt.savefig('expert_usage.png')

if __name__ == "__main__":
    main()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.distributions import Categorical
from torchvision import datasets, transforms
import numpy as np
import random
import matplotlib.pyplot as plt

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Set device to CPU
device = torch.device("cpu")

# ----------------------------
# 1. Generador de texto falso (embedding aleatorio por imagen)
# ----------------------------
def generate_fake_text_embedding(batch_size, embed_dim=50):
    return torch.randn(batch_size, embed_dim, device=device)

# ----------------------------
# 2. Router Tálamo-Cortical
# ----------------------------
class Router(nn.Module):
    def __init__(self, input_dim, n_experts):
        super().__init__()
        self.fc = nn.Linear(input_dim, n_experts)

    def forward(self, x):
        logits = self.fc(x)
        probs = F.softmax(logits, dim=1)
        dist = Categorical(probs)
        expert_idx = dist.sample()
        log_prob = dist.log_prob(expert_idx)
        return expert_idx, log_prob, probs

# ----------------------------
# 3. Expertos
# ----------------------------
class MemoryExpert(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 10)

    def forward(self, x):
        x = x.unsqueeze(1)
        _, (h_n, _) = self.lstm(x)
        return self.fc(h_n.squeeze(0))

class LogicExpert(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x): return self.net(x)

class InstinctExpert(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32), nn.ReLU(),
            nn.Linear(32, 10)
        )

    def forward(self, x): return self.net(x)

class ContextExpert(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.query = nn.Linear(input_dim, 64)
        self.key = nn.Linear(input_dim, 64)
        self.value = nn.Linear(input_dim, 64)
        self.fc = nn.Linear(64, 10)

    def forward(self, x):
        q, k, v = self.query(x).unsqueeze(1), self.key(x).unsqueeze(1), self.value(x).unsqueeze(1)
        attn_weights = F.softmax(torch.bmm(q, k.transpose(1, 2)), dim=-1)
        context = torch.bmm(attn_weights, v).squeeze(1)
        return self.fc(context)

# ----------------------------
# 4. Mezcla Cortical Multimodal
# ----------------------------
class CorticalMoE(nn.Module):
    def __init__(self, image_dim, text_dim):
        super().__init__()
        self.input_dim = image_dim + text_dim
        self.router = Router(self.input_dim, 4)
        self.experts = nn.ModuleList([
            MemoryExpert(self.input_dim, 128),
            LogicExpert(self.input_dim),
            InstinctExpert(self.input_dim),
            ContextExpert(self.input_dim)
        ])

    def forward(self, image, text_embed):
        x = torch.cat((image, text_embed), dim=1)
        expert_idx, log_prob, probs = self.router(x)
        outputs = torch.stack([expert(x) for expert in self.experts])
        selected_output = outputs[expert_idx, torch.arange(x.size(0))]
        return selected_output, log_prob, expert_idx, probs

# ----------------------------
# 5. Entrenamiento Multimodal
# ----------------------------
def train_loop(model, dataloader, optimizer, router_optimizer, epoch, text_embed_dim=50):
    model.train()
    total_loss = 0
    expert_usage = torch.zeros(4, device=device)

    for images, labels in dataloader:
        images = images.view(images.size(0), -1).to(device)
        labels = labels.to(device)
        text_embeds = generate_fake_text_embedding(images.size(0), text_embed_dim)

        optimizer.zero_grad()
        router_optimizer.zero_grad()

        output, log_prob, expert_idx, _ = model(images, text_embeds)
        classification_loss = F.cross_entropy(output, labels)
        reward = -classification_loss.detach()
        reinforce_loss = -log_prob.mean() * reward
        loss = classification_loss + reinforce_loss

        loss.backward()
        optimizer.step()
        router_optimizer.step()

        for i in expert_idx:
            expert_usage[i] += 1
        total_loss += loss.item()

    print(f"Época {epoch} | Pérdida: {total_loss/len(dataloader):.4f} | Uso por experto (entrenamiento): {expert_usage.tolist()}")

# ----------------------------
# 6. Evaluación (Test)
# ----------------------------
def test_loop(model, dataloader, text_embed_dim=50):
    model.eval()
    correct = 0
    total = 0
    expert_usage = torch.zeros(4, device=device)

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.view(images.size(0), -1).to(device)
            labels = labels.to(device)
            text_embeds = generate_fake_text_embedding(images.size(0), text_embed_dim)

            output, _, expert_idx, _ = model(images, text_embeds)
            _, predicted = torch.max(output.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            for i in expert_idx:
                expert_usage[i] += 1

    accuracy = 100 * correct / total
    print(f'Precisión del modelo en el conjunto de prueba: {accuracy:.2f}%')
    print(f"Uso por experto (prueba): {expert_usage.tolist()}")
    return accuracy, expert_usage

# ----------------------------
# 7. Main
# ----------------------------
def main():
    transform = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x.view(-1))])
    train_set = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    test_set = datasets.MNIST(root='./data', train=False, download=True, transform=transform) # Test set
    train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=64, shuffle=False) # Test loader

    image_dim = 28*28
    text_dim = 50

    model = CorticalMoE(image_dim, text_dim).to(device)
    router_params = model.router.parameters()
    expert_params = [p for n, p in model.named_parameters() if 'router' not in n]

    optimizer = torch.optim.Adam(expert_params, lr=1e-3)
    router_optimizer = torch.optim.Adam(router_params, lr=1e-3)

    for epoch in range(1, 6):
        train_loop(model, train_loader, optimizer, router_optimizer, epoch, text_embed_dim=text_dim)

    # Evaluate the model after training
    accuracy, expert_usage = test_loop(model, test_loader, text_embed_dim=text_dim)

    # Visualize expert usage
    expert_names = ['Memoria', 'Lógica', 'Instinto', 'Contexto']
    expert_usage_percentage = (expert_usage / expert_usage.sum() * 100).tolist()

    plt.figure(figsize=(8, 6))
    plt.bar(expert_names, expert_usage_percentage, color=['skyblue', 'lightcoral', 'lightgreen', 'gold'])
    plt.xlabel('Expertos')
    plt.ylabel('Porcentaje de Uso')
    plt.title('Uso de Expertos en el Conjunto de Prueba')
    plt.ylim(0, 100)
    plt.grid(axis='y', linestyle='--')
    plt.tight_layout()
    plt.savefig('expert_usage.png')

if __name__ == "__main__":
    main()

##Ruteo de salidas de expertos

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ------------------------------
# 1. Router tipo Tálamo
# ------------------------------
class Router(nn.Module):
    def __init__(self, input_dim, n_experts):
        super().__init__()
        self.fc = nn.Linear(input_dim, n_experts)

    def forward(self, x):
        logits = self.fc(x)
        probs = F.softmax(logits, dim=1)
        dist = Categorical(probs)
        expert_idx = dist.sample()
        log_prob = dist.log_prob(expert_idx)
        return expert_idx, log_prob, probs

# ------------------------------
# 2. Expertos especializados
# ------------------------------
class MemoryExpert(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 10)

    def forward(self, x):
        x = x.unsqueeze(1)
        _, (h_n, _) = self.lstm(x)
        return self.fc(h_n.squeeze(0))

class LogicExpert(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x): return self.net(x)

class InstinctExpert(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32), nn.ReLU(),
            nn.Linear(32, 10)
        )

    def forward(self, x): return self.net(x)

class ContextExpert(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.query = nn.Linear(input_dim, 64)
        self.key = nn.Linear(input_dim, 64)
        self.value = nn.Linear(input_dim, 64)
        self.fc = nn.Linear(64, 10)

    def forward(self, x):
        q, k, v = self.query(x), self.key(x), self.value(x)
        attn_weights = F.softmax(torch.bmm(q.unsqueeze(1), k.unsqueeze(2)), dim=-1)
        context = torch.bmm(attn_weights, v.unsqueeze(1)).squeeze(1)
        return self.fc(context)

# ------------------------------
# 3. MoE Cortical
# ------------------------------
class CorticalMoE(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.router = Router(input_dim, 4)
        self.experts = nn.ModuleList([
            MemoryExpert(input_dim, 128),
            LogicExpert(input_dim),
            InstinctExpert(input_dim),
            ContextExpert(input_dim)
        ])

    def forward(self, x):
        expert_idx, log_prob, probs = self.router(x)
        outputs = torch.stack([expert(x) for expert in self.experts])  # [4, B, 10]
        selected_output = outputs[expert_idx, torch.arange(x.size(0))]
        return selected_output, log_prob, expert_idx, probs

# ------------------------------
# 4. Entrenamiento
# ------------------------------
def train_loop(model, dataloader, optimizer, router_optimizer, epoch):
    model.train()
    total_loss = 0
    expert_usage = torch.zeros(4)
    for images, labels in dataloader:
        images = images.view(images.size(0), -1)
        optimizer.zero_grad()
        router_optimizer.zero_grad()

        output, log_prob, expert_idx, _ = model(images)
        classification_loss = F.cross_entropy(output, labels)
        reward = -classification_loss.detach()
        reinforce_loss = -log_prob.mean() * reward

        loss = classification_loss + reinforce_loss
        loss.backward()
        optimizer.step()
        router_optimizer.step()

        for i in expert_idx:
            expert_usage[i] += 1
        total_loss += loss.item()

    print(f"Epoch {epoch} | Loss: {total_loss/len(dataloader):.4f} | Expert Usage: {expert_usage.tolist()}")

# ------------------------------
# 5. Main para MNIST
# ------------------------------
def main():
    transform = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x.view(-1))])
    train_set = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    train_loader = DataLoader(train_set, batch_size=64, shuffle=True)

    model = CorticalMoE(28*28)
    optimizer = optim.Adam([p for n, p in model.named_parameters() if 'router' not in n], lr=1e-3)
    router_optimizer = optim.Adam(model.router.parameters(), lr=1e-3)

    for epoch in range(1, 6):
        train_loop(model, train_loader, optimizer, router_optimizer, epoch)

if __name__ == '__main__':
    main()

##Aquí tienes un modelo completo estilo "cortical-thalamic" con:

Router RL que decide qué experto (bioma) usar por entrada.

Cuatro expertos especializados: Memoria (LSTM), Cálculo (MLP profundo), Instinto (shallow), Contexto (autoatención).

Entrenamiento tipo REINFORCE para el router, que aprende a seleccionar expertos útiles.


Incluye un esquema de uso de expertos para visualizar si hay colapso (todos eligen el mismo) y se puede extender fácilmente con mecanismos de entropía o penalización si lo deseas.

¿Quieres que le incorpore feedback de salida al router (memoria interbioma), o implementamos la regularización del router ahora?

### clasificación (memoria)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# ------------------------------
# 1. Router tipo Tálamo
# ------------------------------
class Router(nn.Module):
    def __init__(self, input_dim, n_experts):
        super().__init__()
        self.fc = nn.Linear(input_dim, n_experts)

    def forward(self, x):
        logits = self.fc(x)
        probs = F.softmax(logits, dim=1)
        dist = Categorical(probs)
        expert_idx = dist.sample()
        log_prob = dist.log_prob(expert_idx)
        return expert_idx, log_prob, probs

# ------------------------------
# 2. Expertos especializados
# ------------------------------
class MemoryExpert(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        # input_dim here is 28*28, but LSTM expects sequential data.
        # This Expert seems intended for sequential processing, but MNIST images are single frames.
        # The current implementation treats each flattened image as a sequence of length 1.
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 10)

    def forward(self, x):
        # Unsqueeze to add sequence dimension: (batch_size, 1, input_dim)
        x = x.unsqueeze(1)
        # LSTM output: (batch_size, 1, hidden_dim), (h_n, c_n) where h_n, c_n are (1, batch_size, hidden_dim)
        _, (h_n, _) = self.lstm(x)
        # Squeeze h_n to get (batch_size, hidden_dim)
        return self.fc(h_n.squeeze(0))

class LogicExpert(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x): return self.net(x)

class InstinctExpert(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32), nn.ReLU(),
            nn.Linear(32, 10)
        )

    def forward(self, x): return self.net(x)

class ContextExpert(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.query = nn.Linear(input_dim, 64)
        self.key = nn.Linear(input_dim, 64)
        self.value = nn.Linear(input_dim, 64)
        self.fc = nn.Linear(64, 10)

    def forward(self, x):
        # For a single image (batch_size, input_dim), treating it as a context vector
        # Unsqueeze to add dimension for bmm: (batch_size, 1, input_dim)
        q, k, v = self.query(x).unsqueeze(1), self.key(x).unsqueeze(1), self.value(x).unsqueeze(1)
        # Calculate attention weights: (batch_size, 1, 1)
        attn_weights = F.softmax(torch.bmm(q, k.transpose(1, 2)), dim=-1)
        # Apply attention weights: (batch_size, 1, 64)
        context = torch.bmm(attn_weights, v).squeeze(1) # Squeeze to (batch_size, 64)
        return self.fc(context)

# ------------------------------
# 3. MoE Cortical
# ------------------------------
class CorticalMoE(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.router = Router(input_dim, 4)
        self.experts = nn.ModuleList([
            MemoryExpert(input_dim, 128),
            LogicExpert(input_dim),
            InstinctExpert(input_dim),
            ContextExpert(input_dim)
        ])

    def forward(self, x):
        expert_idx, log_prob, probs = self.router(x)
        # Collect outputs from all experts. Ensure inputs to experts are correctly shaped.
        # All experts expect (batch_size, input_dim) except MemoryExpert which handles unsqueeze internally.
        outputs = torch.stack([expert(x) for expert in self.experts])  # Shape: [num_experts, B, output_dim]
        # Select output from chosen expert for each item in the batch
        # We need to use advanced indexing to select the expert output for each item based on expert_idx
        selected_output = outputs[expert_idx, torch.arange(x.size(0))]
        return selected_output, log_prob, expert_idx, probs

# ------------------------------
# 4. Entrenamiento
# ------------------------------
def train_loop(model, dataloader, optimizer, router_optimizer, epoch):
    model.train()
    total_loss = 0
    expert_usage = torch.zeros(4)
    for images, labels in dataloader:
        # Images are already flattened by the DataLoader's transform
        # images shape: (batch_size, 28*28)
        optimizer.zero_grad()
        router_optimizer.zero_grad()

        # Pass flattened images to the model
        output, log_prob, expert_idx, _ = model(images)
        classification_loss = F.cross_entropy(output, labels)

        # Calculate reward based on classification loss
        # Detach the loss to prevent gradients from flowing back into the classification part via the reward
        reward = -classification_loss.detach()

        # Calculate the REINFORCE loss for the router
        # Penalize the log probability of the chosen action (expert_idx) by the negative reward
        # A positive reward makes -log_prob * reward negative -> minimize this loss means maximize log_prob for good actions
        # A negative reward makes -log_prob * reward positive -> minimize this loss means minimize log_prob for bad actions
        # Mean over the batch
        reinforce_loss = -log_prob.mean() * reward # No explicit gamma needed here unless it's part of the reward function

        # Total loss is classification loss + router policy gradient loss
        loss = classification_loss + reinforce_loss
        loss.backward()

        # Step optimizers
        optimizer.step()
        router_optimizer.step()

        # Track expert usage
        for i in expert_idx:
            expert_usage[i] += 1
        total_loss += loss.item()

    print(f"Epoch {epoch} | Loss: {total_loss/len(dataloader):.4f} | Expert Usage: {expert_usage.tolist()}")

# ------------------------------
# 5. Función de Visualización
# ------------------------------
def visualize_predictions(model, dataset, num_images=3):
    model.eval() # Pone el modelo en modo evaluación
    indices = np.random.choice(len(dataset), num_images, replace=False) # Elige N imágenes al azar

    fig, axes = plt.subplots(1, num_images, figsize=(10, 5))
    if num_images == 1: # Asegura que axes sea iterable incluso para una sola imagen
        axes = [axes]

    # We will use a transform thatToTensor but DOES NOT flatten for visualization
    display_transform = transforms.Compose([transforms.ToTensor()])

    with torch.no_grad(): # Desactiva el cálculo de gradientes
        for i, idx in enumerate(indices):
            # Get the original image tensor from the dataset (before flattening transform)
            # Need to load it manually or adjust the dataset transform for this section only
            # A simpler way is to apply the flattening transform *after* getting the image for prediction
            # and keep the original shape for display.

            # Load the image using the display_transform (ToTensor, NO flattening)
            image_for_display, label = dataset[idx]
            # Apply flattening only for the model prediction
            image_for_model = image_for_display.view(1, -1) # Shape (1, 28*28) for model

            # Get model output
            output, _, expert_idx, probs = model(image_for_model)
            predicted_class = torch.argmax(output).item()
            selected_expert_name = ["Memory", "Logic", "Instinct", "Context"][expert_idx.item()]
            expert_probs = probs.squeeze().tolist()

            ax = axes[i]
            # Use the tensor that has the correct shape for display (1, 28, 28)
            # Squeeze to remove the channel dimension for grayscale display (28, 28)
            # Convert to numpy
            ax.imshow(image_for_display.squeeze().numpy(), cmap='gray')
            ax.set_title(f"Predicción: {predicted_class}\nExperto: {selected_expert_name}\nProbs expertos: {[f'{p:.2f}' for p in expert_probs]}")
            ax.axis('off')
    plt.tight_layout()
    plt.show()


# ------------------------------
# 6. Main para MNIST (con visualización)
# ------------------------------
def main():
    # Define transform for training data (includes flattening)
    train_transform = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x.view(-1))])
    # Define transform for test data used in DataLoader (includes flattening)
    test_loader_transform = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x.view(-1))])
    # Define transform for test data used in visualize_predictions (only ToTensor, NO flattening)
    test_display_transform = transforms.Compose([transforms.ToTensor()])


    train_set = datasets.MNIST(root='./data', train=True, download=True, transform=train_transform)
    # For test set, load it with the display transform initially
    # Then manually apply the flattening when passing to the model in visualize_predictions
    test_set_for_display = datasets.MNIST(root='./data', train=False, download=True, transform=test_display_transform)

    train_loader = DataLoader(train_set, batch_size=64, shuffle=True)

    model = CorticalMoE(28*28)
    # Separate parameters for optimizers
    router_params = model.router.parameters()
    expert_params = [p for n, p in model.named_parameters() if 'router' not in n]

    optimizer = optim.Adam(expert_params, lr=1e-3)
    router_optimizer = optim.Adam(router_params, lr=1e-3)

    print("Entrenando el modelo...")
    for epoch in range(1, 6): # Entrenamos por unas pocas épocas
        train_loop(model, train_loader, optimizer, router_optimizer, epoch)

    print("\nVisualizando algunas predicciones del modelo:")
    # Pass the test_set loaded with the display transform
    visualize_predictions(model, test_set_for_display, num_images=3)

if __name__ == '__main__':
    main()

### calculo

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# ------------------------------
# 1. Router tipo Tálamo
# ------------------------------
class Router(nn.Module):
    def __init__(self, input_dim, n_experts):
        super().__init__()
        self.fc = nn.Linear(input_dim, n_experts)

    def forward(self, x):
        logits = self.fc(x)
        probs = F.softmax(logits, dim=1)
        dist = Categorical(probs)
        expert_idx = dist.sample()
        log_prob = dist.log_prob(expert_idx)
        return expert_idx, log_prob, probs

# ------------------------------
# 2. Expertos especializados
# ------------------------------
class MemoryExpert(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        # input_dim here is 28*28, but LSTM expects sequential data.
        # This Expert seems intended for sequential processing, but MNIST images are single frames.
        # The current implementation treats each flattened image as a sequence of length 1.
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 10)

    def forward(self, x):
        # Unsqueeze to add sequence dimension: (batch_size, 1, input_dim)
        x = x.unsqueeze(1)
        # LSTM output: (batch_size, 1, hidden_dim), (h_n, c_n) where h_n, c_n are (1, batch_size, hidden_dim)
        _, (h_n, _) = self.lstm(x)
        # Squeeze h_n to get (batch_size, hidden_dim)
        return self.fc(h_n.squeeze(0))

class LogicExpert(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x): return self.net(x)

class InstinctExpert(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32), nn.ReLU(),
            nn.Linear(32, 10)
        )

    def forward(self, x): return self.net(x)

class ContextExpert(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.query = nn.Linear(input_dim, 64)
        self.key = nn.Linear(input_dim, 64)
        self.value = nn.Linear(input_dim, 64)
        self.fc = nn.Linear(64, 10)

    def forward(self, x):
        # For a single image (batch_size, input_dim), treating it as a context vector
        # Unsqueeze to add dimension for bmm: (batch_size, 1, input_dim)
        q, k, v = self.query(x).unsqueeze(1), self.key(x).unsqueeze(1), self.value(x).unsqueeze(1)
        # Calculate attention weights: (batch_size, 1, 1)
        attn_weights = F.softmax(torch.bmm(q, k.transpose(1, 2)), dim=-1)
        # Apply attention weights: (batch_size, 1, 64)
        context = torch.bmm(attn_weights, v).squeeze(1) # Squeeze to (batch_size, 64)
        return self.fc(context)

# ------------------------------
# 3. MoE Cortical
# ------------------------------
class CorticalMoE(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.router = Router(input_dim, 4)
        self.experts = nn.ModuleList([
            MemoryExpert(input_dim, 128),
            LogicExpert(input_dim),
            InstinctExpert(input_dim),
            ContextExpert(input_dim)
        ])

    def forward(self, x):
        expert_idx, log_prob, probs = self.router(x)
        # Collect outputs from all experts. Ensure inputs to experts are correctly shaped.
        # All experts expect (batch_size, input_dim) except MemoryExpert which handles unsqueeze internally.
        outputs = torch.stack([expert(x) for expert in self.experts])  # Shape: [num_experts, B, output_dim]
        # Select output from chosen expert for each item in the batch
        # We need to use advanced indexing to select the expert output for each item based on expert_idx
        selected_output = outputs[expert_idx, torch.arange(x.size(0))]
        return selected_output, log_prob, expert_idx, probs

    def predict_with_expert(self, x, expert_index):
        return self.experts[expert_index](x)

# ------------------------------
# 4. Entrenamiento
# ------------------------------
def train_loop(model, dataloader, optimizer, router_optimizer, epoch):
    model.train()
    total_loss = 0
    expert_usage = torch.zeros(4)
    for images, labels in dataloader:
        # Images are already flattened by the DataLoader's transform
        # images shape: (batch_size, 28*28)
        optimizer.zero_grad()
        router_optimizer.zero_grad()

        # Pass flattened images to the model
        output, log_prob, expert_idx, _ = model(images)
        classification_loss = F.cross_entropy(output, labels)

        # Calculate reward based on classification loss
        # Detach the loss to prevent gradients from flowing back into the classification part via the reward
        reward = -classification_loss.detach()

        # Calculate the REINFORCE loss for the router
        # Penalize the log probability of the chosen action (expert_idx) by the negative reward
        # A positive reward makes -log_prob * reward negative -> minimize this loss means maximize log_prob for good actions
        # A negative reward makes -log_prob * reward positive -> minimize this loss means minimize log_prob for bad actions
        # Mean over the batch
        reinforce_loss = -log_prob.mean() * reward # No explicit gamma needed here unless it's part of the reward function

        # Total loss is classification loss + router policy gradient loss
        loss = classification_loss + reinforce_loss
        loss.backward()

        # Step optimizers
        optimizer.step()
        router_optimizer.step()

        # Track expert usage
        for i in expert_idx:
            expert_usage[i] += 1
        total_loss += loss.item()

    print(f"Epoch {epoch} | Loss: {total_loss/len(dataloader):.4f} | Expert Usage: {expert_usage.tolist()}")

# ------------------------------
# 5. Función de Visualización
# ------------------------------
def visualize_predictions(model, dataset, expert_index, num_images=3):
    model.eval() # Pone el modelo en modo evaluación
    indices = np.random.choice(len(dataset), num_images, replace=False) # Elige N imágenes al azar

    fig, axes = plt.subplots(1, num_images, figsize=(10, 5))
    if num_images == 1: # Asegura que axes sea iterable incluso para una sola imagen
        axes = [axes]

    # We will use a transform thatToTensor but DOES NOT flatten for visualization
    display_transform = transforms.Compose([transforms.ToTensor()])

    with torch.no_grad(): # Desactiva el cálculo de gradientes
        for i, idx in enumerate(indices):
            # Get the original image tensor from the dataset (before flattening transform)
            # Need to load it manually or adjust the dataset transform for this section only
            # A simpler way is to apply the flattening transform *after* getting the image for prediction
            # and keep the original shape for display.

            # Load the image using the display_transform (ToTensor, NO flattening)
            image_for_display, label = dataset[idx]
            # Apply flattening only for the model prediction
            image_for_model = image_for_display.view(1, -1) # Shape (1, 28*28) for model

            # Get model output from the specified expert
            output = model.predict_with_expert(image_for_model, expert_index)
            predicted_class = torch.argmax(output).item()
            expert_name = ["Memory", "Logic", "Instinct", "Context"][expert_index]

            ax = axes[i]
            # Use the tensor that has the correct shape for display (1, 28, 28)
            # Squeeze to remove the channel dimension for grayscale display (28, 28)
            # Convert to numpy
            ax.imshow(image_for_display.squeeze().numpy(), cmap='gray')
            ax.set_title(f"Experto {expert_name}\nPredicción: {predicted_class}")
            ax.axis('off')
    plt.tight_layout()
    plt.show()


# ------------------------------
# 6. Main para MNIST (con visualización)
# ------------------------------
def main():
    # Define transform for training data (includes flattening)
    train_transform = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x.view(-1))])
    # Define transform for test data used in DataLoader (includes flattening)
    test_loader_transform = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x.view(-1))])
    # Define transform for test data used in visualize_predictions (only ToTensor, NO flattening)
    test_display_transform = transforms.Compose([transforms.ToTensor()])


    train_set = datasets.MNIST(root='./data', train=True, download=True, transform=train_transform)
    # For test set, load it with the display transform initially
    # Then manually apply the flattening when passing to the model in visualize_predictions
    test_set_for_display = datasets.MNIST(root='./data', train=False, download=True, transform=test_display_transform)

    train_loader = DataLoader(train_set, batch_size=64, shuffle=True)

    model = CorticalMoE(28*28)
    # Separate parameters for optimizers
    router_params = model.router.parameters()
    expert_params = [p for n, p in model.named_parameters() if 'router' not in n]

    optimizer = optim.Adam(expert_params, lr=1e-3)
    router_optimizer = optim.Adam(router_params, lr=1e-3)

    print("Entrenando el modelo...")
    for epoch in range(1, 6): # Entrenamos por unas pocas épocas
        train_loop(model, train_loader, optimizer, router_optimizer, epoch)

    print("\nVisualizando predicciones por experto:")
    for i in range(4):
        print(f"\nVisualizando predicciones del experto {['Memory', 'Logic', 'Instinct', 'Context'][i]}:")
        visualize_predictions(model, test_set_for_display, i, num_images=3)

if __name__ == '__main__':
    main()

##Pensaste sumamente bien, te lo agradezco, si me gustaría cada esperto es una región al final no perdemos la idea lo que es bueno sigua así como estabas 🤔

1. Biomas especializados:



Primer esperto Memoria → con LSTM

Segundo Cálculo → MLP profundo

Tercero Instinto → red shallow

Cuarto: Contexto → con atención

2. Regularización del router:



Penalizar uso excesivo de un solo bioma.

Maximizar entropía para forzar especialización.

3. Memory feedback loop:



Mandar la salida de un bioma al router como hint en el siguiente paso (memoria interbioma).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ThalamusBiomeNet(nn.Module):
    def __init__(self, input_dim, shared_dim, biome_dim, output_dim, k_active=2):
        super().__init__()

        self.k_active = k_active
        self.num_biomes = 4

        # Feature extractor (sensorial cortex)
        self.feature_encoder = nn.Sequential(
            nn.Linear(input_dim, shared_dim),
            nn.ReLU()
        )

        # Router (thalamus)
        self.thalamus_router = nn.Linear(shared_dim + output_dim, self.num_biomes)

        # Biome 1: Memory (LSTM)
        self.memory_lstm = nn.LSTM(shared_dim, biome_dim, batch_first=True)
        self.memory_proj = nn.Linear(biome_dim, output_dim)

        # Biome 2: Deep MLP (Calculation)
        self.calculation_mlp = nn.Sequential(
            nn.Linear(shared_dim, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, output_dim)
        )

        # Biome 3: Instinct (Shallow)
        self.instinct_net = nn.Sequential(
            nn.Linear(shared_dim, 64), nn.ReLU(),
            nn.Linear(64, output_dim)
        )

        # Biome 4: Context (Self-Attention)
        self.context_attention = nn.MultiheadAttention(embed_dim=shared_dim, num_heads=4, batch_first=True)
        self.context_proj = nn.Linear(shared_dim, output_dim)

        # Initial feedback (output from previous step, zeroed initially)
        self.register_buffer("last_output", torch.zeros(1, output_dim))

    def forward(self, x):
        batch_size = x.size(0)
        shared_feat = self.feature_encoder(x)  # [B, shared_dim]

        # Repeat feedback memory to match batch size
        feedback = self.last_output.expand(batch_size, -1)

        # Routing decision
        router_input = torch.cat([shared_feat, feedback], dim=1)
        routing_logits = self.thalamus_router(router_input)

        # Top-k routing
        topk_vals, topk_idx = torch.topk(routing_logits, self.k_active, dim=1)
        routing_mask = torch.zeros_like(routing_logits)
        routing_mask.scatter_(1, topk_idx, 1)
        routing_weights = F.softmax(topk_vals, dim=1)

        # Prepare routing weights
        routing_weights_exp = torch.zeros_like(routing_logits).unsqueeze(-1)
        routing_weights_exp.scatter_(1, topk_idx.unsqueeze(-1), routing_weights.unsqueeze(-1))

        # Compute each biome output
        outputs = []

        # Memory biome (LSTM) — requiere input de 3D: [B, T=1, D]
        memory_input = shared_feat.unsqueeze(1)
        memory_out, _ = self.memory_lstm(memory_input)
        memory_final = self.memory_proj(memory_out[:, -1, :])
        outputs.append(memory_final)

        # Calculation biome (MLP)
        outputs.append(self.calculation_mlp(shared_feat))

        # Instinct biome (Shallow net)
        outputs.append(self.instinct_net(shared_feat))

        # Context biome (Attention) — también input 3D
        context_input = shared_feat.unsqueeze(1)
        attn_output, _ = self.context_attention(context_input, context_input, context_input)
        context_final = self.context_proj(attn_output.squeeze(1))
        outputs.append(context_final)

        # Stack biome outputs: [B, num_biomes, output_dim]
        biome_outputs = torch.stack(outputs, dim=1)

        # Weighted sum
        final_output = torch.sum(biome_outputs * routing_weights_exp, dim=1)

        # Feedback loop update (detached to prevent gradient leak)
        self.last_output = final_output.detach().mean(dim=0, keepdim=True)

        # Regularization: entropy penalty (high entropy => diverse routing)
        routing_probs = F.softmax(routing_logits, dim=1)
        entropy = -torch.sum(routing_probs * torch.log(routing_probs + 1e-8), dim=1).mean()

        return final_output, entropy

##🧠📡 Diseño: Tálamo Cortical Modular (Router de Biomas)

🔬 Concepto funcional:

El input sensorial es procesado por un extractor de características (como la corteza primaria).

El tálamo cortical (router) decide cuáles biomas activar (de forma suave o discreta).

Cada bioma representa un sistema funcional: memoria, cálculo, razonamiento, instinto...

La salida es una mezcla ponderada o selectiva de esos biomas.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ThalamusBiomeNet(nn.Module):
    def __init__(self, input_dim, shared_dim, num_biomes, biome_dim, output_dim, k_active=2):
        super().__init__()

        self.k_active = k_active  # Cuántos biomas se activan (Top-k)

        # Corteza sensorial
        self.feature_encoder = nn.Sequential(
            nn.Linear(input_dim, shared_dim),
            nn.ReLU()
        )

        # Tálamo cortical que decide qué biomas activar
        self.thalamus_router = nn.Linear(shared_dim, num_biomes)

        # Biomas especializados (funcionalmente diferentes si quieres)
        self.biomes = nn.ModuleList([
            nn.Sequential(
                nn.Linear(shared_dim, biome_dim),
                nn.ReLU(),
                nn.Linear(biome_dim, output_dim)
            ) for _ in range(num_biomes)
        ])

    def forward(self, x):
        shared_feat = self.feature_encoder(x)  # [batch, shared_dim]

        # Decisión del tálamo (logits sin softmax todavía)
        routing_logits = self.thalamus_router(shared_feat)  # [batch, num_biomes]

        # Aplicar top-k routing (sparse como Mixtral)
        topk_vals, topk_idx = torch.topk(routing_logits, self.k_active, dim=1)
        routing_mask = torch.zeros_like(routing_logits)
        routing_mask.scatter_(1, topk_idx, 1)
        routing_weights = F.softmax(topk_vals, dim=1)  # Normalizar solo los top-k

        # Expandimos pesos para hacer broadcast correcto
        routing_weights_exp = torch.zeros_like(routing_logits).unsqueeze(-1)
        routing_weights_exp.scatter_(1, topk_idx.unsqueeze(-1), routing_weights.unsqueeze(-1))  # [batch, num_biomes, 1]

        # Ejecutar todos los biomas
        biome_outputs = torch.stack([b(shared_feat) for b in self.biomes], dim=1)  # [batch, num_biomes, output_dim]

        # Mezclar solo los activos
        final_output = torch.sum(biome_outputs * routing_weights_exp, dim=1)  # [batch, output_dim]

        return final_output

##1. Un "tronco" (corteza central) que decide dónde se procesa cada input.


2. Un conjunto de módulos expertos o biomas, cada uno especializado.


3. Un sistema de enrutamiento inteligente, tipo Mixture of Experts (MoE), inspirado en cómo el cerebro delega funciones.



Tu instinto va bien. Lo que faltó fue integrar el Mecanismo de Enrutamiento (Router) de forma explícita y aprendible.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CerebralRouterNet(nn.Module):
    def __init__(self, input_dim, shared_dim, num_experts, expert_dim, output_dim):
        super().__init__()

        # 1. Corteza sensorial (feature extractor básico)
        self.shared_encoder = nn.Sequential(
            nn.Linear(input_dim, shared_dim),
            nn.ReLU()
        )

        # 2. Tronco central que decide (router)
        self.router = nn.Sequential(
            nn.Linear(shared_dim, num_experts),
            nn.Softmax(dim=1)  # Pondera a qué experto enviar el input
        )

        # 3. Biomas especializados (expertos)
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(shared_dim, expert_dim),
                nn.ReLU(),
                nn.Linear(expert_dim, output_dim)
            ) for _ in range(num_experts)
        ])

    def forward(self, x):
        shared_feat = self.shared_encoder(x)
        routing_weights = self.router(shared_feat)  # Tamaño: [batch, num_experts]

        # Procesar con todos los expertos en paralelo
        expert_outputs = torch.stack([expert(shared_feat) for expert in self.experts], dim=1)  # [batch, num_experts, output_dim]

        # Mezclar salidas de expertos según pesos del router
        routing_weights = routing_weights.unsqueeze(-1)  # [batch, num_experts, 1]
        final_output = torch.sum(expert_outputs * routing_weights, dim=1)  # [batch, output_dim]

        return final_output

##

#Router

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch.distributions import Categorical
import matplotlib.pyplot as plt
import numpy as np

# --- Config ---
config = {
    "input_dim": 28*28,
    "batch_size": 64,
    "lr": 1e-3,
    "epochs": 5,
    "model_path": "/content/cortical_moe_best.pth"
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Router y Expertos ---
class Router(nn.Module):
    def __init__(self, input_dim, n_experts):
        super().__init__()
        self.fc = nn.Linear(input_dim, n_experts)

    def forward(self, x):
        logits = self.fc(x)
        probs = F.softmax(logits, dim=1)
        dist = Categorical(probs)
        idx = dist.sample()
        logp = dist.log_prob(idx)
        return idx, logp, probs

class MemoryExpert(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        # Layers must be defined within __init__ and assigned to self
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 10)

    def forward(self, x):
        x = x.unsqueeze(1) # Add time dimension for LSTM
        _, (h, _) = self.lstm(x) # Get hidden state
        return self.fc(h.squeeze(0)) # Use hidden state for classification


class LogicExpert(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        # Layers must be defined within __init__ and assigned to self
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.net(x)

class InstinctExpert(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        # Layers must be defined within __init__ and assigned to self
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32), nn.ReLU(),
            nn.Linear(32, 10)
        )

    def forward(self, x):
        return self.net(x)

class ContextExpert(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        # Layers must be defined within __init__ and assigned to self
        self.q = nn.Linear(input_dim, 64)
        self.k = nn.Linear(input_dim, 64)
        self.v = nn.Linear(input_dim, 64)
        self.out = nn.Linear(64, 10)

    def forward(self, x):
        # Attention expects a sequence dimension, but the input x is [B, input_dim].
        # Unsqueeze to simulate a sequence of length 1 if needed for self-attention,
        # but a single step attention is just a linear combination.
        # The logic below performs a simple self-attention like mechanism on the single input vector.
        q = self.q(x).unsqueeze(1) # [B, 1, 64]
        k = self.k(x).unsqueeze(1) # [B, 1, 64]
        v = self.v(x).unsqueeze(1) # [B, 1, 64]

        attn = torch.bmm(q, k.transpose(1,2)) # [B, 1, 1]
        attn = F.softmax(attn, dim=-1)      # [B, 1, 1]

        context = torch.bmm(attn, v).squeeze(1) # [B, 1, 64] -> [B, 64]
        return self.out(context)


class CorticalMoE(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.router = Router(input_dim, 4)
        # Ensure experts are correctly instantiated with parameters defined in their __init__
        self.experts = nn.ModuleList([
            MemoryExpert(input_dim, 128), # Pass hidden_dim
            LogicExpert(input_dim),
            InstinctExpert(input_dim),
            ContextExpert(input_dim)
        ])
    def forward(self, x):
        idx, logp, probs = self.router(x)
        # Ensure expert outputs are collected and the correct one is selected per batch item
        # This assumes batch size > 1 and each item goes to potentially a different expert
        outs = torch.stack([expert(x) for expert in self.experts])  # [num_experts, B, 10]
        # Select the output based on the sampled index for each item in the batch
        # Use advanced indexing: outs[idx, batch_indices]
        batch_indices = torch.arange(x.size(0)).to(x.device) # Ensure indices are on the same device
        selected_out = outs[idx, batch_indices]

        return selected_out, logp, idx, probs

    def predict_with_expert(self, x, expert_idx):
        # Add batch dimension if input is a single sample
        if x.ndim == 1:
             x = x.unsqueeze(0)
        return self.experts[expert_idx](x)


def train_loop(model, loader, opt, r_opt):
    model.train()
    usage = torch.zeros(4).to(device)
    total_loss = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        r_opt.zero_grad() # Zero gradients for router optimizer

        # out, logp, idx, _ = model(x) # router_probs is not used here
        out, logp, idx, router_probs = model(x) # Capture router_probs if needed later, though not used for loss

        # Classification Loss
        loss_cls = F.cross_entropy(out, y)

        # Reinforcement Learning component for the Router
        # Reward is based on the classification loss (negative loss)
        # Penalize the router for choosing an expert that performed poorly
        # Adding 1.0 to make reward positive or less negative
        reward = -loss_cls.detach() # Use detach() so the loss gradient doesn't flow into the reward calculation

        # Reinforce the action (expert choice) based on the reward
        # The log probability of the chosen action is multiplied by the reward
        # Since we want to maximize reward, we minimize -logp * reward
        reinforce = -logp.mean() * reward # Use mean() of logp across the batch

        # Total loss is the sum of classification loss and the router's reinforcement loss
        total_loss_batch = loss_cls + reinforce

        # Backpropagation
        total_loss_batch.backward()

        # Update parameters
        opt.step() # Update expert parameters
        r_opt.step() # Update router parameters

        # Track expert usage
        for i in idx: usage[i] += 1
        total_loss += loss_cls.item() # Track only classification loss for reporting

    # Return average classification loss and expert usage
    return total_loss / len(loader), usage.tolist()


def main():
    transform = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x.view(-1))])
    train_data = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
    loader = DataLoader(train_data, batch_size=config["batch_size"], shuffle=True)

    model = CorticalMoE(config["input_dim"]).to(device)

    # Correctly separate router and expert parameters
    router_params = model.router.parameters()
    expert_params = [p for n, p in model.named_parameters() if 'router' not in n]

    # Ensure expert_params is not empty
    if not list(expert_params): # Convert to list to check if empty
         raise ValueError("Expert parameters list is empty. Check model layer definitions.")


    opt = optim.Adam(expert_params, lr=config["lr"])
    r_opt = optim.Adam(router_params, lr=config["lr"])

    best_acc, best_state = 0, None
    for epoch in range(1, config["epochs"] + 1):
        loss, usage = train_loop(model, loader, opt, r_opt)
        print(f"Ep {epoch} | Loss {loss:.4f} | Uso expertos: {usage}")

    torch.save(model.state_dict(), config["model_path"])
    print("Modelo guardado:", config["model_path"])

main()

In [ ]:
# Asume que ya ejecutaste Celda 1 antes

def evaluate(model, loader):
    model.eval(); correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out, _, _, _ = model(x)
            pred = out.argmax(1)
            correct += (pred == y).sum().item(); total += y.size(0)
    print(f"Precisión en test: {100 * correct / total:.2f}%")

def main():
    transform = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x.view(-1))])
    test_data = datasets.MNIST(root="./data", train=False, download=True, transform=transform)
    loader = DataLoader(test_data, batch_size=64)

    model = CorticalMoE(config["input_dim"]).to(device)
    model.load_state_dict(torch.load(config["model_path"], map_location=device, weights_only=True))
    print("Modelo cargado desde:", config["model_path"])
    evaluate(model, loader)

main()

#ReroutingNetwork

In [ ]:
# train.py
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from tqdm import trange
import os

# -------- Config --------
config = {
    "state_dim": 5,
    "num_actions": 2,
    "hidden_dim": 128,
    "lr": 0.01,
    "episodes": 500,
    "steps_per_episode": 20,
    "model_path": "/content/routing_rl_best_model.pt"
}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------- Modelo --------
class ReroutingNetwork(nn.Module):
    def __init__(self, state_dim, num_actions, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_actions)
        )
    def forward(self, x): return self.net(x)

# -------- Entorno simple --------
class SimpleEnv:
    def __init__(self):
        self.s = config["state_dim"]
        self.a = config["num_actions"]
        self.cur = np.random.randint(self.s)
        self.rewards = {(0,0):1, (1,0):2, (2,1):1, (3,1):3, (4,1):5}
        self.trans = {(0,0):1, (1,0):2, (2,1):3, (3,1):0, (4,1):1}
    def reset(self):
        self.cur = np.random.randint(self.s)
        s = np.eye(self.s)[self.cur]
        return torch.tensor(s, dtype=torch.float32, device=device).unsqueeze(0)
    def step(self, a):
        r = self.rewards.get((self.cur, a), 0)
        self.cur = self.trans.get((self.cur, a), self.cur)
        s = np.eye(self.s)[self.cur]
        return torch.tensor(s, dtype=torch.float32, device=device).unsqueeze(0), r, False

# -------- Entrenamiento --------
def train():
    env = SimpleEnv()
    model = ReroutingNetwork(config["state_dim"], config["num_actions"], config["hidden_dim"]).to(device)
    optimizer = optim.Adam(model.parameters(), lr=config["lr"])
    best_reward = -float("inf")

    for ep in trange(config["episodes"], desc="Entrenando RL"):
        state = env.reset()
        log_probs, rewards = [], []

        for _ in range(config["steps_per_episode"]):
            logits = model(state)
            dist = torch.distributions.Categorical(logits=logits)
            action = dist.sample()
            log_probs.append(dist.log_prob(action))
            next_state, reward, _ = env.step(action.item())
            rewards.append(reward)
            state = next_state

        G = sum(rewards)
        loss = -torch.stack([lp * G for lp in log_probs]).sum()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if G > best_reward:
            best_reward = G
            torch.save(model.state_dict(), config["model_path"])

    print(f"\nMejor modelo guardado en: {config['model_path']}")

if __name__ == "__main__":
    train()

In [ ]:
# load_model.py
import torch
import os
import numpy as np

config = {
    "state_dim": 5,
    "num_actions": 2,
    "hidden_dim": 128,
    "model_path": "/content/routing_rl_best_model.pt"
}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class ReroutingNetwork(torch.nn.Module):
    def __init__(self, state_dim, num_actions, hidden_dim=128):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(state_dim, hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dim, hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dim, num_actions)
        )
    def forward(self, x): return self.net(x)

def run():
    if not os.path.exists(config["model_path"]):
        raise FileNotFoundError("Modelo no encontrado")

    model = ReroutingNetwork(config["state_dim"], config["num_actions"], config["hidden_dim"]).to(device)
    model.load_state_dict(torch.load(config["model_path"], map_location=device, weights_only=True))
    model.eval()

    # Estado de prueba
    state = np.zeros(config["state_dim"])
    state[0] = 1  # ejemplo: estado inicial 0
    state = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(state)
        action = torch.argmax(logits, dim=1).item()
        print(f"Acción sugerida desde el estado 0: {action}")

if __name__ == "__main__":
    run()

## Agente de direcciones aleatorias

## Agente aleatorio

In [ ]:

import random
import matplotlib.pyplot as plt

def agente():
    direcciones = ["izquierda", "derecha", "arriba", "abajo"]
    return random.choice(direcciones)

contador_izquierda = 0
contador_derecha = 0
contador_arriba = 0
contador_abajo = 0

for i in range(999):
    accion = agente()
    if accion == "izquierda":
        contador_izquierda += 1
    elif accion == "derecha":
        contador_derecha += 1
    elif accion == "arriba":
        contador_arriba += 1
    else:
        contador_abajo += 1

plt.bar(["izquierda", "derecha", "arriba", "abajo"], [contador_izquierda, contador_derecha, contador_arriba, contador_abajo],
        color=["red", "green", "blue", "yellow"],
        label=["izquierda", "derecha", "arriba", "abajo"])
plt.title("Cantidad de veces que cada dirección es elegida")
plt.xlabel("Dirección")
plt.ylabel("Cantidad de veces")
plt.legend()
plt.show()